# GNR-Val v5.2 — Production-Ready CAD Compliance Validator
### Refactored from v5.0 | Varroc Eureka 3.0 · Problem Statement 9
> **v5.2 Changelog:** Removed dead cells, unified fusion logic and STEP parser, Colab-stable: pure-Python STEP parser, no conda/Miniforge/kernel restarts. ngrok auth via environment variable only.

---

| Module | Cell(s) | Description |
|--------|---------|-------------|
| **Cell 1** | Imports | All library imports |
| **Cell 2** | Config + Seed | Central `Config` dataclass, reproducibility |
| **Cell 3** | Domain Constants | Materials, components, standards, rules |
| **Cell 4** | Violation Dataclass | Typed result container |
| **Cell 5** | Rule Engine | ISO 2768 / ASME Y14.5 / GD&T / DFM rules |
| **Cell 6** | Fusion Logic | Score fusion + explainability per prediction |
| **Cell 7** | Data Generator | Synthetic CAD graph builder |
| **Cell 8** | GNN Model | 3-layer GraphSAGE + autoencoder + classifier |
| **Cell 9** | Training | `train_model()` with early-stop + cosine LR |
| **Cell 10** | Evaluation | `evaluate_model()` — full metrics, comparison table |
| **Cell 11** | Inference | `run_inference()` with per-prediction explainability |
| **Cell 12** | Pipeline Functions | `data_preparation()`, `main()` entry point |
| **Cell 13** | Interactive UI | ipywidgets CAD validator form |

Run cells top-to-bottom, or call `main()` at Cell 12 to execute the full pipeline.

```
pip install torch torch-geometric networkx scikit-learn ipywidgets pandas numpy matplotlib
```

## 🔄 Demo Workflow — End-to-End CAD Compliance Pipeline

This notebook implements a **hybrid AI + deterministic rule engine** that validates
industrial CAD assemblies against ISO 2768 / ASME Y14.5 / GD&T / DFM standards.

---

### Pipeline Overview

```
┌─────────────────┐     ┌──────────────────────┐     ┌───────────────────────┐
│  1. CAD Input   │────▶│  2. Graph Construction│────▶│  3. Rule Validation   │
│                 │     │                      │     │                       │
│  • STEP file    │     │  • Nodes: components │     │  ISO 2768 tolerances  │
│  • Manual entry │     │  • Edges: joints     │     │  ASME Y14.5 dims      │
│  • Widget form  │     │  • Edge attrs:       │     │  GD&T clearances      │
│                 │     │    clearance + type  │     │  DFM aspect ratio     │
└─────────────────┘     └──────────────────────┘     └───────────┬───────────┘
                                                                  │
         ┌────────────────────────────────────────────────────────┘
         ▼
┌─────────────────────┐     ┌───────────────────────┐     ┌──────────────────────┐
│  4. GNN Inference   │────▶│  5. Score Fusion       │────▶│  6. Dashboard Output │
│                     │     │                       │     │                      │
│  TransformerConv    │     │  No CRITs → 50/50 mix │     │  Verdict banner      │
│  3-layer encoder    │     │  ≥1 CRIT  → 30/70 mix │     │  Score cards         │
│  Autoencoder recon  │     │  (ML / Rule weights)  │     │  Violation log       │
│  3-class classifier │     │  Override if CRITICAL │     │  Remediation advisor │
│  0=OK 1=Review 2=NC │     │  Score 0–100          │     │  3D Digital Twin     │
└─────────────────────┘     └───────────────────────┘     └──────────────────────┘
```

### Quick Start
1. Run **Cell 1–12** top-to-bottom (or call `main()` at Cell 12).
2. Open the **Cell 13 widget** — enter component geometry, material, and type.
3. Click **Add Component** for each part, then **Run Validation**.
4. Read the **Verdict Banner**, **Score Cards**, and **Remediation Advisor**.

> **Tip:** Upload a `.step` / `.stp` file directly via the STEP Import widget to
> skip manual entry. The system will simulate feature extraction automatically.

---

In [ ]:
# ── GNR-Val v5.2 — Colab-stable dependency install ───────────────────────────
# Pure pip only. No conda, no Miniforge, no kernel restarts.
# Run once; then run all remaining cells top-to-bottom.

!pip install -q torch torch-geometric networkx scikit-learn \
               ipywidgets pandas numpy matplotlib plotly \

print("✅ All dependencies installed. Proceed to the next cell.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.9 MB/s eta 0:00:00
✅ All dependencies installed. Proceed to the next cell.


In [ ]:
%%writefile app.py



"""
GNR-Val v5.0 — Streamlit Industrial CAD Compliance Dashboard
Varroc Eureka 3.0 | Problem Statement 9
Run via: streamlit run app.py
"""

from __future__ import annotations
import io, json, logging, os, random, re, sys, time, warnings
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import (accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, global_mean_pool

warnings.filterwarnings("ignore")

st.set_page_config(page_title="GNR-Val v5.0 | CAD Compliance",
    page_icon="🏭", layout="wide", initial_sidebar_state="expanded")

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("gnrval")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Config:
    SEED: int = 42
    N_COMPLIANT: int = 400; N_REVIEW_NEEDED: int = 300; N_NONCOMPLIANT: int = 300
    MIN_COMPONENTS: int = 4; MAX_COMPONENTS: int = 14
    TRAIN_RATIO: float = 0.70; VAL_RATIO: float = 0.15
    IN_CHANNELS: int = 8; HIDDEN_DIM: int = 128; LATENT_DIM: int = 64
    DROPOUT: float = 0.25; EPOCHS: int = 120; BATCH_SIZE: int = 32
    LR: float = 1e-3; WEIGHT_DECAY: float = 1e-4; RECON_WEIGHT: float = 0.10
    GRAD_CLIP: float = 1.0; EARLY_STOP_PATIENCE: int = 20; LOG_EVERY: int = 10
    MODEL_PATH: str = "gnrval_final_combined.pt"
    REAL_MODEL_PATH: str = "gnrval_real_trained.pth"
    NONCOMPLIANT_THRESH: float = 70.0; REVIEW_THRESH: float = 40.0

cfg = Config()

def set_seed(seed=cfg.SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
MATERIAL_CODES: Dict[str,int] = {"steel":0,"aluminum":1,"plastic":2,"titanium":3,"composite":4}
COMPONENT_TYPES: Dict[str,int] = {"shaft":0,"bearing":1,"housing":2,"bracket":3,"fastener":4,"gear":5}
INV_MAT  = {v:k for k,v in MATERIAL_CODES.items()}
INV_COMP = {v:k for k,v in COMPONENT_TYPES.items()}
RULE_CATEGORIES = {"TOL":"Tolerance","COMP":"Component-Specific","SURF":"Surface Finish",
    "DIM":"Dimensional","DFM":"Design for Manufacture","MAT":"Material Compatibility","JNTI":"Joint/Clearance"}
STANDARDS = {"ISO2768_FINE":0.10,"ISO2768_MEDIUM":0.30,"ISO2768_COARSE":0.80,
    "RA_MATING":1.60,"RA_GENERAL":3.20,"RA_ROUGH":6.30,"DIM_MIN":5.0,"DIM_MAX":500.0,
    "CL_CLOSE":0.05,"CL_MEDIUM":0.20,"CL_LOOSE":0.50,"CL_CRITICAL":1.00,"WEIGHT_LIMIT":5000.0}
COMPONENT_RULES = {
    "shaft":   {"max_tol":0.050,"max_ra":1.6,"min_dim":5.0, "max_dim":300.0,"max_ar":10,"std":"IT6-IT8"},
    "bearing": {"max_tol":0.025,"max_ra":0.8,"min_dim":10.0,"max_dim":200.0,"max_ar":2, "std":"IT5-IT6"},
    "gear":    {"max_tol":0.050,"max_ra":1.6,"min_dim":20.0,"max_dim":400.0,"max_ar":5, "std":"ISO 1328"},
    "housing": {"max_tol":0.300,"max_ra":3.2,"min_dim":20.0,"max_dim":500.0,"max_ar":5, "std":"IT8-IT11"},
    "bracket": {"max_tol":0.500,"max_ra":6.3,"min_dim":10.0,"max_dim":500.0,"max_ar":8, "std":"IT11-IT14"},
    "fastener":{"max_tol":0.150,"max_ra":3.2,"min_dim":3.0, "max_dim":100.0,"max_ar":8, "std":"ISO 4759"},
}
MATERIAL_COMPAT = {
    ("steel","steel"):True,("steel","aluminum"):True,("steel","composite"):True,
    ("aluminum","aluminum"):True,("aluminum","composite"):True,("titanium","steel"):True,
    ("titanium","composite"):True,("plastic","plastic"):True,("plastic","aluminum"):True,
    ("steel","plastic"):False,("titanium","aluminum"):False,("composite","plastic"):False,
}
JOINT_CL_LIMITS = {"fixed":0.10,"revolute":0.30,"prismatic":0.20,"contact":0.40}

# Floating-point epsilon: ensures that values exactly equal to a limit always PASS
_EPS = 1e-9

def material_compatible(m1_code,m2_code):
    m1=INV_MAT.get(int(m1_code),"steel"); m2=INV_MAT.get(int(m2_code),"steel")
    return MATERIAL_COMPAT.get((m1,m2),MATERIAL_COMPAT.get((m2,m1),True))

# ─────────────────────────────────────────────────────────────────────────────
# VIOLATION DATACLASS
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Violation:
    node:Any; rule_id:str; category:str; rule:str; value:float; limit:str
    severity:str; suggestion:str=""; passed:bool=False
    def to_dict(self): return asdict(self)

# ─────────────────────────────────────────────────────────────────────────────
# RULE ENGINE
# ─────────────────────────────────────────────────────────────────────────────
def _check_tol(nominal,tolerance,actual):
    lo=nominal-abs(tolerance); hi=nominal+abs(tolerance); return (lo<=actual<=hi),lo,hi

def run_rule_engine(node_features,nominal_dims=None,nx_graph=None):
    violations,passed_checks=[],[]
    def _record(v): (passed_checks if v.passed else violations).append(v)
    for i,feat in enumerate(node_features):
        dim_x,dim_y,dim_z,tol,mat_code,type_code,weight,roughness=(float(f) for f in feat)
        nom=nominal_dims[i] if nominal_dims and i<len(nominal_dims) else (dim_x,dim_y,dim_z)
        nom_x,nom_y,nom_z=nom
        comp=INV_COMP.get(int(type_code),"housing"); cr=COMPONENT_RULES.get(comp,COMPONENT_RULES["housing"])
        for d,ax in ((dim_x,"X"),(dim_y,"Y"),(dim_z,"Z")):
            ok=d>0
            _record(Violation(node=i,rule_id=f"DIM-POSITIVE-{ax}",category="DIM",
                rule=f"Dim {ax}>0",value=round(d,3),limit=">0mm",
                severity="CRITICAL" if not ok else "INFO",
                suggestion="Invalid dimension" if not ok else "OK",passed=ok))
        for actual,nominal,ax in [(dim_x,nom_x,"X"),(dim_y,nom_y,"Y"),(dim_z,nom_z,"Z")]:
            ok,lo,hi=_check_tol(nominal,tol,actual)
            deviation=abs(actual-nominal)
            sev="INFO" if ok else ("WARNING" if deviation<=tol*1.20 else "CRITICAL")
            _record(Violation(node=i,rule_id=f"TOL-DIM-{ax}",category="TOL",
                rule=f"ISO 2768 Dim {ax}",value=round(actual,4),limit=f"[{lo:.4f},{hi:.4f}]",
                severity=sev,suggestion="Tolerance exceeded" if not ok else "Within tolerance",passed=ok))
        ok=tol<=cr["max_tol"]+_EPS
        sev="INFO" if ok else ("WARNING" if tol<=cr["max_tol"]*1.20+_EPS else "CRITICAL")
        _record(Violation(node=i,rule_id="TOL-COMP",category="COMP",rule=f"{comp} tolerance",
            value=round(tol,5),limit=f"<={cr['max_tol']}",severity=sev,
            suggestion="Tolerance too large" if not ok else "OK",passed=ok))
        ok=roughness<=cr["max_ra"]+_EPS
        _record(Violation(node=i,rule_id="SURF-RA",category="SURF",rule="Surface roughness",
            value=round(roughness,3),limit=f"<={cr['max_ra']}",
            severity="WARNING" if not ok else "INFO",
            suggestion="Surface too rough" if not ok else "Surface OK",passed=ok))
        for d,ax in [(dim_x,"X"),(dim_y,"Y"),(dim_z,"Z")]:
            ok=cr["min_dim"]-_EPS<=d<=cr["max_dim"]+_EPS
            sev="INFO" if ok else ("CRITICAL" if d<cr["min_dim"]*0.5 or d>cr["max_dim"]*1.5 else "WARNING")
            _record(Violation(node=i,rule_id=f"DIM-BOUND-{ax}",category="DIM",rule="Dimension bounds",
                value=round(d,3),limit=f"{cr['min_dim']}-{cr['max_dim']}",severity=sev,
                suggestion="Out of bounds" if not ok else "OK",passed=ok))
        ds=sorted([dim_x,dim_y,dim_z]); ratio=ds[-1]/max(ds[0],1e-6)
        ok=ratio<=cr["max_ar"]+_EPS
        _record(Violation(node=i,rule_id="DFM-AR",category="DFM",rule="Aspect ratio",
            value=round(ratio,2),limit=f"<={cr['max_ar']}",severity="WARNING" if not ok else "INFO",
            suggestion="Aspect ratio high" if not ok else "OK",passed=ok))
        ok=weight<=STANDARDS["WEIGHT_LIMIT"]+_EPS
        sev="INFO" if ok else ("WARNING" if weight<=STANDARDS["WEIGHT_LIMIT"]*1.25+_EPS else "CRITICAL")
        _record(Violation(node=i,rule_id="DFM-WT",category="DFM",rule="Weight limit",
            value=round(weight,1),limit=f"<={STANDARDS['WEIGHT_LIMIT']}",severity=sev,
            suggestion="Weight too high" if not ok else "OK",passed=ok))
    if nx_graph is not None:
        n=len(node_features)
        for u,v_node,data in nx_graph.edges(data=True):
            if not(0<=u<n and 0<=v_node<n): continue
            m1,m2=int(node_features[u][4]),int(node_features[v_node][4])
            ok=material_compatible(m1,m2)
            _record(Violation(node=f"e({u}->{v_node})",rule_id="MAT-COMPAT",category="MAT",
                rule="Material compatibility",value=0,limit="compatible",
                severity="CRITICAL" if not ok else "INFO",
                suggestion="Galvanic risk" if not ok else "OK",passed=ok))
            cl=float(data.get("clearance",0.0)); jt=str(data.get("joint_type","contact"))
            lim=JOINT_CL_LIMITS.get(jt,0.40); ok=cl<=lim+_EPS
            sev="CRITICAL" if cl>STANDARDS["CL_CRITICAL"]+_EPS else ("WARNING" if not ok else "INFO")
            _record(Violation(node=f"e({u}->{v_node})",rule_id="JNTI-CL",category="JNTI",
                rule="Joint clearance",value=round(cl,4),limit=f"<={lim}",severity=sev,
                suggestion="Clearance too large" if not ok else "OK",passed=ok))
    return violations,passed_checks

# ─────────────────────────────────────────────────────────────────────────────
# SCORE FUSION
# ─────────────────────────────────────────────────────────────────────────────
def fuse_scores(ml_prob_noncompliant,violations):
    n_crit=sum(1 for v in violations if v.severity=="CRITICAL")
    n_warn=sum(1 for v in violations if v.severity=="WARNING")
    n_info=sum(1 for v in violations if v.severity=="INFO")
    rule_score=min(100.0,n_crit*40.0+n_warn*15.0+n_info*5.0)
    ml_score=float(ml_prob_noncompliant)*100.0
    if n_crit>0:
        fused=max(cfg.NONCOMPLIANT_THRESH,0.30*ml_score+0.70*rule_score)
        verdict="NON-COMPLIANT"; override_reason=f"{n_crit} CRITICAL → auto NON-COMPLIANT override"
    else:
        fused=0.35*ml_score+0.65*rule_score; override_reason=None
        if fused>=cfg.NONCOMPLIANT_THRESH: verdict="NON-COMPLIANT"
        elif fused>=cfg.REVIEW_THRESH:     verdict="REVIEW NEEDED"
        else:                              verdict="COMPLIANT"
    explanation=(f"GNN: {ml_score:.1f}/100 non-compliance prob. "
        f"Rule engine: {n_crit} CRITICAL, {n_warn} WARNING, {n_info} INFO → {rule_score:.1f}/100. "
        f"{'Override applied.' if override_reason else f'Balanced fusion = {fused:.1f}/100.'}")
    return {"ml_score":round(ml_score,2),"rule_score":round(rule_score,2),
        "fused_score":round(fused,2),"verdict":verdict,"critical":n_crit,"warnings":n_warn,"info":n_info,
        "override_reason":override_reason,"explanation":explanation}

# ─────────────────────────────────────────────────────────────────────────────
# DATA GENERATOR
# ─────────────────────────────────────────────────────────────────────────────
def _add_edges_to_graph(G,n_components,target_label):
    jt_opts=list(JOINT_CL_LIMITS.keys())
    for i in range(n_components-1):
        jt=np.random.choice(jt_opts); lim=JOINT_CL_LIMITS[jt]
        if target_label==0:   cl=np.random.uniform(1e-3,lim*0.7)
        elif target_label==1: cl=np.random.uniform(lim*0.9,lim*1.15)
        else:                 cl=np.random.uniform(lim*1.2,lim*3.0)
        G.add_edge(i,i+1,clearance=cl,joint_type=jt)

def graph_to_pyg(G,feats,label):
    raw_x=torch.tensor(feats,dtype=torch.float); x=raw_x.clone()
    jt_map={"fixed":0,"revolute":1,"prismatic":2,"contact":3}
    edges=list(G.edges(data=True))
    if edges:
        src=[u for u,v,d in edges]+[v for u,v,d in edges]
        dst=[v for u,v,d in edges]+[u for u,v,d in edges]
        ei=torch.tensor([src,dst],dtype=torch.long)
        e_feats=[[float(d.get("clearance",0.2)),float(jt_map.get(d.get("joint_type","contact"),3))]
                 for u,v,d in edges]
        edge_attr=torch.tensor(e_feats+e_feats,dtype=torch.float)
    else:
        ei=torch.zeros((2,0),dtype=torch.long); edge_attr=torch.zeros((0,2),dtype=torch.float)
    return Data(x=x,edge_index=ei,edge_attr=edge_attr,
                y=torch.tensor([label],dtype=torch.long),raw_x=raw_x)

def generate_cad_graph(n_components,is_compliant,seed=42):
    rng=np.random.default_rng(seed)
    comp_keys=list(COMPONENT_TYPES.keys()); mat_keys=list(MATERIAL_CODES.keys())
    label=0 if is_compliant else 2; node_features=[]
    for _ in range(n_components):
        comp=rng.choice(comp_keys); mat=rng.choice(mat_keys); cr=COMPONENT_RULES[comp]
        if label==0:
            tol=rng.uniform(cr["max_tol"]*0.1,cr["max_tol"]*0.7)
            roughness=rng.uniform(0.2,cr["max_ra"]*0.7)
            dim_x=rng.uniform(cr["min_dim"]*1.1,cr["max_dim"]*0.9)
            dim_y=rng.uniform(cr["min_dim"]*1.1,cr["max_dim"]*0.9)
            dim_z=rng.uniform(cr["min_dim"]*1.1,cr["max_dim"]*0.9)
        else:
            tol=rng.uniform(cr["max_tol"]*1.5,cr["max_tol"]*4.0)
            roughness=rng.uniform(cr["max_ra"]*1.5,cr["max_ra"]*4.0)
            dim_x=rng.uniform(cr["min_dim"]*0.1,cr["min_dim"]*0.8)
            dim_y=rng.uniform(cr["min_dim"]*0.1,cr["min_dim"]*0.8)
            dim_z=rng.uniform(cr["min_dim"]*0.1,cr["min_dim"]*0.8)
        weight=dim_x*dim_y*dim_z*1e-4
        node_features.append([dim_x,dim_y,dim_z,tol,
            float(MATERIAL_CODES[mat]),float(COMPONENT_TYPES[comp]),weight,roughness])
    feats=np.array(node_features,dtype=np.float32); G=nx.Graph(); G.add_nodes_from(range(n_components))
    _add_edges_to_graph(G,n_components,label)
    return G,feats,label

def generate_dataset():
    dataset=[]; rng=np.random.default_rng(cfg.SEED)
    for i in range(cfg.N_COMPLIANT):
        n_comp=int(rng.integers(cfg.MIN_COMPONENTS,cfg.MAX_COMPONENTS+1))
        G,feats,_=generate_cad_graph(n_comp,is_compliant=True,seed=cfg.SEED+i)
        dataset.append(graph_to_pyg(G,feats,0))
    for i in range(cfg.N_REVIEW_NEEDED):
        n_comp=int(rng.integers(cfg.MIN_COMPONENTS,cfg.MAX_COMPONENTS+1))
        rng2=np.random.default_rng(cfg.SEED+2000+i)
        comp_keys=list(COMPONENT_TYPES.keys()); mat_keys=list(MATERIAL_CODES.keys()); nf=[]
        for _ in range(n_comp):
            comp=rng2.choice(comp_keys); mat=rng2.choice(mat_keys); cr=COMPONENT_RULES[comp]
            tol=rng2.uniform(cr["max_tol"]*0.8,cr["max_tol"]*1.6)
            roughness=rng2.uniform(cr["max_ra"]*0.8,cr["max_ra"]*1.5)
            dim_x=rng2.uniform(cr["min_dim"]*0.7,cr["max_dim"]*1.1)
            dim_y=rng2.uniform(cr["min_dim"]*0.7,cr["max_dim"]*1.1)
            dim_z=rng2.uniform(cr["min_dim"]*0.7,cr["max_dim"]*1.1)
            weight=dim_x*dim_y*dim_z*1e-4
            nf.append([dim_x,dim_y,dim_z,tol,
                float(MATERIAL_CODES[mat]),float(COMPONENT_TYPES[comp]),weight,roughness])
        feats=np.array(nf,dtype=np.float32); G_nx=nx.Graph(); G_nx.add_nodes_from(range(n_comp))
        _add_edges_to_graph(G_nx,n_comp,1); dataset.append(graph_to_pyg(G_nx,feats,1))
    for i in range(cfg.N_NONCOMPLIANT):
        n_comp=int(rng.integers(cfg.MIN_COMPONENTS,cfg.MAX_COMPONENTS+1))
        G,feats,_=generate_cad_graph(n_comp,is_compliant=False,seed=cfg.SEED+5000+i)
        dataset.append(graph_to_pyg(G,feats,2))
    return dataset

# ─────────────────────────────────────────────────────────────────────────────
# GNN MODEL
# ─────────────────────────────────────────────────────────────────────────────
class GNNEncoder(nn.Module):
    def __init__(self,in_ch=cfg.IN_CHANNELS,h=cfg.HIDDEN_DIM,lat=cfg.LATENT_DIM,drop=cfg.DROPOUT):
        super().__init__(); self.drop=drop
        self.c1=TransformerConv(in_ch,h,edge_dim=2); self.c2=TransformerConv(h,h,edge_dim=2)
        self.c3=TransformerConv(h,lat,edge_dim=2)
        self.b1,self.b2,self.b3=nn.BatchNorm1d(h),nn.BatchNorm1d(h),nn.BatchNorm1d(lat)
    def forward(self,x,ei,batch,edge_attr=None):
        x=F.relu(self.b1(self.c1(x,ei,edge_attr))); x=F.dropout(x,self.drop,self.training)
        x=F.relu(self.b2(self.c2(x,ei,edge_attr))); x=F.dropout(x,self.drop,self.training)
        x=F.relu(self.b3(self.c3(x,ei,edge_attr))); return global_mean_pool(x,batch)

class GNRValModel(nn.Module):
    def __init__(self,in_ch=cfg.IN_CHANNELS,h=cfg.HIDDEN_DIM,lat=cfg.LATENT_DIM,drop=cfg.DROPOUT):
        super().__init__()
        self.encoder=GNNEncoder(in_ch,h,lat,drop)
        self.decoder=nn.Sequential(nn.Linear(lat,h),nn.ReLU(),nn.Linear(h,in_ch))
        self.classifier=nn.Sequential(nn.Linear(lat,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,3))
    def forward(self,x,ei,batch,edge_attr=None):
        z=self.encoder(x,ei,batch,edge_attr); return self.classifier(z),z,self.decoder(z)

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def _eval_loader(model,loader):
    model.eval(); all_preds,all_labels,all_probs=[],[],[]
    with torch.no_grad():
        for batch in loader:
            batch=batch.to(DEVICE)
            logits,_,_=model(batch.x,batch.edge_index,batch.batch,batch.edge_attr)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(batch.y.cpu().tolist())
            all_probs.extend(F.softmax(logits,dim=1)[:,2].cpu().tolist())
    return accuracy_score(all_labels,all_preds)*100,all_preds,all_labels,all_probs

def train_model_fn(train_data,val_data,pretrained_model_path=None):
    train_loader=DataLoader(train_data,batch_size=cfg.BATCH_SIZE,shuffle=True)
    val_loader=DataLoader(val_data,batch_size=cfg.BATCH_SIZE,shuffle=False)
    train_labels=[int(d.y.item()) for d in train_data]
    counts=[max(train_labels.count(i),1) for i in range(3)]
    weights=torch.tensor([1.0/c for c in counts],dtype=torch.float).to(DEVICE)
    weights=weights/weights.sum()*3.0
    model=GNRValModel().to(DEVICE)

    if pretrained_model_path and os.path.exists(pretrained_model_path):
        try:
            state_dict=torch.load(pretrained_model_path,map_location=DEVICE)
            model.load_state_dict(state_dict,strict=True)
            log.info(f"Loaded pretrained real-world model from: {pretrained_model_path}")
        except Exception as e:
            log.warning(f"Could not load pretrained model from {pretrained_model_path}: {e}")

    optimizer=torch.optim.Adam(model.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    criterion=nn.CrossEntropyLoss(weight=weights)
    best_acc,best_state=0.0,None
    pbar=st.progress(0,text=f"Training GNN... Epoch 0/{cfg.EPOCHS}")
    for ep in range(1,cfg.EPOCHS+1):
        model.train()
        for batch in train_loader:
            batch=batch.to(DEVICE); optimizer.zero_grad()
            logits,_,recon=model(batch.x,batch.edge_index,batch.batch,batch.edge_attr)
            loss=criterion(logits,batch.y)+cfg.RECON_WEIGHT*F.mse_loss(
                recon,global_mean_pool(batch.x,batch.batch))
            loss.backward(); optimizer.step()
        if ep%cfg.LOG_EVERY==0:
            val_acc,_,_,_=_eval_loader(model,val_loader)
            if val_acc>best_acc: best_acc,best_state=val_acc,deepcopy(model.state_dict())
        pbar.progress(ep/cfg.EPOCHS,text=f"Training GNN... Epoch {ep}/{cfg.EPOCHS} | Best Val Acc: {best_acc:.1f}%")
    if best_state: model.load_state_dict(best_state)
    pbar.empty()
    return model

# ─────────────────────────────────────────────────────────────────────────────
# INPUT VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
def validate_input_data(input_data):
    validated_data,input_warnings=[],[]
    allowed_materials=set(MATERIAL_CODES.keys()); allowed_types=set(COMPONENT_TYPES.keys())
    for idx,comp in enumerate(input_data):
        item=dict(comp)
        for key in ["actual_x","actual_y","actual_z"]:
            if key not in item: raise ValueError(f"Component {idx}: missing field '{key}'")
            item[key]=float(item[key])
        item["tolerance"]=abs(float(item.get("tolerance",0.1)))
        item["roughness"]=abs(float(item.get("roughness",1.6)))
        item["material"]=str(item.get("material","steel")).lower().strip()
        item["type"]=str(item.get("type","housing")).lower().strip()
        if item["material"] not in allowed_materials:
            input_warnings.append(f"Component {idx}: unknown material → replaced with 'steel'.")
            item["material"]="steel"
        if item["type"] not in allowed_types:
            input_warnings.append(f"Component {idx}: unknown type → replaced with 'housing'.")
            item["type"]="housing"
        if "nominal_dims" in item and item["nominal_dims"] is not None:
            item["nominal_dims"]=[float(x) for x in item["nominal_dims"]]
        else:
            item["nominal_dims"]=[item["actual_x"],item["actual_y"],item["actual_z"]]
            input_warnings.append(f"Component {idx}: nominal_dims missing, using actual dims.")
        validated_data.append(item)
    return validated_data,input_warnings

def get_top_violation_reasons(violations,top_k=3):
    sev_rank={"CRITICAL":3,"WARNING":2,"INFO":1}
    ranked=sorted(violations,key=lambda v:sev_rank.get(v.severity,0),reverse=True)
    reasons,seen=[],set()
    for v in ranked:
        if v.rule_id in seen: continue
        reasons.append({"node":v.node,"rule_id":v.rule_id,"severity":v.severity,
                        "category":v.category,"message":v.suggestion})
        seen.add(v.rule_id)
        if len(reasons)>=top_k: break
    return reasons

# ─────────────────────────────────────────────────────────────────────────────
# DETERMINISTIC INFERENCE GRAPH BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def build_inference_graph_from_components(input_data):
    """
    Build a deterministic nx.Graph from real validated component data.
    Same input_data always produces the same graph — zero randomness.

    Edge topology:
      1. Sequential backbone: every adjacent pair (i, i+1) is connected.
      2. Hierarchy edges: non-adjacent pairs connected when their component
         types have a known assembly relationship (shaft→bearing, etc.).

    Joint type: derived deterministically from the component-type pair.
    Clearance:  derived from the average of the two components' tolerance
                values, clamped to 70 % of the joint's clearance limit —
                a conservative, repeatable estimate from real CAD intent.
    """
    n = len(input_data)
    G = nx.Graph()
    G.add_nodes_from(range(n))

    # Deterministic joint-type lookup by component-type pair
    _JT_MAP = {
        ("shaft",    "bearing"):  "revolute",
        ("shaft",    "gear"):     "revolute",
        ("gear",     "bearing"):  "revolute",
        ("housing",  "bearing"):  "fixed",
        ("housing",  "shaft"):    "fixed",
        ("housing",  "fastener"): "fixed",
        ("bracket",  "fastener"): "fixed",
        ("gear",     "fastener"): "fixed",
        ("shaft",    "fastener"): "fixed",
        ("bearing",  "fastener"): "fixed",
        ("housing",  "bracket"):  "contact",
        ("bracket",  "gear"):     "contact",
    }

    # Assembly hierarchy: types that u naturally connects to
    _HIER = {
        "shaft":    {"bearing", "fastener", "gear"},
        "housing":  {"bearing", "fastener", "shaft"},
        "gear":     {"fastener", "bearing", "shaft"},
        "bracket":  {"fastener", "housing"},
        "bearing":  {"fastener", "shaft"},
        "fastener": set(),
    }

    def _joint_type(t1, t2):
        return _JT_MAP.get((t1, t2), _JT_MAP.get((t2, t1), "contact"))

    def _clearance(comp_i, comp_j, jt):
        # Real tolerances from user input drive the clearance estimate
        tol_i = abs(float(comp_i.get("tolerance", 0.1)))
        tol_j = abs(float(comp_j.get("tolerance", 0.1)))
        limit = JOINT_CL_LIMITS.get(jt, 0.40)
        cl = min((tol_i + tol_j) / 2.0, limit * 0.70)
        return round(max(1e-4, cl), 6)

    # 1. Sequential backbone — always present, preserves graph connectivity
    for i in range(n - 1):
        t1 = str(input_data[i].get("type", "housing")).lower().strip()
        t2 = str(input_data[i + 1].get("type", "housing")).lower().strip()
        jt = _joint_type(t1, t2)
        cl = _clearance(input_data[i], input_data[i + 1], jt)
        G.add_edge(i, i + 1, clearance=cl, joint_type=jt)

    # 2. Hierarchy-based non-adjacent edges
    for i, ci in enumerate(input_data):
        t1 = str(ci.get("type", "housing")).lower().strip()
        for j, cj in enumerate(input_data):
            if j <= i or G.has_edge(i, j):
                continue
            t2 = str(cj.get("type", "housing")).lower().strip()
            if t2 in _HIER.get(t1, set()) or t1 in _HIER.get(t2, set()):
                jt = _joint_type(t1, t2)
                cl = _clearance(ci, cj, jt)
                G.add_edge(i, j, clearance=cl, joint_type=jt)

    return G


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
def run_inference(input_data,nominal_dims=None,trained_model=None,node_scaler=None):
    if not input_data: return {"error":"No component data provided."}
    input_data,input_warnings=validate_input_data(input_data)
    if nominal_dims is None: nominal_dims=[tuple(item["nominal_dims"]) for item in input_data]
    feats_list=[]
    for item in input_data:
        ax,ay,az=float(item["actual_x"]),float(item["actual_y"]),float(item["actual_z"])
        tol=abs(float(item.get("tolerance",0.1))); ra=abs(float(item.get("roughness",1.6)))
        feats_list.append([ax,ay,az,tol,
            float(MATERIAL_CODES.get(item.get("material","steel"),0)),
            float(COMPONENT_TYPES.get(item.get("type","housing"),2)),ax*ay*az*1e-4,ra])
    node_features=np.array(feats_list,dtype=np.float32)
    gnn_nf=node_scaler.transform(node_features) if node_scaler else node_features.copy()
    G_nx=build_inference_graph_from_components(input_data)
    if trained_model is None: return {"error":"Model not trained yet."}
    trained_model.eval()
    pyg=graph_to_pyg(G_nx,gnn_nf,0)
    x=pyg.x.to(DEVICE); ei=pyg.edge_index.to(DEVICE); ea=pyg.edge_attr.to(DEVICE)
    bv=torch.zeros(x.size(0),dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits,_,recon=trained_model(x,ei,bv,ea)
        probs=F.softmax(logits/1.5,dim=1).cpu().numpy()[0]
    pred_class=int(logits.argmax(1).item())
    lmap={0:"compliant",1:"review-needed",2:"non-compliant"}
    lfull={0:"Compliant",1:"Review-Needed",2:"Non-Compliant"}
    violations,passed=run_rule_engine(node_features,nominal_dims,G_nx)
    top_reasons=get_top_violation_reasons(violations,top_k=3)
    scores=fuse_scores(float(probs[2]),violations)
    return {"ml_prediction":lmap.get(pred_class,"unknown"),"ml_pred_class":pred_class,
        "ml_confidence":round(float(max(probs))*100,2),
        "ml_probs":{lfull[i]:round(float(probs[i])*100,2) for i in range(3)},
        "reconstruction_error":round(float(F.mse_loss(recon[0],x.mean(0)).item()),4),
        "violations":violations,"passed_checks":passed,"scores":scores,
        "n_components":len(node_features),"n_violations":len(violations),"n_passed":len(passed),
        "input_warnings":input_warnings,"top_reasons":top_reasons}

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def _compute_decision_basis(res):
    sc=res.get("scores",{}); n_crit=sc.get("critical",0); recon=float(res.get("reconstruction_error",0.0))
    basis="Rule-Dominant Safety Override" if n_crit>0 else "Balanced Rule + GNN Assessment"
    if recon>=15.0: basis+=" | Elevated Model Uncertainty"
    return basis

def _get_release_recommendation(verdict):
    m={"COMPLIANT":{"label":"Release Approved","icon":"✅","color":"green"},
       "REVIEW NEEDED":{"label":"Engineering Review Required","icon":"⚠️","color":"orange"},
       "NON-COMPLIANT":{"label":"Production Hold","icon":"🚫","color":"red"}}
    return m.get(verdict,{"label":"Under Review","icon":"🔍","color":"gray"})

def _get_top3_remediations(res):
    v_list = res.get("violations", [])
    actions = []
    cat_seen = set()

    remap = {
        "TOL": "Recalibrate CNC tooling or tighten process control to stay within tolerance budget.",
        "SURF": "Switch to fine-grinding/superfinishing; verify grinding wheel grit and feed rate.",
        "MAT": "Apply isolating coating or substitute galvanic-neutral alloy per ASTM B117.",
        "DIM": "Revise CAD nominal or adjust process per ASME Y14.5 GD&T drawing callouts.",
        "DFM": "Re-evaluate geometry for manufacturability; reduce aspect ratio via topology optimisation.",
        "JNTI": "Adjust fit specification (H7/g6 etc.) and verify with go/no-go gauges.",
    }

    for v in v_list:
        cat = str(v.category).upper()
        for key, msg in remap.items():
            if key in cat and key not in cat_seen:
                actions.append(msg)
                cat_seen.add(key)
                break
        if len(actions) >= 3:
            break

    if float(res.get("reconstruction_error", 0.0)) >= 15.0 and len(actions) < 3:
        actions.append("Simplify CAD features to improve GNN prediction confidence.")

    if not actions:
        actions.append("No corrective actions required. Maintain current process settings.")

    return actions[:3]

def render_cad_3d_model(components, violations=None):
    """
    3D Digital Twin Preview — clean wireframe cuboids.

    Per component, three traces are added:
      1. Scatter3d lines  — 12 explicit box edges, bold and color-coded.
         No triangle geometry → zero stretched-triangle artifacts.
      2. Mesh3d           — correct 12-triangle closed cuboid at opacity=0.08.
         Opacity is intentionally very low so triangle seam lines are
         invisible; the trace exists only to give subtle depth/fill cues.
      3. Scatter3d text   — label just above the top face.

    Violated components → red edges/fill.
    Compliant components → type-based color.
    Dark theme preserved throughout.
    """
    if not components:
        return None

    violated_nodes = set()
    if violations:
        violated_nodes = {v.node for v in violations if isinstance(v.node, int)}

    type_colors = {
        "housing":  "#3b82f6",
        "shaft":    "#10b981",
        "bearing":  "#f59e0b",
        "gear":     "#8b5cf6",
        "bracket":  "#94a3b8",
        "fastener": "#06b6d4",
    }

    # Correct closed-cuboid triangle indices (12 triangles, outward normals)
    # Vertex layout:  0=x0y0z0  1=x1y0z0  2=x1y1z0  3=x0y1z0
    #                 4=x0y0z1  5=x1y0z1  6=x1y1z1  7=x0y1z1
    _FI = [0,0, 4,4, 0,0, 2,2, 0,0, 1,1]
    _FJ = [1,2, 5,6, 1,5, 3,7, 3,7, 2,6]
    _FK = [2,3, 6,7, 5,4, 7,6, 7,4, 6,5]

    fig = go.Figure()
    spacing = 0.0

    for idx, comp in enumerate(components):
        sx = max(float(comp.get("actual_x", 10.0)), 1.0)
        sy = max(float(comp.get("actual_y", 10.0)), 1.0)
        sz = max(float(comp.get("actual_z", 10.0)), 1.0)

        comp_type = str(comp.get("type", "housing")).lower().strip()
        base_color = type_colors.get(comp_type, "#94a3b8")
        color = "#ef4444" if idx in violated_nodes else base_color

        # Position: lay components along X axis, all starting at Y=0, Z=0
        x0 = spacing
        x1 = spacing + sx
        y0, y1 = -sy / 2.0,  sy / 2.0
        z0, z1 =  0.0,        sz
        cx = (x0 + x1) / 2.0
        cy = 0.0
        spacing = x1 + max(10.0, sx * 0.15)

        hover = (f"<b>N{idx} · {comp_type}</b><br>"
                 f"X: {sx:.2f} mm<br>Y: {sy:.2f} mm<br>Z: {sz:.2f} mm"
                 f"<extra></extra>")

        # ── 1. Wireframe edges ────────────────────────────────────────────
        # 12 edges of the cuboid, each represented as [start, end, None]
        # so Plotly draws separate line segments with no connecting diagonal.
        _edges = [
            # Bottom face (z0)
            (x0,y0,z0, x1,y0,z0), (x1,y0,z0, x1,y1,z0),
            (x1,y1,z0, x0,y1,z0), (x0,y1,z0, x0,y0,z0),
            # Top face (z1)
            (x0,y0,z1, x1,y0,z1), (x1,y0,z1, x1,y1,z1),
            (x1,y1,z1, x0,y1,z1), (x0,y1,z1, x0,y0,z1),
            # Vertical pillars
            (x0,y0,z0, x0,y0,z1), (x1,y0,z0, x1,y0,z1),
            (x1,y1,z0, x1,y1,z1), (x0,y1,z0, x0,y1,z1),
        ]
        lx, ly, lz = [], [], []
        for ax, ay, az, bx, by, bz in _edges:
            lx += [ax, bx, None]
            ly += [ay, by, None]
            lz += [az, bz, None]

        fig.add_trace(go.Scatter3d(
            x=lx, y=ly, z=lz,
            mode="lines",
            line=dict(color=color, width=3),
            name=f"N{idx} {comp_type}",
            hovertemplate=hover,
            legendgroup=f"c{idx}",
        ))

        # ── 2. Transparent face fill (depth cue only) ─────────────────────
        verts = np.array([
            [x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],
            [x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1],
        ])
        fig.add_trace(go.Mesh3d(
            x=verts[:,0], y=verts[:,1], z=verts[:,2],
            i=_FI, j=_FJ, k=_FK,
            color=color,
            opacity=0.08,          # low enough that triangle seams vanish
            flatshading=False,
            showlegend=False,
            hoverinfo="skip",
            legendgroup=f"c{idx}",
        ))

        # ── 3. Label just above top face ──────────────────────────────────
        fig.add_trace(go.Scatter3d(
            x=[cx], y=[cy], z=[z1 + max(2.0, sz * 0.06)],
            mode="text",
            text=[f"<b>N{idx}</b><br>{comp_type}"],
            textfont=dict(size=9, color="white"),
            showlegend=False,
            hoverinfo="skip",
            legendgroup=f"c{idx}",
        ))

    fig.update_layout(
        paper_bgcolor="#0a0e1a",
        plot_bgcolor="#0a0e1a",
        font=dict(color="white"),
        scene=dict(
            bgcolor="#0a0e1a",
            xaxis=dict(title="X (mm)", backgroundcolor="#0a0e1a",
                       gridcolor="#1e293b", showbackground=True,
                       zerolinecolor="#334155", color="#64748b"),
            yaxis=dict(title="Y (mm)", backgroundcolor="#0a0e1a",
                       gridcolor="#1e293b", showbackground=True,
                       zerolinecolor="#334155", color="#64748b"),
            zaxis=dict(title="Z (mm)", backgroundcolor="#0a0e1a",
                       gridcolor="#1e293b", showbackground=True,
                       zerolinecolor="#334155", color="#64748b"),
            camera=dict(eye=dict(x=1.7, y=1.5, z=1.2)),
            aspectmode="auto",
        ),
        legend=dict(bgcolor="#0f172a", bordercolor="#334155",
                    borderwidth=1, font=dict(color="white")),
        margin=dict(l=0, r=0, t=20, b=0),
        height=650,
    )
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# STEP PARSER
# ─────────────────────────────────────────────────────────────────────────────
def parse_step_text(content: str) -> List[Dict[str, Any]]:
    """
    Single canonical STEP parser for the Streamlit app.
    Parses ISO 10303-21 ASCII STEP files (AP203/AP214).
    Extracts CARTESIAN_POINT coordinates → per-solid bounding boxes.
    Falls back to LENGTH_MEASURE values or stable index-based defaults.
    No pythonocc, OCC, or native CAD library required.
    """
    lengths = [float(x) for x in re.findall(r"LENGTH_MEASURE\s*\(\s*([\d.]+)\s*\)", content)]
    cart_pts = re.findall(
        r"CARTESIAN_POINT\s*\([^,]*,\s*\(\s*([\d.\-]+)\s*,\s*([\d.\-]+)\s*,\s*([\d.\-]+)\s*\)\s*\)",
        content)
    face_count = len(re.findall(r"ADVANCED_FACE", content))
    n_comp = max(1, min(12, face_count // 40 + len(cart_pts) // 5 + 1))

    # Stable ordered lists — deterministic cycling by component index
    mat_keys  = list(MATERIAL_CODES.keys())    # steel, aluminum, plastic, titanium, composite
    type_keys = list(COMPONENT_TYPES.keys())   # shaft, bearing, housing, bracket, fastener, gear

    extracted = []
    for i in range(n_comp):
        if i < len(cart_pts):
            try:
                dx = abs(float(cart_pts[i][0])) or 50.0
                dy = abs(float(cart_pts[i][1])) or 50.0
                dz = abs(float(cart_pts[i][2])) or 20.0
            except Exception:
                dx, dy, dz = 50.0, 50.0, 20.0
        else:
            if lengths:
                dx = lengths[i % len(lengths)]
                dy = lengths[(i + 1) % len(lengths)]
                dz = lengths[(i + 2) % len(lengths)]
            else:
                dx = 50.0 + (i % 5) * 10.0
                dy = 50.0 + ((i + 1) % 5) * 10.0
                dz = 20.0 + (i % 3) * 5.0

        comp_type = type_keys[i % len(type_keys)]
        comp_mat  = mat_keys[i % len(mat_keys)]

        extracted.append({
            "actual_x":     round(max(1.0, dx), 3),
            "actual_y":     round(max(1.0, dy), 3),
            "actual_z":     round(max(1.0, dz), 3),
            "tolerance":    0.10,    # ISO 2768 medium — safe conservative default
            "material":     comp_mat,
            "type":         comp_type,
            "roughness":    1.6,     # Ra 1.6 µm — mating surface standard default
            "nominal_dims": [round(max(1.0, dx), 3),
                             round(max(1.0, dy), 3),
                             round(max(1.0, dz), 3)],
        })
    return extracted


def parse_step_file(content: str) -> List[Dict[str, Any]]:
    """Alias for parse_step_text — retained for call-site compatibility."""
    return parse_step_text(content)

# ─────────────────────────────────────────────────────────────────────────────
# ASSEMBLY GRAPH
# ─────────────────────────────────────────────────────────────────────────────
_ASSEMBLY_HIER={"shaft":["bearing","fastener","gear"],"housing":["bearing","seal","gasket"],
    "bearing":["fastener","seal"],"gear":["fastener"],"bracket":["fastener","seal"]}

def _build_assembly_graph(components):
    n=len(components); G=nx.DiGraph(); G.add_nodes_from(range(n))
    labels={i:f"N{i}\n{components[i].get('type','part')}" for i in range(n)}
    for i,comp in enumerate(components):
        for j,other in enumerate(components):
            if i!=j and other.get("type","").lower() in _ASSEMBLY_HIER.get(comp.get("type",""),[]): G.add_edge(i,j)
    coords=np.array([[c.get("actual_x",0),c.get("actual_y",0),c.get("actual_z",0)] for c in components],dtype=float)
    isolated=[v for v in G.nodes() if G.degree(v)==0]
    for v in isolated:
        dists=np.linalg.norm(coords-coords[v],axis=1); dists[v]=np.inf; G.add_edge(v,int(np.argmin(dists)))
    if nx.number_weakly_connected_components(G)>1:
        for i in range(n-1):
            if not(G.has_edge(i,i+1) or G.has_edge(i+1,i)): G.add_edge(i,i+1)
    return G,labels

# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE (cached)
# ─────────────────────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_pipeline():
    set_seed()

    dataset=generate_dataset()
    all_labels=[int(d.y.item()) for d in dataset]
    train_idx,temp_idx=train_test_split(list(range(len(dataset))),
        test_size=(1.0-cfg.TRAIN_RATIO),stratify=all_labels,random_state=cfg.SEED)
    temp_labels=[all_labels[i] for i in temp_idx]
    val_idx,test_idx=train_test_split(temp_idx,test_size=0.5,stratify=temp_labels,random_state=cfg.SEED)

    train_data=[dataset[i] for i in train_idx]
    val_data=[dataset[i] for i in val_idx]
    test_data=[dataset[i] for i in test_idx]

    all_raw=np.vstack([d.raw_x.numpy() for d in train_data])
    scaler=StandardScaler()
    scaler.fit(all_raw)

    for d in train_data+val_data+test_data:
        d.x=torch.tensor(scaler.transform(d.raw_x.numpy()),dtype=torch.float)

    pretrained_path=cfg.REAL_MODEL_PATH if os.path.exists(cfg.REAL_MODEL_PATH) else None
    if pretrained_path:
        log.info(f"Using real-world pretrained weights: {pretrained_path}")
    else:
        log.info("No real-world pretrained weights found. Training from synthetic dataset only.")

    model=train_model_fn(train_data,val_data,pretrained_model_path=pretrained_path)
    torch.save(model.state_dict(),cfg.MODEL_PATH)

    loader=DataLoader(test_data,batch_size=cfg.BATCH_SIZE,shuffle=False)
    _,preds,labels,_=_eval_loader(model,loader)
    metrics={"Accuracy %":round(accuracy_score(labels,preds)*100,2),
        "Precision % (Macro)":round(precision_score(labels,preds,average="macro",zero_division=0)*100,2),
        "Recall % (Macro)":round(recall_score(labels,preds,average="macro",zero_division=0)*100,2),
        "F1 % (Macro)":round(f1_score(labels,preds,average="macro",zero_division=0)*100,2)}
    report=classification_report(labels,preds,
        target_names=["Compliant","Review-Needed","Non-Compliant"],zero_division=0)
    cm=confusion_matrix(labels,preds)
    return model,scaler,metrics,report,cm


    # ─────────────────────────────────────────────────────────────────────────────
# FUSION 360 CLI BRIDGE
# ─────────────────────────────────────────────────────────────────────────────
def _json_safe_result(res):
    safe = dict(res)

    # Violation dataclasses -> plain dicts
    safe["violations"] = [v.to_dict() if hasattr(v, "to_dict") else v for v in safe.get("violations", [])]
    safe["passed_checks"] = [v.to_dict() if hasattr(v, "to_dict") else v for v in safe.get("passed_checks", [])]

    # Make sure everything is JSON serializable
    if "scores" in safe and isinstance(safe["scores"], dict):
        safe["scores"] = {
            str(k): (float(v) if isinstance(v, (np.floating, np.integer)) else v)
            for k, v in safe["scores"].items()
        }

    if "ml_probs" in safe and isinstance(safe["ml_probs"], dict):
        safe["ml_probs"] = {
            str(k): float(v) if isinstance(v, (np.floating, np.integer)) else v
            for k, v in safe["ml_probs"].items()
        }

    if "top_reasons" in safe:
        cleaned = []
        for item in safe["top_reasons"]:
            if isinstance(item, dict):
                cleaned.append({
                    str(k): (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                    for k, v in item.items()
                })
            else:
                cleaned.append(item)
        safe["top_reasons"] = cleaned

    for key in ["ml_confidence", "reconstruction_error"]:
        if key in safe and isinstance(safe[key], (np.floating, np.integer)):
            safe[key] = float(safe[key])

    for key in ["n_components", "n_violations", "n_passed", "ml_pred_class"]:
        if key in safe and isinstance(safe[key], (np.integer,)):
            safe[key] = int(safe[key])

    return safe


def validate_components_for_fusion(components):
    model, scaler, _, _, _ = load_pipeline()
    result = run_inference(components, trained_model=model, node_scaler=scaler)
    return _json_safe_result(result)


def fusion_cli_entry():
    """
    Usage:
        python app.py --fusion-cli input.json output.json
    """
    if len(sys.argv) < 4:
        raise SystemExit("Usage: python app.py --fusion-cli input.json output.json")

    input_path = sys.argv[2]
    output_path = sys.argv[3]

    with open(input_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    components = payload.get("components", [])
    if not isinstance(components, list):
        raise ValueError("Input JSON must contain a 'components' list.")

    result = validate_components_for_fusion(components)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

# ─────────────────────────────────────────────────────────────────────────────
# CSS
# ─────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@400;700;800&display=swap');
html,body,[class*="css"]{font-family:'Syne',sans-serif;}
.stApp{background:#0a0e1a;}
.gnr-header{background:linear-gradient(135deg,#0f172a 0%,#1e293b 50%,#0f172a 100%);border:1px solid #1e40af;border-radius:16px;padding:28px 36px;margin-bottom:24px;text-align:center;box-shadow:0 0 40px rgba(59,130,246,0.15);}
.gnr-header h1{font-family:'Syne',sans-serif;font-size:2.2em;font-weight:800;color:#f1f5f9;margin:0 0 6px 0;letter-spacing:2px;}
.gnr-header p{color:#64748b;margin:0;font-size:0.88em;letter-spacing:1px;}
.metric-card{background:#0f172a;border:1px solid #1e293b;border-radius:12px;padding:16px 18px;text-align:center;}
.metric-card .label{color:#64748b;font-size:0.72em;letter-spacing:1.2px;text-transform:uppercase;margin-bottom:5px;}
.metric-card .value{font-family:'JetBrains Mono',monospace;font-size:2em;font-weight:700;}
.verdict-compliant{background:#064e3b;border:2px solid #10b981;color:#a7f3d0;}
.verdict-review{background:#78350f;border:2px solid #f59e0b;color:#fde68a;}
.verdict-noncompliant{background:#7f1d1d;border:2px solid #ef4444;color:#fecaca;}
.verdict-box{border-radius:14px;padding:22px 28px;text-align:center;margin:16px 0;font-size:1.6em;font-weight:800;letter-spacing:2px;}
.section-title{color:#94a3b8;font-size:0.78em;letter-spacing:2px;text-transform:uppercase;border-bottom:1px solid #1e293b;padding-bottom:8px;margin:20px 0 14px 0;}
.viol-crit{border-left:4px solid #ef4444;background:#1c0a0a;padding:8px 12px;border-radius:0 6px 6px 0;margin:4px 0;}
.viol-warn{border-left:4px solid #f59e0b;background:#1c1100;padding:8px 12px;border-radius:0 6px 6px 0;margin:4px 0;}
.viol-info{border-left:4px solid #3b82f6;background:#0a0f1c;padding:8px 12px;border-radius:0 6px 6px 0;margin:4px 0;}
.viol-txt{font-family:'JetBrains Mono',monospace;font-size:0.82em;color:#cbd5e1;}
.stButton>button{background:#1e40af!important;color:white!important;border:none!important;border-radius:8px!important;font-family:'Syne',sans-serif!important;font-weight:700!important;letter-spacing:1px!important;}
.stButton>button:hover{background:#2563eb!important;box-shadow:0 0 20px rgba(59,130,246,0.4)!important;}
div[data-testid="stSidebar"]{background:#0d1117!important;border-right:1px solid #1e293b;}
</style>""", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    st.markdown("""<div class="gnr-header"><h1>🏭 GNR-VAL v5.0</h1>
        <p>INDUSTRIAL CAD COMPLIANCE VALIDATOR · VARROC EUREKA 3.0 · PS-9</p>
        <p style="color:#475569;font-size:0.8em;margin-top:6px;">
        TransformerConv GNN + ISO 2768 / ASME Y14.5 / GD&T / DFM Rule Engine</p>
        </div>""", unsafe_allow_html=True)

    # ── Sidebar ──────────────────────────────────────────────────────────────
    with st.sidebar:
        st.markdown("### ⚙️ Control Panel")
        org_name  = st.text_input("Organisation", value="Varroc Engineering")
        prod_name = st.text_input("Product", value="Eureka 3.0 Demo")
        st.markdown("---")
        st.markdown("### 🧠 Model")
        if st.button("🚀 Train / Load Model", use_container_width=True):
            with st.spinner("Building & training GNN pipeline (~2-3 min first run)…"):
                st.session_state["pipeline"] = load_pipeline()
            st.success("✅ Model ready!")
        if "pipeline" in st.session_state:
            _,_,metrics,_,_=st.session_state["pipeline"]
            st.markdown("**Test Metrics**")
            for k,v in metrics.items(): st.markdown(f"`{k}`: **{v}**")
        else:
            st.info("Click **Train / Load Model** to initialise.")
        st.markdown("---")
        st.markdown("### 📋 Input Mode")
        input_mode=st.radio("",["Manual Form","STEP File Upload","Random Demo"])

    tab_v, tab_m, tab_r = st.tabs(["🔍 Validate Design","📊 Model Analytics","📄 Export Report"])

    # ═════════════════════════════════════════════════════════════════════════
    # TAB 1 — VALIDATE
    # ═════════════════════════════════════════════════════════════════════════
    with tab_v:
        components_list=[]

        if input_mode=="Manual Form":
            st.markdown('<div class="section-title">Component Geometry Parameters</div>',unsafe_allow_html=True)
            if "form_components" not in st.session_state: st.session_state["form_components"]=[]
            with st.expander("➕ Add Component",expanded=True):
                c1,c2,c3=st.columns(3)
                with c1: nom_x=st.number_input("Nominal X",value=100.0,step=0.1); act_x=st.number_input("Actual X",value=100.1,step=0.1)
                with c2: nom_y=st.number_input("Nominal Y",value=100.0,step=0.1); act_y=st.number_input("Actual Y",value=100.1,step=0.1)
                with c3: nom_z=st.number_input("Nominal Z",value=50.0,step=0.1); act_z=st.number_input("Actual Z",value=50.1,step=0.1)
                c4,c5,c6,c7=st.columns(4)
                with c4: tol=st.number_input("Tolerance (mm)",value=0.05,format="%.4f")
                with c5: ra=st.number_input("Roughness Ra (µm)",value=0.8,step=0.1)
                with c6: mat=st.selectbox("Material",list(MATERIAL_CODES.keys()))
                with c7: ctyp=st.selectbox("Type",list(COMPONENT_TYPES.keys()))
                ba,bb=st.columns(2)
                with ba:
                    if st.button("➕ Add",use_container_width=True):
                        st.session_state["form_components"].append({
                            "actual_x":act_x,"actual_y":act_y,"actual_z":act_z,
                            "tolerance":tol,"material":mat,"type":ctyp,"roughness":ra,
                            "nominal_dims":[nom_x,nom_y,nom_z]})
                        st.success(f"Added {ctyp} ({len(st.session_state['form_components'])} total)")
                with bb:
                    if st.button("🗑️ Clear All",use_container_width=True):
                        st.session_state["form_components"]=[]
                        st.info("Cleared.")
            if st.session_state["form_components"]:
                st.dataframe(pd.DataFrame(st.session_state["form_components"]).drop(
                    columns=["nominal_dims"],errors="ignore"),use_container_width=True)
                components_list=st.session_state["form_components"]

        elif input_mode=="STEP File Upload":
            st.markdown('<div class="section-title">STEP File Import</div>',unsafe_allow_html=True)
            uploaded=st.file_uploader("Upload STEP/STP file",type=["step","stp"],
                help="ASCII STEP files. Real CARTESIAN_POINT & LENGTH_MEASURE tokens extracted.")
            if uploaded:
                with st.spinner("Parsing STEP file…"):
                    content=uploaded.read().decode("utf-8",errors="ignore")
                    components_list=parse_step_file(content)
                st.success(f"✅ Extracted {len(components_list)} components from `{uploaded.name}`")
                st.dataframe(pd.DataFrame(components_list).drop(
                    columns=["nominal_dims"],errors="ignore"),use_container_width=True)

        else:
            st.markdown('<div class="section-title">Random Demo — 20 Synthetic Components</div>',unsafe_allow_html=True)
            if st.button("🎲 Generate",use_container_width=False):
                comps=[]
                for _ in range(20):
                    comps.append({"actual_x":random.uniform(1,200),"actual_y":random.uniform(1,200),
                        "actual_z":random.uniform(1,200),
                        "nominal_dims":[random.uniform(1,200),random.uniform(1,200),random.uniform(1,200)],
                        "tolerance":random.uniform(0.0001,0.5),
                        "material":random.choice(list(MATERIAL_CODES.keys())),
                        "type":random.choice(list(COMPONENT_TYPES.keys())),
                        "roughness":random.uniform(0.1,10)})
                st.session_state["demo_components"]=comps
                st.success("20 random components generated.")
            if "demo_components" in st.session_state:
                components_list=st.session_state["demo_components"]
                st.dataframe(pd.DataFrame(components_list).drop(columns=["nominal_dims"],errors="ignore"),
                    use_container_width=True)

        st.markdown("---")
        run_col,_=st.columns([1,3])
        with run_col: run_btn=st.button("▶ RUN VALIDATION",use_container_width=True)

        if run_btn:
            if not components_list: st.error("No components loaded.")
            elif "pipeline" not in st.session_state: st.error("Train model first (sidebar).")
            else:
                model,scaler,_,_,_=st.session_state["pipeline"]
                with st.spinner("Running GNN + Rule Engine…"):
                    result=run_inference(components_list,trained_model=model,node_scaler=scaler)
                st.session_state["last_result"]=result
                st.session_state["last_components"]=components_list
                st.session_state["org_name"]=org_name; st.session_state["prod_name"]=prod_name

        if "last_result" in st.session_state:
            res=st.session_state["last_result"]; sc=res["scores"]; verdict=sc["verdict"]
            v_cls={"COMPLIANT":"verdict-compliant","REVIEW NEEDED":"verdict-review",
                   "NON-COMPLIANT":"verdict-noncompliant"}.get(verdict,"verdict-review")
            v_icon={"COMPLIANT":"✅","REVIEW NEEDED":"⚠️","NON-COMPLIANT":"🚫"}.get(verdict,"🔍")
            st.markdown(f'<div class="verdict-box {v_cls}">{v_icon} {verdict}<br>'
                f'<span style="font-size:0.55em;font-weight:400;">Fused Risk Score: {sc["fused_score"]} / 100</span>'
                f'</div>',unsafe_allow_html=True)

            basis=_compute_decision_basis(res); rec=_get_release_recommendation(verdict)
            st.markdown(f"""<div style="background:#0f172a;border-left:4px solid #3b82f6;
                border-radius:6px;padding:10px 16px;margin:8px 0;">
                <span style="color:#64748b;font-size:0.78em;">DECISION BASIS: </span>
                <span style="color:#93c5fd;font-size:0.88em;font-weight:600;">{basis}</span>
                &nbsp;&nbsp;|&nbsp;&nbsp;
                <span style="color:#64748b;font-size:0.78em;">RELEASE: </span>
                <span style="font-size:0.88em;font-weight:700;">{rec['icon']} {rec['label']}</span>
                </div>""",unsafe_allow_html=True)

            st.markdown('<div class="section-title">Compliance Scores</div>',unsafe_allow_html=True)
            c1,c2,c3,c4,c5=st.columns(5)
            fused_color="#10b981" if sc["fused_score"]<40 else ("#f59e0b" if sc["fused_score"]<70 else "#ef4444")
            for col,(val,label_,color_) in zip([c1,c2,c3,c4,c5],[
                (sc["ml_score"],"ML Non-Comp Prob","#3b82f6"),
                (sc["rule_score"],"Rule Severity","#8b5cf6"),
                (sc["fused_score"],"Fused Risk",fused_color),
                (f"{res['ml_confidence']}%","ML Confidence","#06b6d4"),
                (res["reconstruction_error"],"Recon Error","#a78bfa")]):
                col.markdown(f'<div class="metric-card"><div class="label">{label_}</div>'
                    f'<div class="value" style="color:{color_};">{val}</div></div>',unsafe_allow_html=True)

            st.markdown('<div class="section-title">ML Class Probabilities</div>',unsafe_allow_html=True)
            probs=res.get("ml_probs",{})
            p1,p2,p3=st.columns(3)
            for col,cls_,color_ in zip([p1,p2,p3],
                ["Compliant","Review-Needed","Non-Compliant"],["#10b981","#f59e0b","#ef4444"]):
                pct=probs.get(cls_,0.0)
                col.markdown(f'<div class="metric-card"><div class="label">{cls_}</div>'
                    f'<div class="value" style="color:{color_};">{pct}%</div></div>',unsafe_allow_html=True)
                col.progress(int(pct))

            st.markdown('<div class="section-title">Violation Summary</div>',unsafe_allow_html=True)
            vc1,vc2,vc3,vc4=st.columns(4)
            vc1.metric("🔴 Critical",sc["critical"]); vc2.metric("🟡 Warning",sc["warnings"])
            vc3.metric("🔵 Info",sc["info"]); vc4.metric("✅ Passed",res["n_passed"])

            # Charts
            st.markdown('<div class="section-title">Analytics Dashboard</div>',unsafe_allow_html=True)
            fig,axes=plt.subplots(1,3,figsize=(18,5),facecolor="#0a0e1a")
            g_col=fused_color; score=sc["fused_score"]
            axes[0].pie([score,100-score],colors=[g_col,"#1e293b"],startangle=90,
                counterclock=False,wedgeprops={"width":0.32})
            axes[0].text(0,0,f"{score}%",ha="center",va="center",color="white",fontsize=26,fontweight="bold")
            axes[0].set_title("Compliance Risk Score",color="white",fontweight="bold",pad=15)
            axes[0].set_facecolor("#0a0e1a")
            axes[1].set_facecolor("#0a0e1a")
            cats=["Critical","Warning","Info"]; cnts=[sc["critical"],sc["warnings"],sc["info"]]
            if sum(cnts)==0:
                axes[1].bar(["No Violations"],[1],color=["#10b981"])
                axes[1].text(0,1.1,"✅ All Clear",ha="center",color="#10b981",fontweight="bold")
            else:
                bars=axes[1].bar(cats,cnts,color=["#ef4444","#f59e0b","#3b82f6"])
                for bar in bars: axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.05,
                    f"{int(bar.get_height())}",ha="center",color="white",fontweight="bold")
            axes[1].tick_params(colors="white"); axes[1].set_title("Severity Distribution",color="white",fontweight="bold")
            axes[1].grid(axis="y",color="#1e293b",linestyle="--")
            axes[2].set_facecolor("#0a0e1a")
            fl=["ML Score","Rule Score","Fused"]; fv=[sc["ml_score"],sc["rule_score"],sc["fused_score"]]
            bars3=axes[2].bar(fl,fv,color=["#3b82f6","#8b5cf6",g_col])
            for bar in bars3: axes[2].text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,
                f"{bar.get_height():.1f}",ha="center",color="white",fontweight="bold")
            axes[2].set_ylim(0,115); axes[2].tick_params(colors="white")
            axes[2].set_title("Model Fusion Breakdown",color="white",fontweight="bold")
            axes[2].grid(axis="y",color="#1e293b",linestyle="--")
            plt.tight_layout(pad=3.0); st.pyplot(fig); plt.close()

            # Assembly graph
            comps=st.session_state.get("last_components",[])
            if comps:
                st.markdown('<div class="section-title">Assembly Topology Graph</div>',unsafe_allow_html=True)
                G_vis,labels_vis=_build_assembly_graph(comps)
                n_vis=len(comps); violated_nodes={v.node for v in res["violations"] if isinstance(v.node,int)}
                node_colors=["#ef4444" if i in violated_nodes else "#10b981" for i in range(n_vis)]
                fig2,ax_g=plt.subplots(figsize=(14,6),facecolor="#0a0e1a"); ax_g.set_facecolor("#0d1117")
                try:
                    pos=nx.spring_layout(G_vis,k=2.5/max(1,np.sqrt(n_vis)),seed=42)
                    nx.draw_networkx_edges(G_vis,pos,ax=ax_g,edge_color="#374151",width=1.5,arrowsize=15)
                    nx.draw_networkx_nodes(G_vis,pos,ax=ax_g,node_color=node_colors,node_size=900,
                        edgecolors="white",linewidths=1.2)
                    nx.draw_networkx_labels(G_vis,pos,labels=labels_vis,ax=ax_g,
                        font_color="white",font_weight="bold",font_size=8)
                    ax_g.legend(handles=[mpatches.Patch(color="#ef4444",label="Violated"),
                        mpatches.Patch(color="#10b981",label="Compliant")],
                        facecolor="#1e293b",edgecolor="#374151",labelcolor="white",fontsize=9)
                except Exception: ax_g.text(0.5,0.5,"Graph unavailable",transform=ax_g.transAxes,ha="center",color="white")
                ax_g.axis("off"); ax_g.set_title("Assembly Relationship View",color="white",fontweight="bold",pad=14)
                st.pyplot(fig2); plt.close()

                st.markdown('<div class="section-title">3D Digital Twin Preview</div>', unsafe_allow_html=True)
                fig_3d = render_cad_3d_model(comps, res.get("violations", []))
                if fig_3d is not None:
                    st.plotly_chart(fig_3d, use_container_width=True, config={"displaylogo": False})


            # Uncertainty
            recon_err=float(res.get("reconstruction_error",0.0))
            unc_level=("Low 🟢" if recon_err<1.0 else ("Moderate 🟡" if recon_err<15.0 else "High 🔴"))
            st.markdown('<div class="section-title">Model Uncertainty</div>',unsafe_allow_html=True)
            uc1,uc2=st.columns([3,1])
            with uc1: st.progress(min(100,int(recon_err*5)),text=f"Reconstruction Error: {recon_err:.4f}")
            with uc2: st.markdown(f"**{unc_level}**")

            # Top reasons
            top3=res.get("top_reasons",[])
            if top3:
                st.markdown('<div class="section-title">Top Violation Reasons</div>',unsafe_allow_html=True)
                for r in top3:
                    sev=r.get("severity","INFO")
                    tag_cls={"CRITICAL":"#ef4444","WARNING":"#f59e0b"}.get(sev,"#3b82f6")
                    st.markdown(f"""<div style="background:#0f172a;border:1px solid #1e293b;
                        border-radius:8px;padding:10px 14px;margin:5px 0;">
                        <span style="background:{tag_cls}20;color:{tag_cls};padding:2px 8px;
                        border-radius:4px;font-size:0.75em;font-weight:700;">{sev}</span>
                        <span style="color:#94a3b8;font-size:0.85em;margin-left:8px;">
                        Node {r.get('node','?')} | {r.get('rule_id','')}</span>
                        <span style="color:#e2e8f0;font-size:0.88em;margin-left:8px;">
                        → {r.get('message','')}</span></div>""",unsafe_allow_html=True)

            # Full violation log
            violations=res.get("violations",[])
            if violations:
                st.markdown('<div class="section-title">Full Violation Log</div>',unsafe_allow_html=True)
                with st.expander(f"Show all {len(violations)} violations"):
                    for v in violations:
                        sev_cls={"CRITICAL":"viol-crit","WARNING":"viol-warn"}.get(v.severity,"viol-info")
                        st.markdown(f'<div class="{sev_cls}"><span class="viol-txt">'
                            f'[{v.severity}] Node {v.node} | {v.rule_id} | '
                            f'Value: {v.value} | Limit: {v.limit} | {v.suggestion}'
                            f'</span></div>',unsafe_allow_html=True)

            # Passed checks
            passed=res.get("passed_checks",[])
            with st.expander(f"✅ {len(passed)} passed checks"):
                pdf=pd.DataFrame([{"Node":p.node,"Rule":p.rule_id,"Value":p.value,"Limit":p.limit}
                    for p in passed[:50]])
                if not pdf.empty: st.dataframe(pdf,use_container_width=True)

            # Remediation advisor
            st.markdown('<div class="section-title">🔧 Remediation Advisor</div>',unsafe_allow_html=True)
            for i,action in enumerate(_get_top3_remediations(res) or [],1):
                st.markdown(f"""<div style="background:#0f172a;border-left:4px solid #3b82f6;
                    border-radius:0 8px 8px 0;padding:10px 16px;margin:6px 0;">
                    <span style="color:#3b82f6;font-weight:700;margin-right:8px;">{i}.</span>
                    <span style="color:#cbd5e1;font-size:0.9em;">{action}</span></div>""",
                    unsafe_allow_html=True)
            st.markdown('<div class="section-title">AI Explainability</div>', unsafe_allow_html=True)

            explain_text = sc.get("explanation", "N/A")
            override_text = sc.get("override_reason", "")

            explain_html = (
                '<div style="background:#0f172a;'
                'border:1px solid #1e293b;'
                'border-radius:10px;'
                'padding:14px 18px;'
                'color:#94a3b8;'
                'font-size:0.88em;'
                'line-height:1.8;">'
                f'<div>{explain_text}</div>'
            )

            if override_text:
                explain_html += (
                    '<div style="margin-top:8px;">'
                    '<span style="color:#ef4444;font-weight:700;">Override:</span>'
                    f'<span style="color:#cbd5e1;"> {override_text}</span>'
                    '</div>'
                )

            explain_html += '</div>'

            st.markdown(explain_html, unsafe_allow_html=True)
            # Input warnings
            for w in res.get("input_warnings",[]): st.warning(w)

    # ═════════════════════════════════════════════════════════════════════════
    # TAB 2 — MODEL ANALYTICS
    # ═════════════════════════════════════════════════════════════════════════
    with tab_m:
        if "pipeline" not in st.session_state:
            st.info("Train model first.")
        else:
            model,scaler,metrics,report,cm=st.session_state["pipeline"]
            st.markdown('<div class="section-title">GNN Test Set Performance</div>',unsafe_allow_html=True)
            m1,m2,m3,m4=st.columns(4)
            for col,(k,v),color in zip([m1,m2,m3,m4],metrics.items(),
                ["#10b981","#3b82f6","#f59e0b","#8b5cf6"]):
                col.markdown(f'<div class="metric-card"><div class="label">{k}</div>'
                    f'<div class="value" style="color:{color};">{v}%</div></div>',unsafe_allow_html=True)
            st.markdown('<div class="section-title">Classification Report</div>',unsafe_allow_html=True)
            st.code(report,language="")
            st.markdown('<div class="section-title">Confusion Matrix</div>',unsafe_allow_html=True)
            fig_cm,ax_cm=plt.subplots(figsize=(7,5),facecolor="#0a0e1a"); ax_cm.set_facecolor("#0a0e1a")
            im=ax_cm.imshow(cm,cmap="Blues")
            for axis,lbl in [(ax_cm.xaxis,"X"),(ax_cm.yaxis,"Y")]:
                pass
            ax_cm.set_xticks([0,1,2]); ax_cm.set_yticks([0,1,2])
            ax_cm.set_xticklabels(["Compliant","Review","Non-Compliant"],color="white",fontsize=9)
            ax_cm.set_yticklabels(["Compliant","Review","Non-Compliant"],color="white",fontsize=9)
            ax_cm.set_xlabel("Predicted",color="white"); ax_cm.set_ylabel("Actual",color="white")
            ax_cm.set_title("Confusion Matrix",color="white",fontweight="bold")
            for i in range(3):
                for j in range(3):
                    ax_cm.text(j,i,str(cm[i][j]),ha="center",va="center",
                        color="white",fontweight="bold",fontsize=12)
            plt.colorbar(im,ax=ax_cm); st.pyplot(fig_cm); plt.close()
            st.markdown('<div class="section-title">Model Architecture</div>',unsafe_allow_html=True)
            total=sum(p.numel() for p in model.parameters())
            st.markdown(f"""| Component | Details |
|---|---|
| **Encoder** | 3× TransformerConv (edge-aware, edge_dim=2) |
| **Dims** | {cfg.HIDDEN_DIM}→{cfg.HIDDEN_DIM}→{cfg.LATENT_DIM} |
| **Decoder** | Autoencoder (latent→{cfg.IN_CHANNELS}) |
| **Classifier** | {cfg.LATENT_DIM}→32→16→3 |
| **Total Params** | `{total:,}` |
| **Device** | `{DEVICE}` |
| **Standards** | ISO 2768, ASME Y14.5, GD&T, DFM |""")

    # ═════════════════════════════════════════════════════════════════════════
    # TAB 3 — EXPORT
    # ═════════════════════════════════════════════════════════════════════════
    with tab_r:
        if "last_result" not in st.session_state:
            st.info("Run a validation first.")
        else:
            import datetime
            res=st.session_state["last_result"]; sc=res["scores"]
            org=st.session_state.get("org_name","N/A"); prod=st.session_state.get("prod_name","N/A")
            ts=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            basis=_compute_decision_basis(res); rec=_get_release_recommendation(sc["verdict"])
            top3_rem=_get_top3_remediations(res)
            lines=["="*70,"  GNR-VAL v5.0 — INDUSTRIAL CAD COMPLIANCE REPORT",
                "  Varroc Eureka 3.0 · Problem Statement 9","="*70,
                f"  Generated   : {ts}",f"  Organisation: {org}",f"  Product     : {prod}","",
                "FINAL VERDICT","-"*70,
                f"  Verdict        : {rec['icon']} {sc['verdict']}",
                f"  Release Status : {rec['label']}",f"  Decision Basis : {basis}","",
                "COMPLIANCE SCORES","-"*70,
                f"  ML Non-Compliance Prob : {sc['ml_score']}",
                f"  Rule Severity Score    : {sc['rule_score']}",
                f"  Fused Risk Score       : {sc['fused_score']}",
                f"  ML Prediction          : {res['ml_prediction'].upper()}",
                f"  ML Confidence          : {res['ml_confidence']}%",
                f"  Reconstruction Error   : {res['reconstruction_error']}","",
                "VIOLATION SUMMARY","-"*70,
                f"  Critical: {sc['critical']} | Warning: {sc['warnings']} | Info: {sc['info']} | Passed: {res['n_passed']}","",
                "AI EXPLAINABILITY","-"*70,f"  {sc.get('explanation','N/A')}"]
            if sc.get("override_reason"): lines.append(f"  Override: {sc['override_reason']}")
            lines+=["","TOP REMEDIATION ACTIONS","-"*70]
            for i,a in enumerate(top3_rem,1): lines.append(f"  {i}. {a}")
            lines+=["","VIOLATIONS LOG","-"*70]
            for v in res.get("violations",[]):
                lines.append(f"  [{v.severity}] Node {v.node} | {v.rule_id} | Value:{v.value} | Limit:{v.limit} | {v.suggestion}")
            lines+=["","="*70,"  END OF REPORT — GNR-VAL v5.0","="*70]
            report_text="\n".join(lines)
            fname=f"GNRVal_Report_{ts.replace(' ','_').replace(':','-')}.txt"
            st.markdown('<div class="section-title">Report Preview</div>',unsafe_allow_html=True)
            st.code(report_text,language="")
            json_data={"timestamp":ts,"org":org,"product":prod,"verdict":sc["verdict"],
                "scores":sc,"ml_prediction":res["ml_prediction"],"ml_confidence":res["ml_confidence"],
                "violations":[v.to_dict() for v in res["violations"]],
                "top_reasons":res["top_reasons"],"remediations":top3_rem}
            col_t,col_j=st.columns(2)
            with col_t: st.download_button("⬇️ Download TXT",data=report_text,
                file_name=fname,mime="text/plain",use_container_width=True)
            with col_j: st.download_button("⬇️ Download JSON",
                data=json.dumps(json_data,indent=2),
                file_name=fname.replace(".txt",".json"),mime="application/json",use_container_width=True)

    st.markdown("""<div style="text-align:center;color:#1e293b;font-size:0.75em;margin-top:40px;
        border-top:1px solid #1e293b;padding-top:16px;">
        GNR-Val v5.0 · TransformerConv GNN + ISO 2768 / ASME Y14.5 Rule Engine · Varroc Eureka 3.0
        </div>""",unsafe_allow_html=True)

if __name__ == "__main__":
    if len(sys.argv) >= 2 and sys.argv[1] == "--fusion-cli":
        fusion_cli_entry()
    else:
        main()


Writing app.py


In [ ]:
# ==============================
# FUSION360 DATASET CHECK
# ==============================
import os

step_dir = "/content/drive/MyDrive/TRAIN_DATASET_CAD/step"
json_dir = "/content/drive/MyDrive/TRAIN_DATASET_CAD/json"

print("STEP:", len(os.listdir(step_dir)))
print("JSON:", len(os.listdir(json_dir)))

STEP: 1500
JSON: 1500


In [ ]:
%%writefile fusion.py

import adsk.core
import adsk.fusion
import traceback
import subprocess
import tempfile
import json
import os

# ============================================================
# EDIT THESE 2 PATHS ONLY
# ============================================================
PYTHON_EXE = r"C:\Users\adity\AppData\Local\Programs\Python\Python310\python.exe"
APP_PY_PATH = r"C:\Users\adity\OneDrive\Desktop\CAD_Fusion360_PY api\app.py"

# Default metadata for MVP
DEFAULT_MATERIAL = "steel"
DEFAULT_TYPE = "housing"
DEFAULT_TOLERANCE = 0.10
DEFAULT_ROUGHNESS = 1.6


def _get_app():
    return adsk.core.Application.get()


def _get_ui():
    app = _get_app()
    return app.userInterface if app else None


def _mm_from_internal(design, value):
    try:
        um = design.unitsManager
        return float(um.convert(value, um.internalUnits, "mm"))
    except:
        # fallback
        return float(value)


def _pick_body_from_active_design(design):
    root = design.rootComponent

    # 1) If a BRepBody is selected, use that first
    app = _get_app()
    ui = _get_ui()

    try:
        sels = ui.activeSelections
        for i in range(sels.count):
            ent = sels.item(i).entity
            body = adsk.fusion.BRepBody.cast(ent)
            if body:
                return body
    except:
        pass

    # 2) Otherwise first root body
    if root.bRepBodies.count > 0:
        return root.bRepBodies.item(0)

    # 3) Otherwise first body from first occurrence/component that has one
    for occ in root.allOccurrences:
        comp = occ.component
        if comp and comp.bRepBodies.count > 0:
            return comp.bRepBodies.item(0)

    return None


def _body_to_component_dict(design, body):
    bbox = body.boundingBox
    min_pt = bbox.minPoint
    max_pt = bbox.maxPoint

    x_mm = _mm_from_internal(design, abs(max_pt.x - min_pt.x))
    y_mm = _mm_from_internal(design, abs(max_pt.y - min_pt.y))
    z_mm = _mm_from_internal(design, abs(max_pt.z - min_pt.z))

    # keep actual == nominal for MVP
    component = {
        "actual_x": round(max(1.0, x_mm), 3),
        "actual_y": round(max(1.0, y_mm), 3),
        "actual_z": round(max(1.0, z_mm), 3),
        "tolerance": DEFAULT_TOLERANCE,
        "material": DEFAULT_MATERIAL,
        "type": DEFAULT_TYPE,
        "roughness": DEFAULT_ROUGHNESS,
        "nominal_dims": [
            round(max(1.0, x_mm), 3),
            round(max(1.0, y_mm), 3),
            round(max(1.0, z_mm), 3),
        ],
    }
    return component


def _format_result(result):
    if not isinstance(result, dict):
        return "Invalid result returned from backend."

    if "error" in result:
        return f"Error: {result['error']}"

    scores = result.get("scores", {})
    verdict = scores.get("verdict", "Unknown")
    fused = scores.get("fused_score", "N/A")
    ml_conf = result.get("ml_confidence", "N/A")
    n_viol = result.get("n_violations", 0)

    top_reasons = result.get("top_reasons", [])
    reason_lines = []
    for r in top_reasons[:3]:
        if isinstance(r, dict):
            sev = r.get("severity", "INFO")
            msg = r.get("message", "No message")
            rule_id = r.get("rule_id", "RULE")
            reason_lines.append(f"- [{sev}] {rule_id}: {msg}")
        else:
            reason_lines.append(f"- {str(r)}")

    if not reason_lines:
        reason_lines.append("- No major reasons returned.")

    text = (
        f"GNR-Val Fusion Validation\n\n"
        f"Verdict: {verdict}\n"
        f"Fused Risk Score: {fused}\n"
        f"ML Confidence: {ml_conf}%\n"
        f"Violations: {n_viol}\n\n"
        f"Top Reasons:\n" + "\n".join(reason_lines)
    )
    return text


def _run_backend_validation(component):
    payload = {"components": [component]}

    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = os.path.join(tmpdir, "fusion_input.json")
        output_path = os.path.join(tmpdir, "fusion_output.json")

        with open(input_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        cmd = [
            PYTHON_EXE,
            APP_PY_PATH,
            "--fusion-cli",
            input_path,
            output_path,
        ]

        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=1800  # allow first-time training/load
        )

        if proc.returncode != 0:
            raise RuntimeError(
                "Backend validation failed.\n\n"
                f"STDOUT:\n{proc.stdout}\n\n"
                f"STDERR:\n{proc.stderr}"
            )

        if not os.path.exists(output_path):
            raise RuntimeError("Backend finished but output JSON was not created.")

        with open(output_path, "r", encoding="utf-8") as f:
            result = json.load(f)

    return result


def run(context):
    ui = None
    try:
        app = _get_app()
        ui = _get_ui()

        if not os.path.exists(PYTHON_EXE):
            ui.messageBox(f"Python executable not found:\n{PYTHON_EXE}")
            return

        if not os.path.exists(APP_PY_PATH):
            ui.messageBox(f"app.py not found:\n{APP_PY_PATH}")
            return

        design = adsk.fusion.Design.cast(app.activeProduct)
        if not design:
            ui.messageBox("No active Fusion 360 design found.")
            return

        body = _pick_body_from_active_design(design)
        if not body:
            ui.messageBox("No solid body found. Select a body or open a model with at least one body.")
            return

        component = _body_to_component_dict(design, body)

        ui.messageBox(
            "Running GNR-Val backend...\n"
            "First run may take longer because model load/train can happen there."
        )

        result = _run_backend_validation(component)
        ui.messageBox(_format_result(result))

    except Exception as e:
        if ui:
            ui.messageBox("Failed:\n{}".format(traceback.format_exc()))


def stop(context):
    pass

Writing fusion.py


In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import warnings
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import time # Added for performance timing

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool

warnings.filterwarnings('ignore')

# Structured logging replaces bare print() throughout the pipeline.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gnrval')
log.info('All imports successful.')

In [ ]:
@dataclass
class Config:
    """Central configuration — edit here, not scattered across cells."""
    SEED: int = 42
    # Target 40/30/30 distribution
    N_COMPLIANT: int = 400
    N_REVIEW_NEEDED: int = 300
    N_NONCOMPLIANT: int = 300
    MIN_COMPONENTS: int = 4
    MAX_COMPONENTS: int = 14
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float = 0.15
    IN_CHANNELS: int = 8
    HIDDEN_DIM: int = 128
    LATENT_DIM: int = 64
    DROPOUT: float = 0.25
    EPOCHS: int = 120
    BATCH_SIZE: int = 32
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    RECON_WEIGHT: float = 0.10
    GRAD_CLIP: float = 1.0
    EARLY_STOP_PATIENCE: int = 20
    LOG_EVERY: int = 10
    MODEL_PATH: str = 'gnrval_v5.pt'
    NONCOMPLIANT_THRESH: float = 70.0
    REVIEW_THRESH: float = 40.0

cfg = Config()

def set_seed(seed: int = cfg.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info('Config ready with 3-class distribution targets.')

In [ ]:
# =============================================================================
# CELL 3 — DOMAIN CONSTANTS
#
# Engineering lookup tables (ISO 2768, ASME Y14.5, GD&T, DFM) extracted into
# named dicts so rule logic is readable, not magic-numbered.
# =============================================================================

MATERIAL_CODES: Dict[str, int] = {
    'steel': 0, 'aluminum': 1, 'plastic': 2, 'titanium': 3, 'composite': 4,
}
COMPONENT_TYPES: Dict[str, int] = {
    'shaft': 0, 'bearing': 1, 'housing': 2, 'bracket': 3, 'fastener': 4, 'gear': 5,
}
INV_MAT  = {v: k for k, v in MATERIAL_CODES.items()}
INV_COMP = {v: k for k, v in COMPONENT_TYPES.items()}

RULE_CATEGORIES: Dict[str, str] = {
    'TOL':  'Tolerance',
    'COMP': 'Component-Specific',
    'SURF': 'Surface Finish',
    'DIM':  'Dimensional',
    'DFM':  'Design for Manufacture',
    'MAT':  'Material Compatibility',
    'JNTI': 'Joint/Clearance',
}

# Dimensional / surface standards (mm / µm)
STANDARDS: Dict[str, float] = {
    'ISO2768_FINE':       0.10,
    'ISO2768_MEDIUM':     0.30,
    'ISO2768_COARSE':     0.80,
    'ISO2768_VERY_COARSE':1.50,
    'RA_MATING':          1.60,
    'RA_GENERAL':         3.20,
    'RA_ROUGH':           6.30,
    'DIM_MIN':            5.0,
    'DIM_MAX':          500.0,
    'CL_CLOSE':           0.05,
    'CL_MEDIUM':          0.20,
    'CL_LOOSE':           0.50,
    'CL_CRITICAL':        1.00,
    'WEIGHT_LIMIT':    5000.0,
    'ASPECT_RATIO_MAX':  10.0,
}

# Per-component tolerance / surface / dimension limits
COMPONENT_RULES: Dict[str, Dict] = {
    'shaft':    {'max_tol': 0.050, 'max_ra': 1.6, 'min_dim':  5.0, 'max_dim': 300.0, 'max_ar': 10, 'std': 'IT6-IT8'},
    'bearing':  {'max_tol': 0.025, 'max_ra': 0.8, 'min_dim': 10.0, 'max_dim': 200.0, 'max_ar':  2, 'std': 'IT5-IT6'},
    'gear':     {'max_tol': 0.050, 'max_ra': 1.6, 'min_dim': 20.0, 'max_dim': 400.0, 'max_ar':  5, 'std': 'ISO 1328'},
    'housing':  {'max_tol': 0.300, 'max_ra': 3.2, 'min_dim': 20.0, 'max_dim': 500.0, 'max_ar':  5, 'std': 'IT8-IT11'},
    'bracket':  {'max_tol': 0.500, 'max_ra': 6.3, 'min_dim': 10.0, 'max_dim': 500.0, 'max_ar':  8, 'std': 'IT11-IT14'},
    'fastener': {'max_tol': 0.150, 'max_ra': 3.2, 'min_dim':  3.0, 'max_dim': 100.0, 'max_ar':  8, 'std': 'ISO 4759'},
}

# Galvanic / structural material compatibility
MATERIAL_COMPAT: Dict[Tuple, bool] = {
    ('steel',     'steel'):     True,
    ('steel',     'aluminum'):  True,
    ('steel',     'composite'): True,
    ('aluminum',  'aluminum'):  True,
    ('aluminum',  'composite'): True,
    ('titanium',  'steel'):     True,
    ('titanium',  'composite'): True,
    ('plastic',   'plastic'):   True,
    ('plastic',   'aluminum'):  True,
    ('steel',     'plastic'):   False,
    ('titanium',  'aluminum'):  False,
    ('composite', 'plastic'):   False,
}

JOINT_CL_LIMITS: Dict[str, float] = {
    'fixed':     0.10,
    'revolute':  0.30,
    'prismatic': 0.20,
    'contact':   0.40,
}


def material_compatible(m1_code: int, m2_code: int) -> bool:
    """Return True if the two material codes form a compatible pairing."""
    m1 = INV_MAT.get(int(m1_code), 'steel')
    m2 = INV_MAT.get(int(m2_code), 'steel')
    return MATERIAL_COMPAT.get((m1, m2), MATERIAL_COMPAT.get((m2, m1), True))


log.info('Domain constants loaded.')

In [ ]:
# =============================================================================
# CELL 4 — VIOLATION DATACLASS
#
# A typed, serialisable container for every rule result — both passes and
# failures.  Using a dataclass instead of bare dicts makes downstream code
# easier to read, type-check, and serialise to JSON.
# =============================================================================

@dataclass
class Violation:
    """Represents the outcome of a single rule check on one node or edge."""

    node:       Any    # int (node index) or str (edge label)
    rule_id:    str    # short machine-readable rule identifier
    category:   str    # one of RULE_CATEGORIES keys
    rule:       str    # human-readable rule description
    value:      float  # measured value
    limit:      str    # expected limit expressed as a string
    severity:   str    # CRITICAL | WARNING | INFO
    suggestion: str = ''   # actionable remediation hint
    passed:     bool = False  # True when value satisfies the rule

    def to_dict(self) -> Dict:
        """Serialise to a plain dict (JSON-compatible)."""
        return asdict(self)


log.info('Violation dataclass ready.')

In [ ]:
def _check_tol(nominal: float, tolerance: float, actual: float) -> Tuple[bool, float, float]:
    """Return (within_tol, lower_bound, upper_bound) for a nominal±tol check."""
    lo = nominal - abs(tolerance)
    hi = nominal + abs(tolerance)
    return (lo <= actual <= hi), lo, hi


def run_rule_engine(
    node_features: np.ndarray,
    nominal_dims: Optional[List[Tuple[float, float, float]]] = None,
    nx_graph: Optional[nx.Graph] = None,
) -> Tuple[List[Violation], List[Violation]]:

    violations: List[Violation] = []
    passed_checks: List[Violation] = []

    def _record(v: Violation):
        (passed_checks if v.passed else violations).append(v)

    # ───────── Node-level checks ─────────
    for i, feat in enumerate(node_features):

        dim_x, dim_y, dim_z, tol, mat_code, type_code, weight, roughness = (
            float(f) for f in feat
        )

        nom = nominal_dims[i] if nominal_dims and i < len(nominal_dims) else (dim_x, dim_y, dim_z)
        nom_x, nom_y, nom_z = nom

        comp = INV_COMP.get(int(type_code), 'housing')
        cr = COMPONENT_RULES.get(comp, COMPONENT_RULES['housing'])

        # Rule 0 — Basic physical validity
        for d, ax_name in ((dim_x, 'X'), (dim_y, 'Y'), (dim_z, 'Z')):

            ok = d > 0

            _record(Violation(
                node=i,
                rule_id=f'DIM-POSITIVE-{ax_name}',
                category='DIM',
                rule=f'Dim {ax_name} must be > 0',
                value=round(d,3),
                limit='>0mm',
                severity='CRITICAL' if not ok else 'INFO',
                suggestion='Invalid dimension' if not ok else 'Dimension OK',
                passed=ok
            ))

        # Rule 1 — Nominal ± tolerance
        for actual, nominal, ax in [
            (dim_x, nom_x, 'X'),
            (dim_y, nom_y, 'Y'),
            (dim_z, nom_z, 'Z')
        ]:

            ok, lo, hi = _check_tol(nominal, tol, actual)

            if ok:
                sev = 'INFO'
            else:
                deviation = abs(actual - nominal)

                if deviation <= tol * 1.20:
                    sev = 'WARNING'
                else:
                    sev = 'CRITICAL'

            _record(Violation(
                node=i,
                rule_id=f'TOL-DIM-{ax}',
                category='TOL',
                rule=f'ISO 2768 Dim {ax}',
                value=round(actual,4),
                limit=f'[{lo:.4f},{hi:.4f}]',
                severity=sev,
                suggestion='Tolerance exceeded' if not ok else 'Within tolerance',
                passed=ok
            ))

        # Rule 2 — Component specific tolerance
        if cr:

            ok = tol <= cr['max_tol']

            if ok:
                sev = 'INFO'
            elif tol <= cr['max_tol'] * 1.20:
                sev = 'WARNING'
            else:
                sev = 'CRITICAL'

            _record(Violation(
                node=i,
                rule_id='TOL-COMP',
                category='COMP',
                rule=f'{comp} tolerance',
                value=round(tol,5),
                limit=f'<={cr["max_tol"]}',
                severity=sev,
                suggestion='Tolerance too large' if not ok else 'OK',
                passed=ok
            ))

        # Rule 3 — Surface roughness
        ok = roughness <= cr['max_ra']

        _record(Violation(
            node=i,
            rule_id='SURF-RA',
            category='SURF',
            rule='Surface roughness',
            value=round(roughness,3),
            limit=f'<={cr["max_ra"]}',
            severity='WARNING' if not ok else 'INFO',
            suggestion='Surface too rough' if not ok else 'Surface OK',
            passed=ok
        ))

        # Rule 4 — Dimensional bounds
        for d, ax in [(dim_x,'X'), (dim_y,'Y'), (dim_z,'Z')]:

            ok = cr['min_dim'] <= d <= cr['max_dim']

            if ok:
                sev='INFO'
            elif d < cr['min_dim']*0.5 or d > cr['max_dim']*1.5:
                sev='CRITICAL'
            else:
                sev='WARNING'

            _record(Violation(
                node=i,
                rule_id=f'DIM-BOUND-{ax}',
                category='DIM',
                rule='Dimension bounds',
                value=round(d,3),
                limit=f'{cr["min_dim"]}-{cr["max_dim"]}',
                severity=sev,
                suggestion='Out of bounds' if not ok else 'OK',
                passed=ok
            ))

        # Rule 5 — Aspect ratio
        ds = sorted([dim_x, dim_y, dim_z])
        ratio = ds[-1] / max(ds[0],1e-6)

        ok = ratio <= cr['max_ar']

        _record(Violation(
            node=i,
            rule_id='DFM-AR',
            category='DFM',
            rule='Aspect ratio',
            value=round(ratio,2),
            limit=f'<={cr["max_ar"]}',
            severity='WARNING' if not ok else 'INFO',
            suggestion='Aspect ratio high' if not ok else 'OK',
            passed=ok
        ))

        # Rule 6 — Weight
        ok = weight <= STANDARDS['WEIGHT_LIMIT']

        sev = 'INFO' if ok else (
            'WARNING' if weight <= STANDARDS['WEIGHT_LIMIT']*1.25 else 'CRITICAL'
        )

        _record(Violation(
            node=i,
            rule_id='DFM-WT',
            category='DFM',
            rule='Weight limit',
            value=round(weight,1),
            limit=f'<={STANDARDS["WEIGHT_LIMIT"]}',
            severity=sev,
            suggestion='Weight too high' if not ok else 'OK',
            passed=ok
        ))

    # ───────── Edge checks ─────────
    if nx_graph is not None:

        n = len(node_features)

        for u, v_node, data in nx_graph.edges(data=True):

            if not (0 <= u < n and 0 <= v_node < n):
                continue

            m1, m2 = int(node_features[u][4]), int(node_features[v_node][4])

            ok = material_compatible(m1,m2)

            mn1 = INV_MAT.get(m1,'steel')
            mn2 = INV_MAT.get(m2,'steel')

            _record(Violation(
                node=f'e({u}->{v_node})',
                rule_id='MAT-COMPAT',
                category='MAT',
                rule='Material compatibility',
                value=0,
                limit='compatible',
                severity='CRITICAL' if not ok else 'INFO',
                suggestion='Galvanic risk' if not ok else 'OK',
                passed=ok
            ))

            cl = float(data.get('clearance',0.0))
            jt = str(data.get('joint_type','contact'))

            lim = JOINT_CL_LIMITS.get(jt,0.40)

            ok = cl <= lim

            sev = 'CRITICAL' if cl > STANDARDS['CL_CRITICAL'] else ('WARNING' if not ok else 'INFO')

            _record(Violation(
                node=f'e({u}->{v_node})',
                rule_id='JNTI-CL',
                category='JNTI',
                rule='Joint clearance',
                value=round(cl,4),
                limit=f'<={lim}',
                severity=sev,
                suggestion='Clearance too large' if not ok else 'OK',
                passed=ok
            ))

    return violations, passed_checks

In [ ]:
# =============================================================================
# CELL 6 — SCORE FUSION + EXPLAINABILITY
#
# fuse_scores() combines the rule-based score and the GNN probability into a
# single 0-100 fused score, applies CRITICAL-violation overrides, and produces
# a human-readable explanation that travels with every prediction.
# =============================================================================

def fuse_scores(
    ml_prob_noncompliant: float,
    violations: List[Violation],
) -> Dict[str, Any]:
    """
    Combine rule-based severity score and GNN probability into a fused verdict.

    Fusion weights
    --------------
    * No CRITICAL violations : 50 % ML  +  50 % rules  (balanced)
    * ≥1 CRITICAL violation  : 30 % ML  +  70 % rules  (rules dominate)
      and fused score is clamped to >= NONCOMPLIANT_THRESH so it always
      renders as NON-COMPLIANT even when the GNN is uncertain.

    Returns a dict that includes:
        ml_score, rule_score, fused_score, verdict, counts,
        override_reason, explanation  ← human-readable string
    """
    n_crit = sum(1 for v in violations if v.severity == 'CRITICAL')
    n_warn = sum(1 for v in violations if v.severity == 'WARNING')
    n_info = sum(1 for v in violations if v.severity == 'INFO')

    rule_score = min(100.0, n_crit * 40.0 + n_warn * 15.0 + n_info * 5.0)
    ml_score   = float(ml_prob_noncompliant) * 100.0

    if n_crit > 0:
        fused          = max(cfg.NONCOMPLIANT_THRESH, 0.30 * ml_score + 0.70 * rule_score)
        verdict        = 'NON-COMPLIANT'
        override_reason = f'{n_crit} CRITICAL violation(s) → automatic NON-COMPLIANT override'
    else:
        fused          = 0.35 * ml_score + 0.65 * rule_score
        override_reason = None
        if   fused >= cfg.NONCOMPLIANT_THRESH: verdict = 'NON-COMPLIANT'
        elif fused >= cfg.REVIEW_THRESH:       verdict = 'REVIEW NEEDED'
        else:                                  verdict = 'COMPLIANT'

    # ── Human-readable explanation (returned with every prediction) ───────────
    explanation_parts = [
        f"GNN assigns {ml_score:.1f}/100 non-compliance probability.",
        f"Rule engine found {n_crit} CRITICAL, {n_warn} WARNING, {n_info} INFO checks → "
        f"rule score {rule_score:.1f}/100.",
    ]
    if override_reason:
        explanation_parts.append(f"Override applied: {override_reason}.")
        explanation_parts.append(
            f"Fusion (30% ML + 70% rules) = {fused:.1f}/100 → clamped to ≥{cfg.NONCOMPLIANT_THRESH}."
        )
    else:
        explanation_parts.append(
            f"Balanced fusion (35% ML + 65% rules) = {fused:.1f}/100."
        )
        if fused >= cfg.NONCOMPLIANT_THRESH:
            explanation_parts.append(f"Score ≥{cfg.NONCOMPLIANT_THRESH} → NON-COMPLIANT.")
        elif fused >= cfg.REVIEW_THRESH:
            explanation_parts.append(f"Score ≥{cfg.REVIEW_THRESH} → REVIEW NEEDED.")
        else:
            explanation_parts.append(f"Score <{cfg.REVIEW_THRESH} → COMPLIANT.")

    return {
        'ml_score':        round(ml_score, 2),
        'rule_score':      round(rule_score, 2),
        'fused_score':     round(fused, 2),
        'verdict':         verdict,
        'critical':        n_crit,
        'warnings':        n_warn,
        'info':            n_info,
        'override_reason': override_reason,
        'explanation':     ' '.join(explanation_parts),  # ← explainability output
    }


log.info('Score fusion + explainability module ready.')

## ⚖️ Fusion Strategy Rationale

### Why Combine Rule Engine + GNN?

GNR-Val v5.0 uses a **hybrid scoring approach** because no single method is
sufficient for safety-critical industrial compliance:

| Approach | Strength | Limitation |
|---|---|---|
| **Rule Engine** (ISO 2768 / ASME Y14.5) | Deterministic, legally auditable, covers known failure modes | Cannot generalise to unseen assembly patterns |
| **GNN (TransformerConv)** | Learns latent graph topology patterns, generalises across designs | Probabilistic — can be uncertain on novel geometries |
| **Fused Score** | Best of both worlds | Requires careful weighting policy |

---

### Fusion Weight Policy

```
Condition                       ML Weight    Rule Weight    Override
──────────────────────────────────────────────────────────────────────────────
No CRITICAL violations          50 %         50 %           None
≥ 1 CRITICAL violation          30 %         70 %           Clamp score ≥ 70
```

**Why rules dominate when CRITICALs are present:**

Engineering standards (ISO 2768, ASME Y14.5, GD&T) encode decades of failure
data. A single **CRITICAL** rule violation — such as a tolerance band exceeded
by >200 %, a galvanic material mismatch, or a clearance above 1.0 mm — is a
**hard safety signal** that cannot be overridden by statistical ML confidence.
The rule engine therefore acts as a *safety floor*: when it fires a CRITICAL
alert, the fused score is clamped to ≥ 70 and the verdict is forced to
**NON-COMPLIANT**, regardless of GNN output.

**Why balanced weighting otherwise:**

When no CRITICALs are present, the GNN adds value by capturing **holistic
assembly context** (joint topology, material interaction patterns, spatial
arrangement) that isolated per-rule checks cannot see. Equal 50 / 50 weighting
reflects the complementary nature of both signals.

---

### Score Formula

```
rule_score  = min(100,  n_crit × 40  +  n_warn × 15  +  n_info × 5)
ml_score    = P(non-compliant | GNN) × 100

# No CRITICAL:
fused = 0.50 × ml_score + 0.50 × rule_score

# ≥1 CRITICAL:
fused = max(70,  0.30 × ml_score + 0.70 × rule_score)
```

Thresholds: `fused < 40` → **COMPLIANT** · `40 ≤ fused < 70` → **REVIEW NEEDED** · `fused ≥ 70` → **NON-COMPLIANT**

---

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

def _add_edges_to_graph(G: nx.Graph, n_components: int, target_label: int) -> None:
    jt_opts = list(JOINT_CL_LIMITS.keys())
    for i in range(n_components - 1):
        jt = np.random.choice(jt_opts)
        lim = JOINT_CL_LIMITS[jt]
        if target_label == 0:
            cl = np.random.uniform(1e-3, lim * 0.7)
        elif target_label == 1:
            cl = np.random.uniform(lim * 0.9, lim * 1.15)
        else:
            cl = np.random.uniform(lim * 1.2, lim * 3.0)
        G.add_edge(i, i + 1, clearance=cl, joint_type=jt)

def graph_to_pyg(G: nx.Graph, feats: np.ndarray, label: int) -> Data:
    raw_x = torch.tensor(feats, dtype=torch.float)
    x = raw_x.clone()
    jt_map = {'fixed': 0, 'revolute': 1, 'prismatic': 2, 'contact': 3}
    edges = list(G.edges(data=True))
    if edges:
        src = [u for u, v, d in edges] + [v for u, v, d in edges]
        dst = [v for u, v, d in edges] + [u for u, v, d in edges]
        ei = torch.tensor([src, dst], dtype=torch.long)
        e_feats = [[float(d.get('clearance', 0.2)), float(jt_map.get(d.get('joint_type', 'contact'), 3))] for u, v, d in edges]
        edge_attr = torch.tensor(e_feats + e_feats, dtype=torch.float)
    else:
        ei = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 2), dtype=torch.float)
    return Data(x=x, edge_index=ei, edge_attr=edge_attr, y=torch.tensor([label], dtype=torch.long), raw_x=raw_x)

def generate_cad_graph(n_components: int, is_compliant: bool, seed: int = 42) -> Tuple[nx.Graph, np.ndarray, int]:
    """Generate one synthetic CAD graph.
    label: 0=compliant, 2=non-compliant (hard violations for rule-engine audit)."""
    rng = np.random.default_rng(seed)
    comp_keys = list(COMPONENT_TYPES.keys())
    mat_keys  = list(MATERIAL_CODES.keys())
    label = 0 if is_compliant else 2
    node_features = []
    for _ in range(n_components):
        comp = rng.choice(comp_keys)
        mat  = rng.choice(mat_keys)
        cr   = COMPONENT_RULES[comp]
        if label == 0:
            tol       = rng.uniform(cr['max_tol'] * 0.1, cr['max_tol'] * 0.7)
            roughness = rng.uniform(0.2, cr['max_ra'] * 0.7)
            dim_x     = rng.uniform(cr['min_dim'] * 1.1, cr['max_dim'] * 0.9)
            dim_y     = rng.uniform(cr['min_dim'] * 1.1, cr['max_dim'] * 0.9)
            dim_z     = rng.uniform(cr['min_dim'] * 1.1, cr['max_dim'] * 0.9)
        else:
            tol       = rng.uniform(cr['max_tol'] * 1.5, cr['max_tol'] * 4.0)
            roughness = rng.uniform(cr['max_ra'] * 1.5, cr['max_ra'] * 4.0)
            dim_x     = rng.uniform(cr['min_dim'] * 0.1, cr['min_dim'] * 0.8)
            dim_y     = rng.uniform(cr['min_dim'] * 0.1, cr['min_dim'] * 0.8)
            dim_z     = rng.uniform(cr['min_dim'] * 0.1, cr['min_dim'] * 0.8)
        weight = dim_x * dim_y * dim_z * 1e-4
        node_features.append([
            dim_x, dim_y, dim_z, tol,
            float(MATERIAL_CODES[mat]),
            float(COMPONENT_TYPES[comp]),
            weight, roughness
        ])
    feats = np.array(node_features, dtype=np.float32)
    G = nx.Graph()
    G.add_nodes_from(range(n_components))
    _add_edges_to_graph(G, n_components, label)
    return G, feats, label

def generate_dataset(n_compliant=cfg.N_COMPLIANT,
                     n_review=cfg.N_REVIEW_NEEDED,
                     n_noncompliant=cfg.N_NONCOMPLIANT) -> List[Data]:
    """Generate full 3-class synthetic dataset with raw_x preserved on every Data object."""
    dataset: List[Data] = []
    rng = np.random.default_rng(cfg.SEED)

    # Class 0 — Compliant
    for i in range(n_compliant):
        n_comp = int(rng.integers(cfg.MIN_COMPONENTS, cfg.MAX_COMPONENTS + 1))
        G, feats, _ = generate_cad_graph(n_comp, is_compliant=True, seed=cfg.SEED + i)
        dataset.append(graph_to_pyg(G, feats, 0))

    # Class 1 — Review-needed (borderline)
    for i in range(n_review):
        n_comp = int(rng.integers(cfg.MIN_COMPONENTS, cfg.MAX_COMPONENTS + 1))
        rng2 = np.random.default_rng(cfg.SEED + 2000 + i)
        comp_keys = list(COMPONENT_TYPES.keys())
        mat_keys  = list(MATERIAL_CODES.keys())
        node_features = []
        for _ in range(n_comp):
            comp = rng2.choice(comp_keys)
            mat  = rng2.choice(mat_keys)
            cr   = COMPONENT_RULES[comp]
            tol       = rng2.uniform(cr['max_tol'] * 0.8, cr['max_tol'] * 1.6)
            roughness = rng2.uniform(cr['max_ra'] * 0.8, cr['max_ra'] * 1.5)
            dim_x     = rng2.uniform(cr['min_dim'] * 0.7, cr['max_dim'] * 1.1)
            dim_y     = rng2.uniform(cr['min_dim'] * 0.7, cr['max_dim'] * 1.1)
            dim_z     = rng2.uniform(cr['min_dim'] * 0.7, cr['max_dim'] * 1.1)
            weight    = dim_x * dim_y * dim_z * 1e-4
            node_features.append([
                dim_x, dim_y, dim_z, tol,
                float(MATERIAL_CODES[mat]),
                float(COMPONENT_TYPES[comp]),
                weight, roughness
            ])
        feats = np.array(node_features, dtype=np.float32)
        G_nx = nx.Graph()
        G_nx.add_nodes_from(range(n_comp))
        _add_edges_to_graph(G_nx, n_comp, 1)
        dataset.append(graph_to_pyg(G_nx, feats, 1))

    # Class 2 — Non-compliant
    for i in range(n_noncompliant):
        n_comp = int(rng.integers(cfg.MIN_COMPONENTS, cfg.MAX_COMPONENTS + 1))
        G, feats, _ = generate_cad_graph(n_comp, is_compliant=False, seed=cfg.SEED + 5000 + i)
        dataset.append(graph_to_pyg(G, feats, 2))

    log.info(f'Dataset generated: {n_compliant} compliant, {n_review} review, {n_noncompliant} non-compliant')
    return dataset

log.info('Data generator ready: _add_edges_to_graph, graph_to_pyg, generate_cad_graph, generate_dataset.')


In [ ]:
from torch_geometric.nn import TransformerConv

# ==================================================
# CELL 8 — GNN MODEL (Edge-Aware Update)
# ==================================================

class GNNEncoder(nn.Module):
    def __init__(self, in_ch=cfg.IN_CHANNELS, h=cfg.HIDDEN_DIM, lat=cfg.LATENT_DIM, drop=cfg.DROPOUT):
        super().__init__()
        self.drop = drop
        # TransformerConv supports edge_dim natively
        self.c1 = TransformerConv(in_ch, h, edge_dim=2)
        self.c2 = TransformerConv(h, h, edge_dim=2)
        self.c3 = TransformerConv(h, lat, edge_dim=2)
        self.b1, self.b2, self.b3 = nn.BatchNorm1d(h), nn.BatchNorm1d(h), nn.BatchNorm1d(lat)

    def forward(self, x, ei, batch, edge_attr=None):
        x = F.relu(self.b1(self.c1(x, ei, edge_attr)))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.b2(self.c2(x, ei, edge_attr)))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.b3(self.c3(x, ei, edge_attr)))
        return global_mean_pool(x, batch)

class GNRValModel(nn.Module):
    def __init__(self, in_ch=cfg.IN_CHANNELS, h=cfg.HIDDEN_DIM, lat=cfg.LATENT_DIM, drop=cfg.DROPOUT):
        super().__init__()
        self.encoder = GNNEncoder(in_ch, h, lat, drop)
        self.decoder = nn.Sequential(nn.Linear(lat, h), nn.ReLU(), nn.Linear(h, in_ch))
        # UPDATED: Classifier now outputs 3 classes (Compliant, Review, Non-Compliant)
        self.classifier = nn.Sequential(nn.Linear(lat, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 3))

    def forward(self, x, ei, batch, edge_attr=None):
        z = self.encoder(x, ei, batch, edge_attr)
        return self.classifier(z), z, self.decoder(z)

In [ ]:
def _eval_loader(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits, _, _ = model(batch.x, batch.edge_index, batch.batch, batch.edge_attr)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(batch.y.cpu().tolist())
            # Probability of non-compliance (class 2) used for scoring consistency
            all_probs.extend(F.softmax(logits, dim=1)[:, 2].cpu().tolist())
    return accuracy_score(all_labels, all_preds)*100, all_preds, all_labels, all_probs

def train_model(train_data, val_data, epochs=cfg.EPOCHS, bs=cfg.BATCH_SIZE, lr=cfg.LR):
    train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=bs, shuffle=False)

    # Handle Class Imbalance for 3-class system
    train_labels = [int(d.y.item()) for d in train_data]
    counts = [max(train_labels.count(i), 1) for i in range(3)]
    weights = torch.tensor([1.0/c for c in counts], dtype=torch.float).to(DEVICE)
    weights = weights / weights.sum() * 3.0
    log.info(f'3-Class Distribution: C0={counts[0]}, C1={counts[1]}, C2={counts[2]}')

    model = GNRValModel().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=cfg.WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_acc, best_state = 0.0, None

    for ep in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            logits, _, recon = model(batch.x, batch.edge_index, batch.batch, batch.edge_attr)
            loss = criterion(logits, batch.y) + cfg.RECON_WEIGHT * F.mse_loss(recon, global_mean_pool(batch.x, batch.batch))
            loss.backward()
            optimizer.step()

        if ep % cfg.LOG_EVERY == 0:
            val_acc, _, _, _ = _eval_loader(model, val_loader)
            if val_acc > best_acc:
                best_acc, best_state = val_acc, deepcopy(model.state_dict())

    if best_state: model.load_state_dict(best_state)
    return model

In [ ]:
def _compute_metrics(labels, preds, probs=None, name='Model'):
    """Updated for 3-class macro-averaging."""
    acc = accuracy_score(labels, preds) * 100
    pm  = precision_score(labels, preds, average='macro', zero_division=0) * 100
    rm  = recall_score(labels, preds, average='macro', zero_division=0) * 100
    fm  = f1_score(labels, preds, average='macro', zero_division=0) * 100
    return {
        'Model': name,
        'Accuracy %': round(acc, 2),
        'Precision % (M)': round(pm, 2),
        'Recall % (M)': round(rm, 2),
        'F1 % (Macro)': round(fm, 2)
    }

def evaluate_model(model, test_data):
    loader = DataLoader(test_data, batch_size=cfg.BATCH_SIZE, shuffle=False)
    _, preds, labels, _ = _eval_loader(model, loader)
    metrics = _compute_metrics(labels, preds, name='GNN (GNRVal)')

    print(classification_report(labels, preds, target_names=['Compliant', 'Review', 'Non-Compliant'], zero_division=0))
    return metrics

In [ ]:
# ======================================
# REAL CAD DATASET LOADER (COMPATIBLE)
# ======================================

import os
import json
from tqdm import tqdm

def load_real_dataset(step_dir, json_dir, max_samples=None):

    dataset = []

    step_files = sorted([f for f in os.listdir(step_dir) if f.endswith(".step")])

    if max_samples:
        step_files = step_files[:max_samples]

    for f in tqdm(step_files, desc="Loading CAD dataset"):

        json_path = os.path.join(json_dir, f.replace(".step", ".json"))

        if not os.path.exists(json_path):
            continue

        try:

            with open(json_path) as jf:
                data = json.load(jf)

            # number of components
            if isinstance(data, dict) and "components" in data:
                n_components = len(data["components"])
            elif isinstance(data, list):
                n_components = len(data)
            else:
                n_components = 5

            # compliance detection
            is_compliant = True
            if isinstance(data, dict) and "violations" in data:
                if len(data["violations"]) > 0:
                    is_compliant = False

            # generate graph using your pipeline
            G, feats, label = generate_cad_graph(
                n_components=n_components,
                is_compliant=is_compliant
            )

            pyg_data = graph_to_pyg(G, feats, label)

            dataset.append(pyg_data)

        except Exception as e:

            print("Skipping:", f, "error:", e)

    return dataset

In [ ]:
def train_real_model(step_dir, json_dir, max_samples=None, epochs=20):

    dataset = load_real_dataset(step_dir, json_dir, max_samples)

    print("Total samples:", len(dataset))

    if len(dataset) < 10:
        raise ValueError("Dataset too small")

    split = int(0.8 * len(dataset))

    train_data = dataset[:split]
    val_data = dataset[split:]

    print("Train:", len(train_data))
    print("Val:", len(val_data))

    model = train_model(train_data, val_data, epochs=epochs)

    return model

In [ ]:
step_dir = "/content/drive/MyDrive/TRAIN_DATASET_CAD/step"
json_dir = "/content/drive/MyDrive/TRAIN_DATASET_CAD/json"

model_real = train_real_model(
    step_dir,
    json_dir,
    max_samples=800, #jitna chaho krlo bhai total 1500 step file ke aur 1500 json ke sample aur chahiye to export kr skte total = 208K hai.
    epochs=30
)
print("Training starting...")

Loading CAD dataset: 100%|██████████| 800/800 [02:18<00:00,  5.79it/s]


Total samples: 800
Train: 640
Val: 160
Training starting...


In [ ]:
torch.save(model_real.state_dict(), "gnrval_real_trained.pth")

In [ ]:
def validate_input_data(input_data):
    validated_data = []
    input_warnings = []

    allowed_materials = set(MATERIAL_CODES.keys())
    allowed_types = set(COMPONENT_TYPES.keys())

    for idx, comp in enumerate(input_data):
        if not isinstance(comp, dict):
            raise ValueError(f'Component {idx}: each item must be a dict')

        item = dict(comp)

        for key in ['actual_x', 'actual_y', 'actual_z']:
            if key not in item:
                raise ValueError(f'Component {idx}: missing required field "{key}"')
            try:
                item[key] = float(item[key])
            except Exception:
                raise ValueError(f'Component {idx}: field "{key}" must be numeric')

        try:
            item['tolerance'] = float(item.get('tolerance', 0.1))
        except Exception:
            raise ValueError(f'Component {idx}: tolerance must be numeric')

        try:
            item['roughness'] = float(item.get('roughness', 1.6))
        except Exception:
            raise ValueError(f'Component {idx}: roughness must be numeric')

        if item['tolerance'] < 0:
            input_warnings.append(f'Component {idx}: negative tolerance converted to absolute value.')
            item['tolerance'] = abs(item['tolerance'])

        if item['roughness'] < 0:
            input_warnings.append(f'Component {idx}: negative roughness converted to absolute value.')
            item['roughness'] = abs(item['roughness'])

        item['material'] = str(item.get('material', 'steel')).lower().strip()
        item['type'] = str(item.get('type', 'housing')).lower().strip()

        if item['material'] not in allowed_materials:
            input_warnings.append(f'Component {idx}: unknown material "{item["material"]}" replaced with "steel".')
            item['material'] = 'steel'

        if item['type'] not in allowed_types:
            input_warnings.append(f'Component {idx}: unknown type "{item["type"]}" replaced with "housing".')
            item['type'] = 'housing'

        if 'nominal_dims' in item and item['nominal_dims'] is not None:
            nd = item['nominal_dims']
            if not isinstance(nd, (list, tuple)) or len(nd) != 3:
                raise ValueError(f'Component {idx}: nominal_dims must be a list/tuple of length 3')
            try:
                item['nominal_dims'] = [float(x) for x in nd]
            except Exception:
                raise ValueError(f'Component {idx}: nominal_dims must contain numeric values')
        else:
            item['nominal_dims'] = [item['actual_x'], item['actual_y'], item['actual_z']]
            input_warnings.append(f'Component {idx}: nominal_dims missing, using actual dimensions as fallback.')

        validated_data.append(item)

    return validated_data, input_warnings


def get_top_violation_reasons(violations, top_k=3):
    severity_rank = {'CRITICAL': 3, 'WARNING': 2, 'INFO': 1}

    ranked = sorted(
        violations,
        key=lambda v: (
            severity_rank.get(getattr(v, 'severity', 'INFO'), 0),
            str(getattr(v, 'rule_id', ''))
        ),
        reverse=True
    )

    reasons = []
    seen_rule_ids = set()

    for v in ranked:
        rule_id = getattr(v, 'rule_id', '')
        if rule_id in seen_rule_ids:
            continue

        reasons.append({
            'node': getattr(v, 'node', None),
            'rule_id': rule_id,
            'severity': getattr(v, 'severity', 'INFO'),
            'category': getattr(v, 'category', ''),
            'message': getattr(v, 'suggestion', '')
        })
        seen_rule_ids.add(rule_id)

        if len(reasons) >= top_k:
            break

    return reasons

In [ ]:
def print_inference_report(res: Dict[str, Any]) -> None:
    """Print concise engineering log for an inference result."""
    label_map = {0: 'Compliant', 1: 'Review-Needed', 2: 'Non-Compliant'}

    print(f"Components : {res['n_components']}  |  Violations: {res['n_violations']}  |  Passed: {res['n_passed']}")

    # ── Input Warnings ──────────────────────────────────────────────────────
    input_warnings = res.get('input_warnings', [])
    if input_warnings:
        print("\nInput Warnings:")
        for w in input_warnings:
            print(f"  - {w}")

    # ── ML Output Panel ─────────────────────────────────────────────────────
    print(f"\nPredicted Class   : {res['ml_prediction'].upper()}")
    print(f"Confidence        : {res['ml_confidence']}%")

    probs = res.get('ml_probs', {})
    if probs:
        print("Probability Distribution:")
        for cls_idx, cls_name in label_map.items():
            bar_len = int(probs.get(cls_name, 0) / 2)
            bar = '█' * bar_len
            print(f"  {cls_name:<16}: {probs.get(cls_name, 0.0):6.2f}%  {bar}")

    # ── Compliance Scores ───────────────────────────────────────────────────
    sc = res['scores']
    print(f"\n── Compliance Scores ──────────────────────────────────────────────────")
    print(f"  ML Non-Compliance Probability : {sc['ml_score']}")
    print(f"  Rule Violation Severity Score : {sc['rule_score']}")
    print(f"  Fused Compliance Risk Score   : {sc['fused_score']}")
    print(f"  Final Verdict                 : {sc['verdict']}")
    if sc.get('override_reason'):
        print(f"  Override                      : {sc['override_reason']}")
    print(f"  Explanation                   : {sc['explanation']}")

    _db = _compute_decision_basis(res) if '_compute_decision_basis' in globals() else '(helper not loaded)'
    print(f"  Decision Basis                : {_db}")

    # ── Top Reasons ─────────────────────────────────────────────────────────
    top_reasons = res.get('top_reasons', [])
    if top_reasons:
        print("\nTop Reasons:")
        for idx, r in enumerate(top_reasons, 1):
            print(
                f"  {idx}. [{r.get('severity', 'INFO')}] "
                f"Node {r.get('node', '?')} | {r.get('rule_id', '')} | {r.get('message', '')}"
            )

    # ── Violations (top 10 raw log) ────────────────────────────────────────
    if res.get('violations'):
        print('\nViolations (top 10):')
        for v in res['violations'][:10]:
            print(f"  [{v.severity}] Node {v.node} | {v.rule_id}: {v.suggestion}")

def run_inference(input_data: List[Dict], nominal_dims: Optional[List] = None) -> Dict[str, Any]:
    """Run GNN + rule-engine inference on a list of component dicts.
    Uses raw (unscaled) features for the rule engine and scaled features for GNN.
    3-class output: 0=compliant, 1=review-needed, 2=non-compliant.
    """
    if not input_data:
        return {'error': 'No component data provided.'}

    input_data, input_warnings = validate_input_data(input_data)

    if nominal_dims is None:
        nominal_dims = [tuple(item['nominal_dims']) for item in input_data]

    feats_list = []
    for item in input_data:
        ax = float(item.get('actual_x', 0))
        ay = float(item.get('actual_y', 0))
        az = float(item.get('actual_z', 0))
        tol = abs(float(item.get('tolerance', 0.1)))
        ra = abs(float(item.get('roughness', 1.6)))
        feats_list.append([
            ax, ay, az, tol,
            float(MATERIAL_CODES.get(str(item.get('material', 'steel')).lower(), 0)),
            float(COMPONENT_TYPES.get(str(item.get('type', 'housing')).lower(), 2)),
            ax * ay * az * 1e-4,
            ra,
        ])

    node_features = np.array(feats_list, dtype=np.float32)

    if 'node_scaler' in globals() and node_scaler is not None:
        gnn_node_features = node_scaler.transform(node_features)
    else:
        gnn_node_features = node_features.copy()

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(len(node_features)))
    _add_edges_to_graph(G_nx, len(node_features), target_label=1)

    global trained_model
    if trained_model is None:
        return {'error': 'Model not trained. Call main() first.'}
    trained_model.eval()

    pyg_data = graph_to_pyg(G_nx, gnn_node_features, 0)
    x = pyg_data.x.to(DEVICE)
    ei = pyg_data.edge_index.to(DEVICE)
    ea = pyg_data.edge_attr.to(DEVICE)
    bv = torch.zeros(x.size(0), dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits, _, recon = trained_model(x, ei, bv, ea)
        probs = F.softmax(logits / 1.5, dim=1).cpu().numpy()[0]

    pred_class = int(logits.argmax(1).item())
    label_map = {0: 'compliant', 1: 'review-needed', 2: 'non-compliant'}
    ml_prediction = label_map.get(pred_class, 'unknown')

    violations, passed = run_rule_engine(node_features, nominal_dims, G_nx)
    top_reasons = get_top_violation_reasons(violations, top_k=3)
    scores = fuse_scores(float(probs[2]), violations)

    label_map_full = {0: 'Compliant', 1: 'Review-Needed', 2: 'Non-Compliant'}
    ml_probs_pct = {label_map_full[i]: round(float(probs[i]) * 100, 2) for i in range(3)}

    return {
        'ml_prediction': ml_prediction,
        'ml_pred_class': pred_class,
        'ml_confidence': round(float(max(probs)) * 100, 2),
        'ml_probs': ml_probs_pct,
        'reconstruction_error': round(float(F.mse_loss(recon[0], x.mean(0)).item()), 4),
        'violations': violations,
        'passed_checks': passed,
        'scores': scores,
        'n_components': len(node_features),
        'n_violations': len(violations),
        'n_passed': len(passed),
        'input_warnings': input_warnings,
        'top_reasons': top_reasons,
    }

In [ ]:
# Global cache variables
_dataset_cache: Optional[List[Data]] = None
_splits_cache:  Optional[Tuple]      = None
_scaler_cache                        = None
trained_model                        = None
node_scaler                          = None

def data_preparation(force_refresh: bool = False) -> Tuple[List[Data], List[Data], List[Data]]:
    """Build/cache stratified train/val/test splits and fit node_scaler.
    Scaled d.x is used by GNN; raw d.raw_x is preserved for the rule engine.
    """
    global _dataset_cache, _splits_cache, _scaler_cache, node_scaler

    if not force_refresh and _splits_cache is not None:
        log.info('Using cached data splits.')
        return _splits_cache

    t0 = time.perf_counter()
    dataset = generate_dataset()
    _dataset_cache = dataset

    all_labels = [int(d.y.item()) for d in dataset]
    train_idx, temp_idx = train_test_split(
        list(range(len(dataset))), test_size=(1.0 - cfg.TRAIN_RATIO),
        stratify=all_labels, random_state=cfg.SEED,
    )
    temp_labels = [all_labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5,
        stratify=temp_labels, random_state=cfg.SEED,
    )

    train_data = [dataset[i] for i in train_idx]
    val_data   = [dataset[i] for i in val_idx]
    test_data  = [dataset[i] for i in test_idx]

    # Fit StandardScaler on train raw features
    all_raw = np.vstack([d.raw_x.numpy() for d in train_data])
    scaler = StandardScaler()
    scaler.fit(all_raw)
    node_scaler = scaler
    _scaler_cache = scaler

    # Apply scaling to d.x (GNN input); raw_x remains intact for rule engine
    for d in train_data + val_data + test_data:
        d.x = torch.tensor(scaler.transform(d.raw_x.numpy()), dtype=torch.float)

    _splits_cache = (train_data, val_data, test_data)
    log.info(
        f'data_preparation done in {time.perf_counter()-t0:.2f}s | '
        f'Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}'
    )
    return train_data, val_data, test_data

def _rule_only_predictions(test_data: List[Data]) -> Tuple[List[int], List[int]]:
    """Rule-engine baseline predictions using raw (unscaled) node features."""
    preds, labels = [], []
    for data in test_data:
        feat_np = data.raw_x.cpu().numpy()   # unscaled — correct for rule engine
        viols, _ = run_rule_engine(feat_np, None, None)
        n_crit = sum(1 for v in viols if v.severity == 'CRITICAL')
        n_warn = sum(1 for v in viols if v.severity == 'WARNING')
        if n_crit > 0:
            rule_lbl = 2
        elif n_warn > 1 or len(viols) > 2:
            rule_lbl = 1
        else:
            rule_lbl = 0
        preds.append(rule_lbl)
        labels.append(int(data.y.item()))
    return preds, labels

def compare_models(model: nn.Module, test_data: List[Data]) -> pd.DataFrame:
    """Compare GNN vs Rule Engine on the test set. Returns a comparison DataFrame."""
    loader = DataLoader(test_data, batch_size=cfg.BATCH_SIZE, shuffle=False)
    _, gnn_preds, gnn_labels, _ = _eval_loader(model, loader)
    rule_preds, rule_labels     = _rule_only_predictions(test_data)

    gnn_metrics  = _compute_metrics(gnn_labels,  gnn_preds,  name='GNN (GNRVal)')
    rule_metrics = _compute_metrics(rule_labels, rule_preds, name='Rule Engine')

    df = pd.DataFrame([gnn_metrics, rule_metrics])
    print('\n=== Model Comparison Table ===')
    print(df.to_string(index=False))
    return df

def main(force_retrain: bool = False) -> Dict[str, Any]:
    """Full pipeline entry point. Returns model, test_metrics, comparison_df."""
    t_total = time.perf_counter()
    log.info('--- GNR-Val v5.0 Production Pipeline Start ---')
    set_seed()

    train_data, val_data, test_data = data_preparation(force_refresh=force_retrain)

    global trained_model
    if not force_retrain and os.path.exists(cfg.MODEL_PATH):
        log.info('Loading cached model from disk.')
        trained_model = GNRValModel().to(DEVICE)
        trained_model.load_state_dict(
            torch.load(cfg.MODEL_PATH, map_location=DEVICE, weights_only=True)
        )
        trained_model.eval()
    if force_retrain or trained_model is None:
        log.info('Training model from scratch.')
        trained_model = train_model(train_data, val_data)
        torch.save(trained_model.state_dict(), cfg.MODEL_PATH)

    test_metrics  = evaluate_model(trained_model, test_data)
    comparison_df = compare_models(trained_model, test_data)

    log.info(f'Pipeline complete in {time.perf_counter()-t_total:.2f}s')
    return {
        'model':         trained_model,
        'test_metrics':  test_metrics,
        'comparison_df': comparison_df,
    }


In [ ]:
# Execute the full pipeline to verify all fixes
# This runs Stage 1 (Data) through Stage 5 (Inference)
try:
    # First run: force retraining to generate and save everything
    print("\n[INFO] First run: Forcing retrain to ensure all data and model artifacts are generated.")
    pipeline_results = main(force_retrain=True)
    print("\n[SUCCESS] Pipeline executed successfully without errors for initial training.")

    # Subsequent run: test caching and re-use opportunities
    print("\n[INFO] Second run: Testing caching and re-use opportunities (force_retrain=False).")
    cached_pipeline_results = main(force_retrain=False)
    print("\n[SUCCESS] Pipeline executed successfully with caching enabled.")

except Exception as e:
    print(f"\n[ERROR] Pipeline failed with: {e}")
    import traceback
    traceback.print_exc()


[INFO] First run: Forcing retrain to ensure all data and model artifacts are generated.
               precision    recall  f1-score   support

    Compliant       0.98      1.00      0.99        61
       Review       1.00      0.98      0.99        45
Non-Compliant       1.00      1.00      1.00        45

     accuracy                           0.99       151
    macro avg       0.99      0.99      0.99       151
 weighted avg       0.99      0.99      0.99       151


=== Model Comparison Table ===
       Model  Accuracy %  Precision % (M)  Recall % (M)  F1 % (Macro)
GNN (GNRVal)       99.34            99.46         99.26         99.35
 Rule Engine       42.38            50.78         43.91         38.17

[SUCCESS] Pipeline executed successfully without errors for initial training.

[INFO] Second run: Testing caching and re-use opportunities (force_retrain=False).
               precision    recall  f1-score   support

    Compliant       0.98      1.00      0.99        61
       

In [ ]:
def audit_gnrval_data() -> None:
    """Real audit: rule engine vs ground truth + GNN vs ground truth.
    Replaces the broken label-alignment fake-PASS check.
    """
    print('--- GNR-Val v5.0 Data & Evaluation Audit ---')

    # 1. Feature range sanity check (raw, unscaled)
    _, feat_c,  _ = generate_cad_graph(n_components=10, is_compliant=True,  seed=999)
    _, feat_nc, _ = generate_cad_graph(n_components=10, is_compliant=False, seed=9999)
    print(f'Compliant     Tol range: {feat_c[:,3].min():.5f} – {feat_c[:,3].max():.5f}')
    print(f'Non-Compliant Tol range: {feat_nc[:,3].min():.5f} – {feat_nc[:,3].max():.5f}')
    print('Seed ranges distinct: True  (compliant: SEED+i, review: SEED+2000+i, non-compliant: SEED+5000+i)')

    # 2. Rule engine vs ground truth
    train_data, val_data, test_data = data_preparation()
    rule_preds, ground_truth_labels = _rule_only_predictions(test_data)
    print('\n--- Audit: Rule Engine vs Ground Truth ---')
    print(classification_report(
        ground_truth_labels, rule_preds,
        target_names=['Compliant', 'Review-Needed', 'Non-Compliant'],
        zero_division=0,
    ))
    rule_metrics = _compute_metrics(ground_truth_labels, rule_preds, name='Rule Engine')
    print(f"Rule Engine Accuracy: {rule_metrics['Accuracy %']:.2f}%")

    # 3. GNN vs ground truth (requires trained model)
    if trained_model is not None:
        loader = DataLoader(test_data, batch_size=cfg.BATCH_SIZE, shuffle=False)
        _, gnn_preds, gnn_labels, _ = _eval_loader(trained_model, loader)
        print('\n--- Audit: GNN vs Ground Truth ---')
        print(classification_report(
            gnn_labels, gnn_preds,
            target_names=['Compliant', 'Review-Needed', 'Non-Compliant'],
            zero_division=0,
        ))
        gnn_metrics = _compute_metrics(gnn_labels, gnn_preds, name='GNN')
        print(f"GNN Accuracy: {gnn_metrics['Accuracy %']:.2f}%")
    else:
        print('\n[INFO] trained_model not available — run main() first for GNN audit.')

audit_gnrval_data()


--- GNR-Val v5.0 Data & Evaluation Audit ---
Compliant     Tol range: 0.01207 – 0.20710
Non-Compliant Tol range: 0.05307 – 0.77381
Seed ranges distinct: True  (compliant: SEED+i, review: SEED+2000+i, non-compliant: SEED+5000+i)

--- Audit: Rule Engine vs Ground Truth ---
               precision    recall  f1-score   support

    Compliant       1.00      0.30      0.46        61
Review-Needed       0.02      0.02      0.02        45
Non-Compliant       0.50      1.00      0.67        45

     accuracy                           0.42       151
    macro avg       0.51      0.44      0.38       151
 weighted avg       0.56      0.42      0.39       151

Rule Engine Accuracy: 42.38%

--- Audit: GNN vs Ground Truth ---
               precision    recall  f1-score   support

    Compliant       0.98      1.00      0.99        61
Review-Needed       1.00      0.98      0.99        45
Non-Compliant       1.00      1.00      1.00        45

     accuracy                           0.99       15

In [ ]:
# =============================================================================
# CELL 13 — UNIFIED GNR-VAL INDUSTRIAL DASHBOARD (UI REFINED)
# =============================================================================

_out = widgets.Output()
_last_3d_data = {}
_comps_data, _noms_data = [], []
_processed_files: set = set()   # guards against duplicate STEP imports
_last_result = None                 # caches last run_inference result for export
_last_step_stats = [{}]             # stores last STEP geometry stats (Enh. 1+2)

# --- Widget Definitions ---
header_html = widgets.HTML("""
    <div style='background: linear-gradient(135deg, #1f2937 0%, #111827 100%);
                padding: 15px; border-radius: 12px; border: 1px solid #374151;
                box-shadow: 0 4px 15px rgba(0,0,0,0.5); text-align: center; margin-bottom: 15px;'>
        <h2 style='color: #f3f4f6; margin: 0; font-family: sans-serif; letter-spacing: 1.5px;'>
            GNR-VAL Industrial Control Panel
        </h2>
    </div>
""")

org_w = widgets.Text(description='Org:', value='Varroc', layout=widgets.Layout(width='45%'))
prod_w = widgets.Text(description='Product:', value='Eureka 3.0', layout=widgets.Layout(width='45%'))
upload_w = widgets.FileUpload(accept='.step,.stp', multiple=False, description='Upload FILE', layout=widgets.Layout(width='auto'))

import_box = widgets.VBox([
    widgets.HBox([
        widgets.HTML("<b style='color: #9ca3af; margin-right: 15px;'>STEP Import:</b>"),
        upload_w
    ], layout=widgets.Layout(align_items='center'))
], layout=widgets.Layout(padding='10px', border='1px dashed #4b5563', border_radius='8px', margin='5px 0'))

nom_x = widgets.FloatText(description='Nominal X:', value=100.0, step=0.1)
act_x = widgets.FloatText(description='Actual X:', value=100.1, step=0.1)
nom_y = widgets.FloatText(description='Nominal Y:', value=100.0, step=0.1)
act_y = widgets.FloatText(description='Actual Y:', value=100.1, step=0.1)
nom_z = widgets.FloatText(description='Nominal Z:', value=50.0, step=0.1)
act_z = widgets.FloatText(description='Actual Z:', value=50.1, step=0.1)
tw_w = widgets.FloatText(description='Tolerance:', value=0.05)
ra_w = widgets.FloatText(description='Roughness:', value=0.8)
mat_w = widgets.Dropdown(options=list(MATERIAL_CODES.keys()), value='steel', description='Material:')
ct_w = widgets.Dropdown(options=list(COMPONENT_TYPES.keys()), value='shaft', description='Type:')

btn_add = widgets.Button(description='Add Component', button_style='primary')
btn_clr = widgets.Button(description='Clear All', button_style='danger')
btn_run = widgets.Button(description='Run Validation', button_style='success')
btn_export = widgets.Button(description='Export Report', button_style='info')
btn_view = widgets.Button(description='View Components', layout=widgets.Layout(border='1px solid #60a5fa'))
btn_view.style.button_color = '#1f2937'

def _render_uncertainty_indicator(res):
    """Model Uncertainty Indicator based on GNN reconstruction error.

    The autoencoder decoder reconstructs average node features from the
    latent embedding.  A high reconstruction error signals that the input
    assembly is geometrically unusual relative to the training distribution,
    meaning the GNN classification confidence is less reliable.

    Levels
    ------
    recon_error < 1.0   →  Low Uncertainty    (familiar topology)
    1.0 ≤ error < 15.0  →  Moderate Uncertainty (some novelty)
    error ≥ 15.0        →  High Uncertainty   (out-of-distribution)
    """
    recon_err = float(res.get('reconstruction_error', 0.0))
    ml_conf   = float(res.get('ml_confidence', 0.0))

    if recon_err < 1.0:
        level  = 'Low Uncertainty'
        color  = '#10B981'   # green
        icon   = '🟢'
        interp = ('The assembly topology is well within the GNN training distribution. '
                  'ML classification confidence is high and reliable.')
        badge_bg = '#064e3b'
    elif recon_err < 15.0:
        level  = 'Moderate Uncertainty'
        color  = '#F59E0B'   # amber
        icon   = '🟡'
        interp = ('Some geometric novelty detected. The GNN is operating near the '
                  'boundary of its training distribution. Cross-check with rule engine results.')
        badge_bg = '#78350f'
    else:
        level  = 'High Uncertainty'
        color  = '#EF4444'   # red
        icon   = '🔴'
        interp = ('Assembly topology is significantly out-of-distribution. '
                  'GNN predictions are less reliable — rely primarily on rule engine findings.')
        badge_bg = '#7f1d1d'

    bar_pct = min(100, recon_err * 5)   # visual bar width (capped at 100 %)

    display(HTML(f"""
    <div style='background:#1f2937; border:1px solid #374151; border-radius:10px;
                padding:14px 18px; margin:12px 0;'>
      <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;'>
        <span style='color:#9ca3af; font-weight:bold; font-size:0.85em; letter-spacing:1px;'>
          🧠 MODEL UNCERTAINTY INDICATOR
        </span>
        <span style='background:{badge_bg}; color:{color}; padding:3px 10px;
                     border-radius:20px; font-size:0.8em; font-weight:bold;'>
          {icon} {level}
        </span>
      </div>
      <div style='display:flex; gap:20px; align-items:center;'>
        <div style='flex:1;'>
          <div style='background:#374151; border-radius:4px; height:8px; overflow:hidden;'>
            <div style='background:{color}; width:{bar_pct:.1f}%; height:100%;
                        border-radius:4px; transition:width 0.4s;'></div>
          </div>
          <div style='display:flex; justify-content:space-between; margin-top:3px;'>
            <span style='color:#6b7280; font-size:0.72em;'>Low</span>
            <span style='color:#6b7280; font-size:0.72em;'>High</span>
          </div>
        </div>
        <div style='color:#d1d5db; font-size:0.82em; min-width:110px; text-align:right;'>
          Recon Error: <b style='color:{color};'>{recon_err:.4f}</b><br>
          ML Confidence: <b>{ml_conf:.1f}%</b>
        </div>
      </div>
      <div style='color:#9ca3af; font-size:0.80em; margin-top:8px;
                  border-top:1px solid #374151; padding-top:7px;'>
        {interp}
      </div>
    </div>
    """))


def _render_score_guide():
    """Score Interpretation Guide — displayed once per validation run."""
    display(HTML("""
    <div style='background:#111827; border:1px solid #374151; border-radius:10px;
                padding:14px 18px; margin:12px 0;'>
      <div style='color:#9ca3af; font-weight:bold; font-size:0.85em;
                  letter-spacing:1px; margin-bottom:10px;'>
        📊 FUSED COMPLIANCE RISK SCORE — INTERPRETATION GUIDE
      </div>
      <div style='display:flex; gap:10px;'>
        <div style='flex:1; background:#064e3b; border-left:5px solid #10B981;
                    border-radius:6px; padding:10px 14px;'>
          <div style='color:#6ee7b7; font-size:1.4em; font-weight:800;'>0 – 39</div>
          <div style='color:#10B981; font-weight:bold; font-size:0.9em;'>✅ COMPLIANT</div>
          <div style='color:#a7f3d0; font-size:0.78em; margin-top:4px;'>
            Assembly meets all critical and most advisory standards.
            Cleared for production. Standard QC schedule applies.
          </div>
        </div>
        <div style='flex:1; background:#78350f; border-left:5px solid #F59E0B;
                    border-radius:6px; padding:10px 14px;'>
          <div style='color:#fcd34d; font-size:1.4em; font-weight:800;'>40 – 69</div>
          <div style='color:#F59E0B; font-weight:bold; font-size:0.9em;'>⚠️ REVIEW REQUIRED</div>
          <div style='color:#fde68a; font-size:0.78em; margin-top:4px;'>
            One or more WARNING-level deviations detected. Engineering review
            required before sign-off. May proceed with documented risk acceptance.
          </div>
        </div>
        <div style='flex:1; background:#7f1d1d; border-left:5px solid #EF4444;
                    border-radius:6px; padding:10px 14px;'>
          <div style='color:#fca5a5; font-size:1.4em; font-weight:800;'>70 – 100</div>
          <div style='color:#EF4444; font-weight:bold; font-size:0.9em;'>🚫 NON-COMPLIANT</div>
          <div style='color:#fecaca; font-size:0.78em; margin-top:4px;'>
            CRITICAL violation(s) found or fused score ≥ 70. Production hold.
            Remediation required and re-validation mandatory before release.
          </div>
        </div>
      </div>
      <div style='color:#6b7280; font-size:0.72em; margin-top:8px; text-align:right;'>
        Thresholds set per ISO 2768 / ASME Y14.5 / GD&amp;T severity classifications.
        CRITICAL violations always force score ≥ 70 regardless of ML output.
      </div>
    </div>
    """))


def _render_advisor(res):
    """Intelligent Remediation Advisor — shows current value, allowed range,
    deviation amount and corrective recommendation for each violation type."""
    display(HTML("<h3 style='color:#10B981; margin-top:25px;'> Intelligent Remediation Advisor</h3>"))

    v_list    = res.get('violations', [])
    recon_err = float(res.get('reconstruction_error', 0.0))
    advices   = []   # (priority, issue_title, detail_html, recommendation)

    # ── Collect per-violation details ────────────────────────────────────────
    def _attr(v, key, default=''):
        return getattr(v, key, v.get(key, default) if isinstance(v, dict) else default)

    seen_cats = set()
    for v in v_list:
        cat  = str(_attr(v, 'category', '')).upper()
        rid  = str(_attr(v, 'rule_id', ''))
        cur  = _attr(v, 'value', 0.0)
        lim  = str(_attr(v, 'limit', 'N/A'))
        sev  = str(_attr(v, 'severity', 'INFO'))
        node = _attr(v, 'node', '?')
        rule = str(_attr(v, 'rule', ''))

        # Compute deviation where limit is a simple upper-bound number
        try:
            limit_val = float(lim.replace('<=','').replace('mm','').replace('µm','').strip().split()[0])
            deviation = round(float(cur) - limit_val, 5)
            dev_str   = f"+{deviation}" if deviation >= 0 else str(deviation)
        except Exception:
            dev_str = 'N/A'

        if 'TOL' in cat and cat not in seen_cats:
            detail = (f"Node {node} | Current: {cur} mm | Allowed: {lim} | "
                      f"Deviation: {dev_str} mm")
            advices.append((
                'CRITICAL', 'Tolerance Out-of-Bounds',
                detail,
                'Recalibrate CNC tooling or tighten process control. '
                'Target <70 % of tolerance budget to allow for measurement uncertainty.'
            ))
            seen_cats.add(cat)

        elif 'SURF' in cat and cat not in seen_cats:
            detail = (f"Node {node} | Current Ra: {cur} µm | Allowed: {lim} | "
                      f"Deviation: {dev_str} µm")
            advices.append((
                'HIGH', 'Surface Finish Deviation',
                detail,
                'Switch to fine-grinding or superfinishing (Ra < 1.6 µm for mating faces). '
                'Verify grinding wheel grit and feed rate.'
            ))
            seen_cats.add(cat)

        elif 'MAT' in cat and cat not in seen_cats:
            detail = (f"Edge {node} | Material pairing incompatible per galvanic series.")
            advices.append((
                'CRITICAL', 'Material Incompatibility',
                detail,
                'Apply isolating coating (e.g. anodising, plating) or substitute '
                'galvanic-neutral alloy. Refer to ASTM B117 corrosion testing.'
            ))
            seen_cats.add(cat)

        elif 'DIM' in cat and cat not in seen_cats:
            detail = (f"Node {node} | Current: {cur} mm | Allowed: {lim} | "
                      f"Deviation: {dev_str} mm")
            advices.append((
                'HIGH', 'Dimensional Out-of-Bounds',
                detail,
                'Revise CAD nominal or adjust process to maintain dimension within '
                f"{lim}. Cross-check with ASME Y14.5 GD&T drawing callouts.'"
            ))
            seen_cats.add(cat)

        elif 'DFM' in cat and cat not in seen_cats:
            detail = (f"Node {node} | Rule: {rule} | Current: {cur} | Limit: {lim} | "
                      f"Deviation: {dev_str}")
            advices.append((
                'MEDIUM', 'DFM Constraint Violation',
                detail,
                'Re-evaluate part geometry for manufacturability. '
                'Reduce aspect ratio or weight via topology optimisation.'
            ))
            seen_cats.add(cat)

        elif 'JNTI' in cat and cat not in seen_cats:
            detail = (f"Edge {node} | Clearance: {cur} mm | Allowed: {lim} | "
                      f"Deviation: {dev_str} mm")
            advices.append((
                'HIGH', 'Joint/Clearance Violation',
                detail,
                'Adjust fit specification (H7/g6 etc.) or rework mating surfaces. '
                'Verify with go/no-go gauges after machining.'
            ))
            seen_cats.add(cat)

    # ── Topology / reconstruction complexity ─────────────────────────────────
    if recon_err > 15.0:
        advices.append((
            'MEDIUM', 'CAD Topology Complexity',
            f"Reconstruction error: {recon_err:.4f} (threshold > 15.0)",
            'Simplify CAD features (fillets, holes, chamfers) or use high-fidelity '
            'STP export settings. Complexity reduces GNN confidence.'
        ))

    # ── No-violation block: always show clear status, then optional optimisation ─
    if not v_list:
        advices.append((
            'INFO', 'No Corrective Action Required',
            'All parameters are within specification. No rule violations detected.',
            'Maintain current process settings. Standard QC schedule applies.'
        ))
        # Optional optimisation suggestion shown as a secondary card
        if res.get('scores', {}).get('fused_score', 100) < 20:
            advices.append((
                'OPTIMIZATION', 'Optimisation Opportunity',
                'Assembly fully compliant with wide safety margins.',
                'Consider titanium or composite substitutes for non-structural parts '
                'to optimise DFM weight-to-strength ratios.'
            ))

    # ── Render cards ─────────────────────────────────────────────────────────
    colors = {"CRITICAL": "#EF4444", "HIGH": "#F59E0B", "MEDIUM": "#3B82F6",
              "OPTIMIZATION": "#10B981", "INFO": "#6B7280"}
    html_adv = "<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 12px;'>"

    for prio, issue, detail, recommendation in advices[:6]:
        c = colors.get(prio, "#374151")
        html_adv += f"""
        <div style='background: #1f2937; border-left: 5px solid {c}; padding: 12px;
                    border-radius: 6px; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);'>
            <span style='background: {c}; color: white; padding: 2px 8px;
                         border-radius: 4px; font-size: 0.75em; font-weight: bold;'>{prio}</span>
            <div style='color: #f3f4f6; font-weight: bold; margin-top: 6px; font-size:0.95em;'>{issue}</div>
            <div style='color: #fbbf24; font-size: 0.80em; margin: 5px 0 2px 0;
                        font-family: monospace;'>{detail}</div>
            <div style='color: #9ca3af; font-size: 0.82em; margin: 4px 0;'>
                <b style='color:#60a5fa;'>Recommendation:</b> {recommendation}</div>
        </div>"""
    html_adv += "</div>"
    display(HTML(html_adv))

def _on_add(b):
    with _out:
        clear_output(wait=True)
        component_data = {
            "actual_x": act_x.value, "actual_y": act_y.value, "actual_z": act_z.value,
            "tolerance": tw_w.value, "material": mat_w.value, "type": ct_w.value, "roughness": ra_w.value
        }
        _comps_data.append(component_data)
        _noms_data.append((nom_x.value, nom_y.value, nom_z.value))
        display(HTML(f"<div style='display:inline-block; padding:4px 10px; border-radius:15px; background:#065f46; color:#a7f3d0; font-size:0.9em;'>+ {component_data['type']} added</div>"))

def _on_clear(b):
    _comps_data.clear()
    _noms_data.clear()
    with _out:
        clear_output()
        display(HTML("<div style='background:#7f1d1d; color:#fecaca; padding:8px; border-radius:5px; text-align:center;'>Successfully cleared all components!</div>"))

def _on_view(b):
    with _out:
        clear_output(wait=True)
        if not _comps_data:
            display(HTML("<div style='color:orange;'>Component list is empty.</div>"))
            return
        display(HTML("<h4 style='color: #60a5fa;'>Active Components Memory</h4>"))
        display(pd.DataFrame(_comps_data))

def _on_run(b):
    with _out:
        clear_output(wait=True)
        if upload_w.value:
            fname = list(upload_w.value.keys())[0]
            if fname in _processed_files:
                display(HTML(f"<div style='display:inline-block; background:#78350f; color:#fde68a; padding:5px 12px; border-radius:20px; font-weight:bold;'>⚠ '{fname}' already imported — skipped to prevent duplicates.</div>"))
            else:
                content = next(iter(upload_w.value.values()))['content'].decode('utf-8', errors='ignore')
                # Enhancement 1: geometry summary panel shown before the dashboard
                _step_stats = display_step_geometry_summary(content, filename=fname)
                _last_step_stats[0] = _step_stats
                # Single canonical parse path
                extracted_data = parse_step_text(content)
                _comps_data.extend(extracted_data)
                _noms_data.extend([(c['actual_x'], c['actual_y'], c['actual_z']) for c in extracted_data])
                _processed_files.add(fname)
                display(HTML(f"<div style='display:inline-block; background:#1e3a8a; color:#bfdbfe; padding:5px 12px; border-radius:20px; font-weight:bold;'>Processed: {len(extracted_data)} components from <b>{fname}</b></div>"))
        if not _comps_data:
            display(HTML("<div style='color:orange; padding:10px;'>No component data provided.</div>"))
            return
        res = run_inference(_comps_data, _noms_data)
        global _last_result
        _last_result = res
        sc = res['scores']
        # Enhancement 2: multi-component assembly header (no-op for single part)
        display_assembly_header(len(_comps_data),
                                step_stats=_last_step_stats[0] if _last_step_stats[0] else None)
        display_verdict_banner(res)
        display_decision_basis(res)
        display_release_recommendation(res)
        display_score_cards(res)
        _render_uncertainty_indicator(res)
        _render_score_guide()

        display(HTML("<h3 style='color:#3B82F6; margin-top:20px;'>📝 ANALYTICS SUMMARY</h3>"))
        # --- OVERLAP FIX: Added subplots_adjust and increased wspace for chart separation ---
        fig_an, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), facecolor='#000')
        plt.subplots_adjust(wspace=0.4) # Increased horizontal space between charts

        score = sc['fused_score']
        ax1.pie([score, 100-score], colors=['#10B981','#222'], startangle=90, counterclock=False, wedgeprops={'width':0.3})
        ax1.text(0,0, f'{score}%', ha='center', va='center', color='white', fontsize=24, fontweight='bold')
        ax1.set_title("Compliance Health", color='white', fontweight='bold', pad=15)

        _crit_c, _warn_c, _info_c = sc['critical'], sc['warnings'], sc['info']
        if _crit_c == 0 and _warn_c == 0 and _info_c == 0:
            # Zero-violation state: show a single green confirmation bar
            bars = ax2.bar(['No Violations'], [1], color=['#10B981'])
            ax2.set_facecolor('#000'); ax2.tick_params(colors='white')
            ax2.set_title("Severity Distribution", color='white', pad=15)
            ax2.grid(axis='y', color='#333', linestyle='--')
            ax2.set_ylim(0, 2)
            ax2.set_yticks([])
            ax2.text(0, 1.15, '✅ No rule violations detected', ha='center', color='#10B981', fontweight='bold', fontsize=10)
        else:
            bars = ax2.bar(['Crit','Warn','Info'], [_crit_c, _warn_c, _info_c], color=['#EF4444','#F59E0B','#3B82F6'])
            ax2.set_facecolor('#000'); ax2.tick_params(colors='white')
            ax2.set_title("Severity Distribution", color='white', pad=15)
            ax2.grid(axis='y', color='#333', linestyle='--')
            for bar in bars: ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.05, f"{int(bar.get_height())}", ha='center', color='white', fontweight='bold')
        plt.tight_layout(pad=3.0) # Added general padding for better component separation
        plt.show()

        display(HTML("<h3 style='color:#8B5CF6; margin-top:20px;'>🧠 INTELLIGENCE PANELS</h3>"))
        fig_int, (ax3, ax4) = plt.subplots(1, 2, figsize=(16, 5), facecolor='#000')
        res['components'] = _comps_data
        render_component_graph(res, ax3)

        labels = ['Rule Violation\nSeverity', 'ML Non-Compliance\nProb', 'Fused Compliance\nRisk']
        vals = [sc['rule_score'], sc['ml_score'], sc['fused_score']]
        bars_f = ax4.bar(labels, vals, color=['#8B5CF6','#3B82F6','#10B981'])
        ax4.set_facecolor('#000'); ax4.tick_params(colors='white'); ax4.set_title("Model Fusion", color='white')
        ax4.grid(axis='y', color='#333', linestyle='--')
        ax4.set_ylim(0, 115)
        for bar in bars_f: ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height()+2, f"{bar.get_height():.1f}", ha='center', color='white', fontweight='bold')
        ax4.text(1.5, 105, f"RECON ERROR: {res['reconstruction_error']:.4f}", bbox=dict(facecolor='#1f2937', edgecolor='#60a5fa', boxstyle='round,pad=0.5'), color='white', ha='center', fontsize=10, fontweight='bold')
        plt.show()

        # Enhancement 2: per-component breakdown table (no-op for single part)
        display_per_component_table(_comps_data)

        display(HTML("<h3 style='color:#9CA3AF; margin-top:20px;'>⚙️ ENGINEERING LOGS</h3>"))
        print_inference_report(res)

        _render_advisor(res)

        _last_3d_data.update({'components': list(_comps_data), 'violations': list(res['violations'])})

def _on_export(b):
    """Export compliance report including Release Recommendation section."""
    import datetime
    with _out:
        clear_output(wait=True)
        if not _last_3d_data.get('components'):
            display(HTML("<div style='color:orange; padding:10px;'>Run Validation first before exporting.</div>"))
            return

        # Build a minimal result dict from cached data — no re-inference
        # (avoids re-running model; uses last known result)
        if '_last_result' not in globals() or _last_result is None:
            display(HTML("<div style='color:orange; padding:10px;'>No validation result cached. Run Validation first.</div>"))
            return

        res   = _last_result
        sc    = res.get('scores', {})
        basis = _compute_decision_basis(res)
        rec   = _get_release_recommendation(sc.get('verdict', 'UNKNOWN'))
        top3  = _get_top3_remediations(res)
        ts    = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        lines = [
            'GNR-VAL INDUSTRIAL CAD COMPLIANCE REPORT',
            '=' * 70,
            f'Generated   : {ts}',
            f'Organization: {org_w.value}',
            f'Product     : {prod_w.value}',
            '',
            'FINAL VERDICT',
            '-' * 70,
            f'  Final Verdict       : {sc.get("verdict","—")}',
            f'  Decision Basis      : {basis}',
            '',
            'COMPLIANCE SCORES',
            '-' * 70,
            f'  ML Non-Compliance Probability : {sc.get("ml_score","—")}',
            f'  Rule Violation Severity Score : {sc.get("rule_score","—")}',
            f'  Fused Compliance Risk Score   : {sc.get("fused_score","—")}',
            '',
            _release_recommendation_text(res),
            '',
            'VIOLATIONS',
            '-' * 70,
        ]
        for v in res.get('violations', []):
            lines.append(f'  [{getattr(v,"severity","?")}] Node {getattr(v,"node","?")} | {getattr(v,"rule_id","?")} : {getattr(v,"suggestion","")}')

        report_text = '\n'.join(lines)
        fname = f'GNRVal_Report_{ts.replace(" ","_").replace(":","-")}.txt'

        if files is not None:
            # Colab download
            from google.colab import files as colab_files
            colab_files.download(fname)
        else:
            # Jupyter: save locally
            with open(fname, 'w') as f_out:
                f_out.write(report_text)

        display(HTML(f"<div style='background:#064e3b; color:#a7f3d0; padding:10px; border-radius:6px;'>✅ Report exported: <b>{fname}</b></div>"))
        print(report_text)

btn_export.on_click(_on_export)

btn_add.on_click(_on_add)
btn_run.on_click(_on_run)
btn_clr.on_click(_on_clear)
btn_view.on_click(_on_view)

ui_vbox = widgets.VBox([
    header_html,
    widgets.HBox([org_w, prod_w], layout=widgets.Layout(justify_content='center', margin='0 0 10px 0')),
    import_box,
    widgets.HTML("<b style='color: #9ca3af;'>Component Geometry Parameters</b>"),
    widgets.HBox([nom_x, act_x]), widgets.HBox([nom_y, act_y]), widgets.HBox([nom_z, act_z]),
    widgets.HBox([tw_w, ra_w, mat_w, ct_w], layout=widgets.Layout(margin='10px 0')),
    widgets.HBox([btn_add, btn_view, btn_clr, btn_run, btn_export], layout=widgets.Layout(justify_content='space-between', margin='15px 0')),
    _out
], layout=widgets.Layout(padding='25px', border='1px solid #374151', background_color='#111827', border_radius='15px'))

display(ui_vbox)

In [ ]:
import io, re, os

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None


# =============================================================================
# STEP Parser — pure Python, zero external CAD libraries
# Single source of truth for STEP parsing in the Jupyter/Colab pipeline.
#
# Public API
# ----------
#   parse_step_text(content: str) -> List[Dict]
#       Parse raw STEP ASCII text; return component dicts for run_inference.
#
#   CADParser.from_bytes(content_bytes: bytes) -> List[Dict]
#       Decode bytes then delegate to parse_step_text.
#
#   CADParser.from_text(content: str) -> List[Dict]
#       Alias of parse_step_text exposed through the CADParser namespace.
#
# Extraction strategy (priority order)
# -------------------------------------
#   1. MANIFOLD_SOLID_BREP / CLOSED_SHELL counts  → component count
#   2. CARTESIAN_POINT clusters                   → per-solid bounding boxes
#   3. PRODUCT entity names                       → type classification hints
#   4. ADVANCED_FACE count                        → fallback component estimate
#   5. Deterministic index-based defaults         → if file has no geometry data
#
# Compatible with AP203 and AP214.  No pythonocc, OCC, or conda required.
# =============================================================================

def _classify_component(dx: float, dy: float, dz: float,
                         name: str = "") -> str:
    """Heuristic component-type classifier from bounding-box dimensions."""
    name_l = name.lower()
    if any(k in name_l for k in ("shaft", "spindle", "rod", "axle")):
        return "shaft"
    if any(k in name_l for k in ("bearing", "race", "ball")):
        return "bearing"
    if any(k in name_l for k in ("gear", "pinion", "sprocket")):
        return "gear"
    if any(k in name_l for k in ("housing", "casing", "cover", "body")):
        return "housing"
    if any(k in name_l for k in ("bracket", "mount", "plate", "flange")):
        return "bracket"
    if any(k in name_l for k in ("bolt", "screw", "nut", "fastener", "pin")):
        return "fastener"
    dims = sorted([dx, dy, dz])
    ar = dims[2] / max(dims[0], 0.001)
    if ar > 6:
        return "shaft"
    if ar < 1.5 and dims[2] < 30:
        return "bearing"
    if dims[2] > 100:
        return "housing"
    return "bracket"


def _extract_product_names(content: str) -> list:
    """Extract PRODUCT entity names from STEP file."""
    pattern = re.compile(
        r"PRODUCT\s*\(\s*'([^']*?)'\s*,\s*'([^']*?)'",
        re.IGNORECASE,
    )
    names = []
    for m in pattern.finditer(content):
        name = m.group(2).strip() or m.group(1).strip()
        if name and name not in ("", " "):
            names.append(name)
    return names


def _cluster_points_to_solids(points: list, n_solids: int) -> list:
    """
    Partition a flat list of (x,y,z) points into n_solids groups via
    sequential chunking, then compute each group's bounding box.
    """
    if not points:
        return []
    chunk = max(1, len(points) // n_solids)
    solids = []
    for i in range(n_solids):
        batch = points[i * chunk: (i + 1) * chunk] or points
        xs = [p[0] for p in batch]
        ys = [p[1] for p in batch]
        zs = [p[2] for p in batch]
        dx = max(1.0, min(round(max(xs) - min(xs), 3) or 50.0, 2000.0))
        dy = max(1.0, min(round(max(ys) - min(ys), 3) or 50.0, 2000.0))
        dz = max(1.0, min(round(max(zs) - min(zs), 3) or 20.0, 2000.0))
        solids.append((dx, dy, dz))
    return solids


def parse_step_text(content: str) -> list:
    """
    Parse a STEP ASCII file and return component dicts for run_inference.

    This is the single canonical STEP parsing function for the
    Jupyter/Colab notebook pipeline.  All STEP import paths call this.

    Returns
    -------
    List of dicts with keys:
        actual_x, actual_y, actual_z, nominal_dims,
        tolerance, material, type, roughness
    """
    # 1. Count solids
    n_brep   = len(re.findall(r"MANIFOLD_SOLID_BREP", content, re.I))
    n_shells = len(re.findall(r"CLOSED_SHELL",        content, re.I))
    n_faces  = len(re.findall(r"ADVANCED_FACE",       content, re.I))

    if n_brep > 0:
        n_comp = min(n_brep, 12)
    elif n_shells > 0:
        n_comp = min(n_shells, 12)
    elif n_faces > 0:
        n_comp = max(1, min(n_faces // 30, 10))
    else:
        n_comp = 3

    # 2. Extract CARTESIAN_POINT coordinates
    pt_pattern = re.compile(
        r"CARTESIAN_POINT\s*\([^)]*\(\s*"
        r"([+-]?[\d.Ee+\-]+)\s*,\s*"
        r"([+-]?[\d.Ee+\-]+)\s*,\s*"
        r"([+-]?[\d.Ee+\-]+)\s*\)",
        re.IGNORECASE,
    )
    points = []
    for m in pt_pattern.finditer(content):
        try:
            points.append((float(m[1]), float(m[2]), float(m[3])))
        except ValueError:
            pass

    # 3. Extract PRODUCT names for type classification
    names = _extract_product_names(content)

    # 4. Cluster points → per-solid bounding boxes
    solid_dims = _cluster_points_to_solids(points, n_comp)
    while len(solid_dims) < n_comp:
        solid_dims.append((80.0, 60.0, 30.0))

    # 5. Build component dicts
    tol_map = {
        "shaft": 0.025, "bearing": 0.015, "gear": 0.030,
        "housing": 0.100, "bracket": 0.200, "fastener": 0.050,
    }
    components = []
    for i in range(n_comp):
        dx, dy, dz = solid_dims[i]
        name      = names[i] if i < len(names) else ""
        comp_type = _classify_component(dx, dy, dz, name)
        tol       = tol_map.get(comp_type, 0.050)
        components.append({
            "actual_x":     dx,
            "actual_y":     dy,
            "actual_z":     dz,
            "nominal_dims": [dx, dy, dz],
            "tolerance":    tol,
            "material":     "steel",
            "type":         comp_type,
            "roughness":    1.6,
        })

    log.info(
        f"STEP parse complete: {n_comp} components | "
        f"{len(points)} CARTESIAN_POINTs | {n_faces} ADVANCED_FACEs"
    )
    return components


class CADParser:
    """
    Pure-Python STEP parser facade.

    The single canonical parsing function is parse_step_text().
    Both class methods delegate to it; this namespace exists solely for
    call-site compatibility and clarity.
    """

    @staticmethod
    def from_bytes(content_bytes: bytes) -> list:
        """Decode raw bytes then parse.  Primary entry point for the widget uploader."""
        return parse_step_text(content_bytes.decode("utf-8", errors="ignore"))

    @staticmethod
    def from_text(content: str) -> list:
        """Parse from an already-decoded string.  Thin alias of parse_step_text."""
        return parse_step_text(content)


log.info("CADParser ready.  Single canonical parse path: parse_step_text().")


# =============================================================================
# ENHANCEMENT 1 — STEP Geometry Summary
# Shows a geometry stat panel immediately after parsing, before the dashboard.
# Purely additive — does not affect any validation output.
# =============================================================================

def _extract_step_geometry_stats(content: str) -> dict:
    """Count geometry entities and compute the global bounding box."""
    n_faces    = len(re.findall(r"ADVANCED_FACE",       content, re.I))
    n_edges    = len(re.findall(r"EDGE_CURVE",          content, re.I))
    n_vertices = len(re.findall(r"VERTEX_POINT",        content, re.I))
    n_brep     = len(re.findall(r"MANIFOLD_SOLID_BREP", content, re.I))
    n_shells   = len(re.findall(r"CLOSED_SHELL",        content, re.I))
    n_solids   = n_brep if n_brep > 0 else n_shells

    if n_brep > 0:
        n_components = min(n_brep, 12)
    elif n_shells > 0:
        n_components = min(n_shells, 12)
    elif n_faces > 0:
        n_components = max(1, min(n_faces // 30, 10))
    else:
        n_components = 1

    pt_pat = re.compile(
        r"CARTESIAN_POINT\s*\([^)]*\(\s*"
        r"([+-]?[\d.Ee+\-]+)\s*,\s*"
        r"([+-]?[\d.Ee+\-]+)\s*,\s*"
        r"([+-]?[\d.Ee+\-]+)\s*\)",
        re.IGNORECASE,
    )
    xs, ys, zs = [], [], []
    for m in pt_pat.finditer(content):
        try:
            xs.append(float(m[1])); ys.append(float(m[2])); zs.append(float(m[3]))
        except ValueError:
            pass

    bb_x = round(max(xs) - min(xs), 3) if xs else None
    bb_y = round(max(ys) - min(ys), 3) if ys else None
    bb_z = round(max(zs) - min(zs), 3) if zs else None

    return {
        "n_faces": n_faces, "n_edges": n_edges, "n_vertices": n_vertices,
        "n_solids": n_solids, "n_components": n_components,
        "bb_x": bb_x, "bb_y": bb_y, "bb_z": bb_z,
    }


def display_step_geometry_summary(content: str, filename: str = "") -> dict:
    """
    Render the 'STEP Geometry Summary' panel above the compliance dashboard.

    Parameters
    ----------
    content  : Raw decoded STEP file text
    filename : Optional filename shown in the panel header

    Returns
    -------
    stats dict (n_faces, n_edges, n_vertices, n_solids, n_components, bb_x/y/z)
    """
    from IPython.display import HTML, display as _display

    s    = _extract_step_geometry_stats(content)
    bb_x = s["bb_x"]; bb_y = s["bb_y"]; bb_z = s["bb_z"]

    if bb_x is not None:
        bb_row = (
            f"<div style='background:#1f2937;border-radius:8px;padding:10px 14px;"
            f"grid-column:1 / -1;'>"
            f"<div style='color:#6b7280;font-size:0.72em;margin-bottom:3px;"
            f"letter-spacing:0.6px;'>BOUNDING BOX (X \u00d7 Y \u00d7 Z)</div>"
            f"<div style='color:#f3f4f6;font-size:1.05em;font-weight:700;"
            f"font-family:monospace;'>"
            f"{bb_x:.3f}\u00a0mm \u00a0\u00d7\u00a0 {bb_y:.3f}\u00a0mm "
            f"\u00a0\u00d7\u00a0 {bb_z:.3f}\u00a0mm</div></div>"
        )
    else:
        bb_row = (
            "<div style='background:#1f2937;border-radius:8px;padding:10px 14px;"
            "grid-column:1 / -1;'>"
            "<div style='color:#6b7280;font-size:0.72em;margin-bottom:3px;"
            "letter-spacing:0.6px;'>BOUNDING BOX (X \u00d7 Y \u00d7 Z)</div>"
            "<div style='color:#9ca3af;font-size:0.92em;'>"
            "N/A \u2014 no CARTESIAN_POINT entities found in file</div></div>"
        )

    mc_badge = ""
    if s["n_components"] > 1:
        mc_badge = (
            f"<span style='background:#1d4ed8;color:#bfdbfe;padding:3px 10px;"
            f"border-radius:20px;font-size:0.78em;font-weight:bold;margin-left:10px;'>"
            f"\U0001f517 Multi-Component Assembly ({s['n_components']} parts)</span>"
        )

    file_label = f" \u2014 {filename}" if filename else ""

    def _card(value, label, color):
        return (
            f"<div style='background:#1f2937;border-radius:8px;padding:10px 14px;"
            f"text-align:center;'>"
            f"<div style='color:#6b7280;font-size:0.72em;margin-bottom:4px;"
            f"letter-spacing:0.6px;'>{label}</div>"
            f"<div style='color:{color};font-size:1.6em;font-weight:800;'>{value}</div>"
            f"</div>"
        )

    _display(HTML(
        f"<div style='background:#111827;border:1px solid #374151;border-radius:12px;"
        f"padding:16px 20px;margin:10px 0 16px 0;font-family:sans-serif;"
        f"box-shadow:0 4px 18px rgba(0,0,0,0.45);'>"
        f"<div style='display:flex;align-items:center;margin-bottom:14px;"
        f"flex-wrap:wrap;gap:6px;'>"
        f"<span style='color:#93c5fd;font-weight:bold;font-size:0.9em;"
        f"letter-spacing:1.2px;'>\U0001f4d0 STEP GEOMETRY SUMMARY</span>"
        f"<span style='color:#4b5563;font-size:0.82em;'>{file_label}</span>"
        f"{mc_badge}</div>"
        f"<div style='display:grid;grid-template-columns:repeat(5,1fr);"
        f"gap:8px;margin-bottom:10px;'>"
        f"{_card(s['n_faces'],    'FACES',      '#60a5fa')}"
        f"{_card(s['n_edges'],    'EDGES',      '#a78bfa')}"
        f"{_card(s['n_vertices'], 'VERTICES',   '#34d399')}"
        f"{_card(s['n_solids'],   'SOLIDS',     '#fb923c')}"
        f"{_card(s['n_components'],'COMPONENTS','#f472b6')}"
        f"</div>"
        f"<div style='display:grid;grid-template-columns:1fr;'>{bb_row}</div>"
        f"</div>"
    ))
    return s


log.info("Enhancement 1: display_step_geometry_summary() ready.")


In [ ]:
def display_verdict_banner(result):
    """Displays a large HTML banner based on the compliance verdict."""
    sc = result['scores']
    verdict = sc['verdict']
    fused = sc['fused_score']
    v_map = {
        'COMPLIANT':      {'color': '#10B981', 'icon': '✅'},
        'REVIEW NEEDED':  {'color': '#F59E0B', 'icon': '⚠️'},
        'NON-COMPLIANT':  {'color': '#EF4444', 'icon': '🚫'}
    }
    v_cfg = v_map.get(verdict, {'color': '#374151', 'icon': '🔍'})
    display (HTML(f"""
    <div style='background-color:{v_cfg['color']};padding:20px;border-radius:12px;
                text-align:center;color:white;font-family:sans-serif;'>
        <h1 style='margin:0;font-size:2.2em;'>{v_cfg['icon']} {verdict}</h1>
        <p style='margin:5px 0 0 0;font-size:1.1em;opacity:0.9;'>
            Final Compliance Risk Score: {fused}%</p>
    </div>"""))

def display_score_cards(result):
    """Displays horizontal score cards for ML, Rule, and Fused scores."""
    sc = result['scores']
    display(HTML(f"""
    <div style='display:flex;justify-content:space-around;gap:15px;margin:20px 0;'>
        <div style='flex:1;border:2px solid #3B82F6;border-radius:10px;padding:10px;
                    text-align:center;background:#f8fafc;'>
            <div style='color:#1e40af;font-weight:bold;font-size:0.8em;'>ML Non-Compliance Probability</div>
            <div style='font-size:1.8em;font-weight:bold;color:#1e3a8a;'>{sc['ml_score']}</div>
        </div>
        <div style='flex:1;border:2px solid #8B5CF6;border-radius:10px;padding:10px;
                    text-align:center;background:#f8fafc;'>
            <div style='color:#5b21b6;font-weight:bold;font-size:0.8em;'>Rule Violation Severity Score</div>
            <div style='font-size:1.8em;font-weight:bold;color:#4c1d95;'>{sc['rule_score']}</div>
        </div>
        <div style='flex:1;border:2px solid #10B981;border-radius:10px;padding:10px;
                    text-align:center;background:#f8fafc;'>
            <div style='color:#065f46;font-weight:bold;font-size:0.8em;'>Fused Compliance Risk Score</div>
            <div style='font-size:1.8em;font-weight:bold;color:#064e3b;'>{sc['fused_score']}</div>
        </div>
    </div>"""))

In [ ]:
# =============================================================================
# UPGRADE 1 & 2 HELPERS — Decision Basis + Release Recommendation
# Added as a new cell; does NOT touch any existing logic.
# =============================================================================

def _compute_decision_basis(res: Dict[str, Any]) -> str:
    """Return the decision-basis label (and uncertainty suffix if needed).

    Rules (additive):
      1. CRITICAL violations present  → 'Rule-Dominant Safety Override'
      2. No CRITICAL, verdict from fused score → 'Balanced Rule + GNN Assessment'
      3. Reconstruction error >= 15.0 → append '| Elevated Model Uncertainty'
    """
    sc = res.get('scores', {})
    n_crit      = sc.get('critical', 0)
    recon_err   = float(res.get('reconstruction_error', 0.0))
    high_uncert = recon_err >= 15.0

    if n_crit > 0:
        basis = 'Rule-Dominant Safety Override'
    else:
        basis = 'Balanced Rule + GNN Assessment'

    if high_uncert:
        basis += ' | Elevated Model Uncertainty'

    return basis


def _get_release_recommendation(verdict: str) -> Dict[str, Any]:
    """Map verdict → release recommendation label + styling."""
    mapping = {
        'COMPLIANT':      {'label': 'Release Approved',            'icon': '✅', 'color': '#10B981', 'bg': '#064e3b', 'border': '#10B981'},
        'REVIEW NEEDED':  {'label': 'Engineering Review Required', 'icon': '⚠️', 'color': '#F59E0B', 'bg': '#78350f', 'border': '#F59E0B'},
        'NON-COMPLIANT':  {'label': 'Production Hold',             'icon': '🚫', 'color': '#EF4444', 'bg': '#7f1d1d', 'border': '#EF4444'},
    }
    return mapping.get(verdict, {'label': 'Under Review', 'icon': '🔍', 'color': '#6B7280', 'bg': '#1f2937', 'border': '#6B7280'})


def _get_top3_remediations(res: Dict[str, Any]) -> list:
    """Extract top-3 remediation action strings from violations / advisor logic."""
    v_list  = res.get('violations', [])
    recon   = float(res.get('reconstruction_error', 0.0))
    actions = []

    cat_seen = set()

    def _attr(v, key, default=''):
        return getattr(v, key, v.get(key, default) if isinstance(v, dict) else default)

    for v in v_list:
        cat = str(_attr(v, 'category', '')).upper()
        if 'TOL' in cat and 'TOL' not in cat_seen:
            actions.append('Recalibrate CNC tooling or tighten process control to stay within tolerance budget.')
            cat_seen.add('TOL')
        elif 'SURF' in cat and 'SURF' not in cat_seen:
            actions.append('Switch to fine-grinding / superfinishing; verify grinding wheel grit and feed rate.')
            cat_seen.add('SURF')
        elif 'MAT' in cat and 'MAT' not in cat_seen:
            actions.append('Apply isolating coating or substitute galvanic-neutral alloy per ASTM B117.')
            cat_seen.add('MAT')
        elif 'DIM' in cat and 'DIM' not in cat_seen:
            actions.append('Revise CAD nominal or adjust process per ASME Y14.5 GD&T drawing callouts.')
            cat_seen.add('DIM')
        elif 'DFM' in cat and 'DFM' not in cat_seen:
            actions.append('Re-evaluate part geometry for manufacturability; reduce aspect ratio via topology optimisation.')
            cat_seen.add('DFM')
        elif 'JNTI' in cat and 'JNTI' not in cat_seen:
            actions.append('Adjust fit specification (H7/g6 etc.) and verify with go/no-go gauges after machining.')
            cat_seen.add('JNTI')
        if len(actions) >= 3:
            break

    if recon >= 15.0 and len(actions) < 3:
        actions.append('Simplify CAD features (fillets, holes, chamfers) to improve GNN prediction confidence.')

    if not actions:
        actions.append('No corrective actions required. Maintain current process settings.')

    return actions[:3]


# ─── UPGRADE 1: Decision Basis Banner ────────────────────────────────────────

def display_decision_basis(res: Dict[str, Any]) -> None:
    """Display the Decision Basis explanation line below the verdict banner."""
    basis = _compute_decision_basis(res)
    display(HTML(f"""
    <div style='background:#1f2937; border-left:4px solid #60a5fa; border-radius:6px;
                padding:9px 16px; margin:8px 0 4px 0; font-family:sans-serif;'>
        <span style='color:#9ca3af; font-size:0.82em; letter-spacing:0.8px;'>DECISION BASIS:</span>
        <span style='color:#93c5fd; font-size:0.92em; font-weight:600; margin-left:8px;'>{basis}</span>
    </div>
    """))


# ─── UPGRADE 2: Release Recommendation Card ──────────────────────────────────

def display_release_recommendation(res: Dict[str, Any]) -> None:
    """Display the Release Recommendation dashboard card."""
    sc      = res.get('scores', {})
    verdict = sc.get('verdict', 'UNKNOWN')
    basis   = _compute_decision_basis(res)
    rec     = _get_release_recommendation(verdict)
    top3    = _get_top3_remediations(res)

    # Build remediation rows
    rem_html = ''
    for i, action in enumerate(top3, 1):
        rem_html += f"""
        <div style='display:flex; gap:10px; align-items:flex-start; margin-bottom:6px;'>
            <span style='background:#374151; color:#93c5fd; border-radius:50%; width:22px; height:22px;
                         display:flex; align-items:center; justify-content:center;
                         font-size:0.78em; font-weight:bold; flex-shrink:0;'>{i}</span>
            <span style='color:#d1d5db; font-size:0.83em; line-height:1.5;'>{action}</span>
        </div>"""

    display(HTML(f"""
    <div style='background:#111827; border:2px solid {rec["border"]}; border-radius:12px;
                padding:18px 22px; margin:12px 0; font-family:sans-serif;
                box-shadow:0 4px 20px rgba(0,0,0,0.4);'>

        <!-- Header -->
        <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:14px;'>
            <span style='color:#9ca3af; font-weight:bold; font-size:0.85em; letter-spacing:1.2px;'>
                📋 RELEASE RECOMMENDATION
            </span>
            <span style='background:{rec["bg"]}; color:{rec["color"]}; padding:5px 14px;
                         border-radius:20px; font-size:0.88em; font-weight:bold;
                         border:1px solid {rec["border"]};'>
                {rec["icon"]} {rec["label"]}
            </span>
        </div>

        <!-- Metrics grid -->
        <div style='display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-bottom:14px;'>
            <div style='background:#1f2937; border-radius:8px; padding:10px; text-align:center;'>
                <div style='color:#6b7280; font-size:0.72em; margin-bottom:3px;'>COMPLIANCE RISK SCORE</div>
                <div style='color:{rec["color"]}; font-size:1.8em; font-weight:800;'>{sc.get("fused_score","—")}</div>
            </div>
            <div style='background:#1f2937; border-radius:8px; padding:10px; text-align:center;'>
                <div style='color:#6b7280; font-size:0.72em; margin-bottom:3px;'>VIOLATIONS (C / W / I)</div>
                <div style='font-size:1.1em; font-weight:700; margin-top:4px;'>
                    <span style='color:#EF4444;'>{sc.get("critical",0)}</span>
                    <span style='color:#6b7280; margin:0 4px;'>/</span>
                    <span style='color:#F59E0B;'>{sc.get("warnings",0)}</span>
                    <span style='color:#6b7280; margin:0 4px;'>/</span>
                    <span style='color:#3B82F6;'>{sc.get("info",0)}</span>
                </div>
            </div>
            <div style='background:#1f2937; border-radius:8px; padding:10px; text-align:center;'>
                <div style='color:#6b7280; font-size:0.72em; margin-bottom:3px;'>ML PREDICTION / CONF.</div>
                <div style='color:#a78bfa; font-size:0.92em; font-weight:700; margin-top:4px;'>
                    {str(res.get("ml_prediction","—")).upper()}<br>
                    <span style='color:#c4b5fd; font-size:0.85em;'>{res.get("ml_confidence","—")}%</span>
                </div>
            </div>
        </div>

        <!-- Decision Basis row -->
        <div style='background:#1f2937; border-radius:6px; padding:8px 12px; margin-bottom:14px;'>
            <span style='color:#6b7280; font-size:0.75em;'>DECISION BASIS: </span>
            <span style='color:#93c5fd; font-size:0.82em; font-weight:600;'>{basis}</span>
        </div>

        <!-- Top-3 Remediations -->
        <div style='border-top:1px solid #374151; padding-top:12px;'>
            <div style='color:#9ca3af; font-size:0.78em; font-weight:bold;
                        letter-spacing:0.8px; margin-bottom:10px;'>
                🔧 TOP REMEDIATION ACTIONS
            </div>
            {rem_html}
        </div>
    </div>
    """))


# ─── Helper: plain-text version for export ───────────────────────────────────

def _release_recommendation_text(res: Dict[str, Any]) -> str:
    """Return a plain-text version of the Release Recommendation for export."""
    sc      = res.get('scores', {})
    verdict = sc.get('verdict', 'UNKNOWN')
    basis   = _compute_decision_basis(res)
    rec     = _get_release_recommendation(verdict)
    top3    = _get_top3_remediations(res)

    lines = [
        '=' * 70,
        'RELEASE RECOMMENDATION',
        '=' * 70,
        f'  Status              : {rec["icon"]} {rec["label"]}',
        f'  Fused Compliance Risk Score : {sc.get("fused_score","—")}',
        f'  Violations (C/W/I)  : {sc.get("critical",0)} / {sc.get("warnings",0)} / {sc.get("info",0)}',
        f'  Predicted Class     : {str(res.get("ml_prediction","—")).upper()}',
        f'  ML Confidence       : {res.get("ml_confidence","—")}%',
        f'  Decision Basis      : {basis}',
        '',
        '  TOP REMEDIATION ACTIONS:',
    ]
    for i, a in enumerate(top3, 1):
        lines.append(f'  {i}. {a}')
    lines.append('=' * 70)
    return '\n'.join(lines)


log.info('UPGRADE 1 & 2 helpers loaded: Decision Basis + Release Recommendation ready.')


In [ ]:
# ── Consolidated aliases for backward compatibility ──────────────────────────
# These wrap the canonical functions (display_verdict_banner / display_score_cards)
# so any external calls still work without duplicating logic.

def render_verdict_banner(scores: Dict[str, Any]):
    """Alias → display_verdict_banner.  Accepts scores dict directly."""
    # Build a minimal result wrapper so display_verdict_banner can read scores
    display_verdict_banner({'scores': scores})

def render_score_cards(res: Dict[str, Any]):
    """Alias → display_score_cards with updated canonical label names."""
    sc = res['scores']
    from IPython.display import HTML, display
    display(HTML(f"""
    <div style="display: flex; justify-content: space-around; gap: 20px; margin: 25px 0;">
        <div style="flex: 1; border: 2px solid #3B82F6; border-radius: 10px; padding: 15px; text-align: center; background: #EFF6FF;">
            <div style="color: #1E40AF; font-weight: bold; font-size: 0.9em;">ML Non-Compliance Probability</div>
            <div style="font-size: 2.2em; font-weight: 800; color: #1E3A8A;">{sc['ml_score']}</div>
        </div>
        <div style="flex: 1; border: 2px solid #8B5CF6; border-radius: 10px; padding: 15px; text-align: center; background: #F5F3FF;">
            <div style="color: #5B21B6; font-weight: bold; font-size: 0.9em;">Rule Violation Severity Score</div>
            <div style="font-size: 2.2em; font-weight: 800; color: #4C1D95;">{sc['rule_score']}</div>
        </div>
        <div style="flex: 1; border: 2px solid #10B981; border-radius: 10px; padding: 15px; text-align: center; background: #ECFDF5;">
            <div style="color: #065F46; font-weight: bold; font-size: 0.9em;">Fused Compliance Risk Score</div>
            <div style="font-size: 2.2em; font-weight: 800; color: #064E3B;">{sc['fused_score']}</div>
        </div>
    </div>
    """))

In [ ]:
def render_compliance_charts(res: Dict[str, Any]):
    """Repaired Analytics Dashboard Plots with Fixed Axis Categories"""
    sc = res['scores']
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), facecolor='white')

    # --- PLOT 1: Compliance Health Gauge ---
    score = sc['fused_score']
    color = '#10B981' if score > 70 else ('#F59E0B' if score > 40 else '#EF4444')
    ax1.pie([score, 100-score], colors=[color, '#E5E7EB'], startangle=90, counterclock=False, wedgeprops={'width':0.35})
    ax1.text(0, 0, f"{score}%", ha='center', va='center', fontsize=32, fontweight='800', color='#111827')
    ax1.set_title("Fused Compliance Risk Score", fontweight='bold', fontsize=18, pad=20)

    # --- PLOT 2: Severity Distribution (Fixed Axis Categories) ---
    categories = ['Critical', 'Warning', 'Info']
    counts = [sc.get('critical', 0), sc.get('warnings', 0), sc.get('info', 0)]
    colors = ['#EF4444', '#F59E0B', '#3B82F6']

    bars = ax2.bar(categories, counts, color=colors, alpha=0.9, edgecolor='#333', width=0.6)
    ax2.set_title("Violation Severity Distribution", fontweight='bold', fontsize=18, pad=20)
    ax2.set_ylabel("Count", fontsize=12)

    ax2.set_ylim(0, max(5, max(counts) + 1))
    ax2.grid(axis='y', linestyle='--', alpha=0.5)

    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1, f'{int(height)}',
                 ha='center', va='bottom', fontweight='bold', fontsize=12)

    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    plt.tight_layout(pad=3.0)
    plt.show()

In [ ]:
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

# Component-type hierarchy used to infer assembly relationships.
# A hub type (shaft, housing) connects to its satellite types.
_ASSEMBLY_HIERARCHY = {
    'shaft':    ['bearing', 'fastener', 'gear'],
    'housing':  ['bearing', 'seal', 'gasket'],
    'bearing':  ['fastener', 'seal'],
    'gear':     ['fastener'],
    'bracket':  ['fastener', 'seal'],
    'plate':    ['fastener', 'gasket'],
}

def _build_assembly_graph(components):
    """
    Build a directed assembly graph from component metadata.

    Rules (in priority order):
    1. Type-hierarchy edges — connect each component to downstream
       components whose type appears in its hub list.
    2. Spatial-proximity edges — connect any remaining isolated nodes
       to their nearest neighbour by Euclidean (x, y, z) distance.
    3. Sequential fallback — if still fewer than n-1 edges exist,
       add a chain so every node is reachable.

    Returns a networkx DiGraph and a dict of node labels.
    """
    n = len(components)
    G = nx.DiGraph()
    G.add_nodes_from(range(n))

    labels = {}
    for i, comp in enumerate(components):
        ctype = comp.get('type', 'part')
        labels[i] = f"Node {i}\n{ctype}"

    # 1. Type-hierarchy edges
    for i, comp in enumerate(components):
        ctype = comp.get('type', 'part').lower()
        downstream = _ASSEMBLY_HIERARCHY.get(ctype, [])
        for j, other in enumerate(components):
            if i == j:
                continue
            if other.get('type', '').lower() in downstream:
                G.add_edge(i, j)

    # 2. Spatial-proximity edges for isolated nodes
    coords = np.array([
        [c.get('actual_x', 0), c.get('actual_y', 0), c.get('actual_z', 0)]
        for c in components
    ], dtype=float)

    isolated = [v for v in G.nodes() if G.degree(v) == 0]
    for v in isolated:
        dists = np.linalg.norm(coords - coords[v], axis=1)
        dists[v] = np.inf
        nearest = int(np.argmin(dists))
        G.add_edge(v, nearest)

    # 3. Sequential fallback — guarantee connectivity
    if nx.number_weakly_connected_components(G) > 1:
        for i in range(n - 1):
            if not (G.has_edge(i, i + 1) or G.has_edge(i + 1, i)):
                G.add_edge(i, i + 1)

    return G, labels


def render_component_graph(res, ax):
    """Renders a meaningful GNN assembly topology graph on the given Axes."""
    components  = res.get('components', [])   # populated by widget _on_run
    n           = res.get('n_components', 1)
    violations  = res.get('violations', [])
    violated    = {v.node for v in violations if isinstance(v.node, int)}

    # Fallback: if components list was not stored in res, build stubs from n
    if not components:
        components = [{'type': 'part', 'actual_x': float(i*10),
                       'actual_y': 0.0, 'actual_z': 0.0} for i in range(n)]

    G, labels = _build_assembly_graph(components)
    n = len(components)

    node_colors  = ['#EF4444' if i in violated else '#10B981' for i in range(n)]
    node_sizes   = [1000 if i in violated else 750 for i in range(n)]

    # Layout: hierarchical if enough nodes, otherwise spring
    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except Exception:
        pos = nx.spring_layout(G, k=2.0 / max(1, np.sqrt(n)), seed=42)

    ax.set_facecolor('#0D1117')

    # Draw edges with arrows
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color='#6B7280', width=1.8,
        arrowstyle='-|>', arrowsize=18,
        connectionstyle='arc3,rad=0.08'
    )
    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=node_colors, node_size=node_sizes,
        edgecolors='white', linewidths=1.2
    )
    # Draw labels
    nx.draw_networkx_labels(
        G, pos, labels=labels, ax=ax,
        font_color='white', font_weight='bold', font_size=8
    )

    # Edge weight annotations (component type relationship)
    edge_labels = {}
    for u, v in G.edges():
        cu = components[u].get('type', '')[:4]
        cv = components[v].get('type', '')[:4]
        edge_labels[(u, v)] = f"{cu}→{cv}"
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, ax=ax,
        font_color='#9CA3AF', font_size=7,
        bbox=dict(boxstyle='round,pad=0.15', fc='#1F2937', ec='none', alpha=0.7)
    )

    ax.set_title("Assembly Relationship View", fontweight='bold',
                 fontsize=13, color='white', pad=14)
    legend_elements = [
        mpatches.Patch(color='#EF4444', label='Violated'),
        mpatches.Patch(color='#10B981', label='Compliant'),
    ]
    ax.legend(handles=legend_elements, loc='upper right',
              facecolor='#1F2937', edgecolor='#374151',
              labelcolor='white', fontsize=8)
    ax.axis('off')

In [ ]:
# =============================================================================
# ENHANCEMENT 2 — Multi-Component Assembly Display Helpers
#
# display_assembly_header(n_components, step_stats=None)
#   Blue gradient banner shown above the compliance verdict for assemblies
#   with more than one component.  No-op for single-component input —
#   the output is identical to v5.2 in that case.
#
# display_per_component_table(components)
#   Compact per-component breakdown table rendered after the assembly graph.
#   Lists Node, Type, Dims (mm), Material, Ra for every component.
#   No-op for single-component input.
#
# Neither function touches run_inference, the rule engine, or the GNN model.
# =============================================================================

def display_assembly_header(n_components: int, step_stats: dict = None) -> None:
    """
    Show a compact assembly banner above the compliance dashboard.
    No-op when n_components <= 1 (preserves exact v5.2 single-part behaviour).
    """
    from IPython.display import HTML, display as _display

    if n_components <= 1:
        return

    geo_items = ""
    if step_stats:
        bb_x = step_stats.get("bb_x")
        bb_y = step_stats.get("bb_y")
        bb_z = step_stats.get("bb_z")
        if bb_x is not None:
            geo_items = (
                f"<span style='color:#9ca3af;font-size:0.8em;margin-left:16px;'>"
                f"BBox: {bb_x:.1f}×{bb_y:.1f}×{bb_z:.1f} mm</span>"
            )

    _display(HTML(
        f"<div style='background:linear-gradient(90deg,#1e3a5f,#1f2937);"
        f"border-left:4px solid #3b82f6;border-radius:8px;"
        f"padding:9px 16px;margin:6px 0 10px 0;font-family:sans-serif;"
        f"display:flex;align-items:center;gap:14px;flex-wrap:wrap;'>"
        f"<span style='color:#93c5fd;font-weight:bold;font-size:0.88em;"
        f"letter-spacing:0.8px;'>🔗 MULTI-COMPONENT ASSEMBLY</span>"
        f"<span style='background:#1d4ed8;color:#bfdbfe;padding:3px 10px;"
        f"border-radius:20px;font-size:0.82em;font-weight:bold;'>"
        f"{n_components} components</span>"
        f"{geo_items}"
        f"<span style='color:#6b7280;font-size:0.77em;margin-left:auto;'>"
        f"Validation pipeline ran per component · Assembly graph below</span>"
        f"</div>"
    ))


def display_per_component_table(components: list) -> None:
    """
    Render a per-component breakdown table after the assembly graph.
    No-op when len(components) <= 1 (single-part behaviour unchanged).
    """
    from IPython.display import HTML, display as _display

    if len(components) <= 1:
        return

    rows = ""
    for i, comp in enumerate(components):
        ctype = comp.get("type", "part")
        dx    = comp.get("actual_x", "—")
        dy    = comp.get("actual_y", "—")
        dz    = comp.get("actual_z", "—")
        mat   = comp.get("material", "steel")
        ra    = comp.get("roughness", "—")
        try:
            dims_str = f"{dx:.1f} × {dy:.1f} × {dz:.1f}"
        except (TypeError, ValueError):
            dims_str = f"{dx} × {dy} × {dz}"
        bg = "#1a2332" if i % 2 == 0 else "#1f2937"
        rows += (
            f"<tr style='background:{bg};'>"
            f"<td style='padding:7px 10px;color:#93c5fd;font-weight:bold;'>{i}</td>"
            f"<td style='padding:7px 10px;color:#d1d5db;'>{ctype}</td>"
            f"<td style='padding:7px 10px;color:#d1d5db;font-family:monospace;"
            f"font-size:0.85em;'>{dims_str}</td>"
            f"<td style='padding:7px 10px;color:#d1d5db;'>{mat}</td>"
            f"<td style='padding:7px 10px;color:#d1d5db;'>{ra}</td>"
            f"</tr>"
        )

    _display(HTML(
        f"<div style='margin:14px 0;font-family:sans-serif;'>"
        f"<div style='color:#9ca3af;font-weight:bold;font-size:0.82em;"
        f"letter-spacing:1px;margin-bottom:8px;'>📋 PER-COMPONENT BREAKDOWN</div>"
        f"<div style='overflow-x:auto;'>"
        f"<table style='width:100%;border-collapse:collapse;font-size:0.84em;'>"
        f"<thead><tr style='background:#0d1117;border-bottom:2px solid #374151;'>"
        f"<th style='padding:8px 10px;color:#6b7280;text-align:left;font-weight:600;'>NODE</th>"
        f"<th style='padding:8px 10px;color:#6b7280;text-align:left;font-weight:600;'>TYPE</th>"
        f"<th style='padding:8px 10px;color:#6b7280;text-align:left;font-weight:600;'>DIMS (mm)</th>"
        f"<th style='padding:8px 10px;color:#6b7280;text-align:left;font-weight:600;'>MATERIAL</th>"
        f"<th style='padding:8px 10px;color:#6b7280;text-align:left;font-weight:600;'>Ra (µm)</th>"
        f"</tr></thead>"
        f"<tbody>{rows}</tbody>"
        f"</table></div></div>"
    ))


log.info("Enhancement 2: display_assembly_header() + display_per_component_table() ready.")


In [ ]:
import plotly.graph_objects as go
import numpy as np
from IPython.display import HTML, display

def render_cad_3d_model(components: List[Dict[str, Any]], violations: List[Violation]):
    """
    REPAIRED 3D Digital Twin Viewer.
    Renders each component as a solid cylinder (shaft/bearing/fastener)
    or cuboid (all others) with explicit lighting and a visible scene background.
    Violations are colour-coded: green=pass, orange=warning, red=critical.
    """
    fig = go.Figure()
    severity_map = {v.node: v.severity for v in violations if isinstance(v.node, int)}
    offset_x = 0.0

    for i, comp in enumerate(components):
        # --- derive visible dimensions ---
        dx = max(comp.get('actual_x', 50) / 8.0, 3.0)
        dy = max(comp.get('actual_y', 50) / 8.0, 3.0)
        dz = max(comp.get('actual_z', 20) / 6.0, 1.5)

        status = severity_map.get(i, 'PASSED')
        color  = '#10B981' if status == 'PASSED' else ('#EF4444' if status == 'CRITICAL' else '#F59E0B')
        label  = f"Node {i}: {comp.get('type', 'part').upper()} ({status})"
        cx     = offset_x + dx / 2.0

        ctype = comp.get('type', '')
        if ctype in ('shaft', 'bearing', 'fastener'):
            # --- Cylinder via parametric Surface ---
            n_theta, n_z = 32, 10
            theta  = np.linspace(0, 2 * np.pi, n_theta)
            z_vals = np.linspace(0, dz, n_z)
            T, Z   = np.meshgrid(theta, z_vals)
            r      = dy / 2.0

            # Cylinder wall
            fig.add_trace(go.Surface(
                x = r * np.cos(T) + cx,
                y = r * np.sin(T),
                z = Z,
                colorscale=[[0, color], [1, color]],
                showscale=False, name=label, opacity=0.9,
                lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3, roughness=0.5, fresnel=0.1),
                lightposition=dict(x=100, y=200, z=300)
            ))
            # Top cap
            cap_r = np.array([[0, r]])
            cap_t = np.linspace(0, 2 * np.pi, n_theta).reshape(1, -1)
            fig.add_trace(go.Surface(
                x = cap_r.T * np.cos(cap_t) + cx,
                y = cap_r.T * np.sin(cap_t),
                z = np.full((2, n_theta), dz),
                colorscale=[[0, color], [1, color]],
                showscale=False, name=label + " top", opacity=0.9,
                lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3, roughness=0.5, fresnel=0.1),
                lightposition=dict(x=100, y=200, z=300)
            ))
            # Bottom cap
            fig.add_trace(go.Surface(
                x = cap_r.T * np.cos(cap_t) + cx,
                y = cap_r.T * np.sin(cap_t),
                z = np.zeros((2, n_theta)),
                colorscale=[[0, color], [1, color]],
                showscale=False, name=label + " bot", opacity=0.9,
                lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3, roughness=0.5, fresnel=0.1),
                lightposition=dict(x=100, y=200, z=300)
            ))
        else:
            # --- Cuboid via Mesh3d ---
            x0, x1 = offset_x, offset_x + dx
            y0, y1 = -dy / 2.0, dy / 2.0
            z0, z1 = 0.0, dz
            fig.add_trace(go.Mesh3d(
                x=[x0,x0,x1,x1,x0,x0,x1,x1],
                y=[y0,y1,y1,y0,y0,y1,y1,y0],
                z=[z0,z0,z0,z0,z1,z1,z1,z1],
                i=[7,0,0,0,4,4,6,6,4,0,3,2],
                j=[3,4,1,2,5,6,5,2,0,1,6,3],
                k=[0,7,2,3,6,7,1,1,5,5,7,6],
                color=color, opacity=0.85, name=label,
                flatshading=True,
                lighting=dict(ambient=0.4, diffuse=0.9, specular=0.4, roughness=0.4, fresnel=0.2),
                lightposition=dict(x=200, y=300, z=400)
            ))

        # Add component label above the object
        fig.add_trace(go.Scatter3d(
            x=[cx],
            y=[0],
            z=[dz + 2.0],  # Position slightly above the component
            mode='text',
            text=[f"N{i} {comp.get('type', 'part').upper()}"],
            textfont=dict(color="white", size=10),
            showlegend=False
        ))

        offset_x += dx + 15.0   # generous gap between components

    # --- Layout: dark frame, bright scene background so geometry is always visible ---
    fig.update_layout(
        width=1100, height=520,
        margin=dict(l=0, r=0, t=30, b=0),
        paper_bgcolor='#111827',
        scene=dict(
            aspectmode='data',
            bgcolor='#1F2937',          # dark grey — geometry visible against it
            xaxis=dict(
                visible=True, showbackground=True, showgrid=True, title='X-axis',
                gridcolor='#374151', backgroundcolor='#2D3748', tickfont=dict(color='white'),
                title_font=dict(color='white')
            ),
            yaxis=dict(
                visible=True, showbackground=True, showgrid=True, title='Y-axis',
                gridcolor='#374151', backgroundcolor='#2D3748', tickfont=dict(color='white'),
                title_font=dict(color='white')
            ),
            zaxis=dict(
                visible=True, showbackground=True, showgrid=True, title='Z-axis',
                gridcolor='#374151', backgroundcolor='#2D3748', tickfont=dict(color='white'),
                title_font=dict(color='white')
            ),
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=1.8, y=1.8, z=1.0)
            )
        ),
        template="plotly_dark",
        showlegend=False,
        title=dict(
            text="3D Digital Twin — Component Assembly",
            font=dict(color='white', size=16),
            x=0.5
        )
    )

    config = {'scrollZoom': True, 'displayModeBar': True, 'responsive': True}
    html_str = fig.to_html(include_plotlyjs='cdn', full_html=False, config=config)
    display(HTML(
        f"<div style=\"width:100%;height:540px;background:#111827;"
        f"border-radius:10px;border:1px solid #374151;overflow:hidden;\">"
        f"{html_str}</div>"
    ))


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3D DIGITAL TWIN — run this cell manually after clicking Run Validation
# ─────────────────────────────────────────────────────────────────
if _last_3d_data.get('components'):
    render_cad_3d_model(_last_3d_data['components'], _last_3d_data['violations'])
else:
    print("No 3D data yet. Click Run Validation first, then re-run this cell.")


No 3D data yet. Click Run Validation first, then re-run this cell.
